# 1. MRI MNI registration, N4 correction and final preprocessing

This notebook is part of the reproducible data-preparation pipeline used by the downstream modelling notebooks.


Yes, this aligns well ,  and it actually improves on what I gave you in two ways worth calling out.

**Where it matches exactly:**
- N4 bias correction first, registration second, normalization last ,  same order, same logic (bias-correct before registration so the registration cost function isn't thrown off by intensity gradients).
- No skull-stripping ,  consistent with your decision based on the Graz paper.
- Registering from the RAS NIfTI directly, with the final voxel spacing baked into the registration output (step 2) ,  this is actually *more correct* than how I left it. It does the resampling in one pass (RAS → 1.5mm MNI space directly), rather than resampling once to 1.5mm and then resampling again during registration. Avoiding that double interpolation is good practice ,  each resampling step introduces some blurring, so doing it once is better than twice.

**Where it adds something useful I hadn't spelled out:**
- **Step 3's conditional crop/pad** ("skip if MNI output already has the desired shape") is a sharper point than what I said. Since every volume gets registered onto the *same* template grid at the *same* spacing, they'll all come out the same shape automatically ,  there's no per-subject variability to crop away anymore. The only reason you'd still need to crop/pad is if the template's native grid isn't the exact shape you want for your model (e.g., you want a specific size for architecture reasons, or want to trim empty space around the head to shrink the tensor). So "skip if not needed" is the right framing ,  don't treat it as a mandatory step anymore now that registration handles shape consistency for you.
- **Step 5 (save as .nii.gz + .npy/.pt)** is a practical step I hadn't included ,  useful since you're now at the point of feeding a model. Good to keep .nii.gz around for visual QC (you'll want to eyeball a few registrations before trusting the batch) and use .npy/.pt for actual training.

**One thing to pin down before running it:** which MNI152 template and resolution variant (e.g., MNI152NLin2009cSym vs ICBM152) and what final fixed shape you're targeting (e.g., if you want to match TriFormer's 128×128×128). That determines what step 3 actually has to do, if anything.

Otherwise ,  yes, this is consistent with everything we've worked through, and the two refinements are genuine improvements over my version.

In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount("/content/drive")

In [ ]:

!pip install SimpleITK -q

In [ ]:

# Imports

from pathlib import Path
import os
import json
import time
import traceback

import numpy as np
import pandas as pd

import nibabel as nib
import SimpleITK as sitk

from tqdm.auto import tqdm

## 1.1. Define notebook paths and output folders

In [ ]:
# Main ADNI MRI project folder
PROJECT_DIR = Path("/content/drive/MyDrive/adni_mri")

# Fallback for older Colab Drive path format
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path("/content/drive/My Drive/adni_mri")

assert PROJECT_DIR.exists(), f"Project folder not found: {PROJECT_DIR}"

# Input manifest: RAS NIfTI files + scanner metadata
INPUT_MANIFEST = PROJECT_DIR / "manifest" / "clean_90d_ras_nifti_manifest_1063_with_scanner_metadata.csv"

# Input image folder: use RAS images, not the already-resampled 1.5mm branch
INPUT_RAS_DIR = PROJECT_DIR / "processed" / "nifti_ras"

# Output branch name
BRANCH_NAME = "clean_90d_mni_n4_syn_jacobian"

# Final output folders
OUTPUT_NII_DIR = PROJECT_DIR / "processed" / "nifti_mni_1p5mm_n4_affine"
OUTPUT_NPY_DIR = PROJECT_DIR / "processed" / "npy_mni_1p5mm_n4_affine"

# Output files
OUTPUT_MANIFEST = PROJECT_DIR / "manifest" / f"{BRANCH_NAME}_manifest_1063.csv"
OUTPUT_REPORT = PROJECT_DIR / "qc" / f"{BRANCH_NAME}_report.csv"

# Create output folders
OUTPUT_NII_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_NPY_DIR.mkdir(parents=True, exist_ok=True)

# Basic path checks
assert INPUT_MANIFEST.exists(), f"Input manifest not found: {INPUT_MANIFEST}"
assert INPUT_RAS_DIR.exists(), f"Input RAS folder not found: {INPUT_RAS_DIR}"

print("PROJECT_DIR:", PROJECT_DIR)
print("INPUT_MANIFEST:", INPUT_MANIFEST)
print("INPUT_RAS_DIR:", INPUT_RAS_DIR)
print("OUTPUT_NII_DIR:", OUTPUT_NII_DIR)
print("OUTPUT_NPY_DIR:", OUTPUT_NPY_DIR)
print("OUTPUT_MANIFEST:", OUTPUT_MANIFEST)
print("OUTPUT_REPORT:", OUTPUT_REPORT)

## 1.2. Load the RAS manifest and inspect available columns

In [ ]:
manifest = pd.read_csv(INPUT_MANIFEST)

print("Manifest shape:", manifest.shape)
print("\nColumns:")
for col in manifest.columns:
    print("-", col)

display(manifest.head())

## 1.3. Quick overview of labels and scanner metadata

In [ ]:
print("Number of rows:", len(manifest))

if "RID" in manifest.columns:
    print("Unique RIDs:", manifest["RID"].nunique())

if "image_id" in manifest.columns:
    print("Unique image IDs:", manifest["image_id"].nunique())
elif "Image ID" in manifest.columns:
    print("Unique image IDs:", manifest["Image ID"].nunique())

if "final_group" in manifest.columns:
    print("\nFinal group counts:")
    print(manifest["final_group"].value_counts(dropna=False))

scanner_cols = [
    col for col in manifest.columns
    if any(key in col.lower() for key in ["field", "strength", "manufacturer", "model"])
]

if scanner_cols:
    print("\nScanner metadata columns:")
    print(scanner_cols)
    display(manifest[scanner_cols].head())
else:
    print("\nNo obvious scanner metadata columns found.")

## 1.4. Verify RAS input paths and RAS QC status

In [ ]:
INPUT_PATH_COL = "output_nifti_path"
LABEL_COL = "final_group"
ID_COL = "RID"
IMAGE_ID_COL = "image_id"

required_cols = [INPUT_PATH_COL, LABEL_COL, ID_COL, IMAGE_ID_COL]
missing_cols = [col for col in required_cols if col not in manifest.columns]

assert not missing_cols, f"Missing required columns: {missing_cols}"

# Check RAS QC status if available
if "final_ras_qc_status" in manifest.columns:
    print("RAS QC status counts:")
    print(manifest["final_ras_qc_status"].value_counts(dropna=False))
else:
    print("No final_ras_qc_status column found.")

# Check whether every RAS NIfTI file exists
manifest["ras_nifti_path_exists"] = manifest[INPUT_PATH_COL].apply(lambda p: Path(str(p)).exists())

print("\nRAS NIfTI path existence:")
print(manifest["ras_nifti_path_exists"].value_counts(dropna=False))

missing_files = manifest[~manifest["ras_nifti_path_exists"]]

if len(missing_files) > 0:
    print(f"\nMissing files: {len(missing_files)}")
    display(missing_files[[ID_COL, IMAGE_ID_COL, LABEL_COL, INPUT_PATH_COL]].head(20))
else:
    print("\nAll RAS NIfTI input files exist.")

display(manifest[[ID_COL, IMAGE_ID_COL, LABEL_COL, INPUT_PATH_COL, "ras_nifti_path_exists"]].head())

## 1.5. Create output paths for the Route B MNI preprocessing branch

In this step, The notebook creates a working copy of the RAS NIfTI manifest and add the output locations for the new Route B preprocessing branch.

At this stage, I am not fixing the final voxel resolution in the folder names or filenames. The MNI registration step may initially use the default MNI grid, and decide later whether the output resolution is acceptable or whether another target grid is needed.

The goal of this step is only to prepare traceable output paths for each subject: one `.nii.gz` file for the final processed image and one `.npy` file for model training. I also add resume flags so that if the notebook is interrupted, I can later skip images that were already processed.

In [ ]:
# Create a resolution-neutral working manifest for Route B

# Define a neutral branch name.
# I am not including 1mm, 1.5mm, or 2mm here because the final MNI resolution has not been decided yet.
BRANCH_NAME = "clean_90d_mni_n4_affine"

# Define resolution-neutral output folders for the final Route B files.
# The NIfTI folder will store inspection-friendly neuroimaging files.
# The NumPy folder will store model-friendly arrays for faster PyTorch loading.
OUTPUT_NII_DIR = PROJECT_DIR / "processed" / "nifti_mni_n4_affine"
OUTPUT_NPY_DIR = PROJECT_DIR / "processed" / "npy_mni_n4_affine"

# Define resolution-neutral output manifest and report paths.
# These will later describe exactly what happened during N4 correction and MNI registration.
OUTPUT_MANIFEST = PROJECT_DIR / "manifest" / f"{BRANCH_NAME}_manifest_1063.csv"
OUTPUT_REPORT = PROJECT_DIR / "qc" / f"{BRANCH_NAME}_report.csv"

# Create the output folders if they do not already exist.
# This does not modify any MRI images; it only prepares the destination folders.
OUTPUT_NII_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_NPY_DIR.mkdir(parents=True, exist_ok=True)

# Define the key columns needed for traceable preprocessing.
# output_nifti_path is the path to the already RAS-reoriented NIfTI input image.
INPUT_PATH_COL = "output_nifti_path"
LABEL_COL = "final_group"
RID_COL = "RID"
IMAGE_ID_COL = "image_id"

# Check that the manifest contains the columns needed for this branch.
required_cols = [RID_COL, IMAGE_ID_COL, LABEL_COL, INPUT_PATH_COL]
missing_cols = [col for col in required_cols if col not in manifest.columns]

assert not missing_cols, f"Missing required columns in manifest: {missing_cols}"

# Create a working copy of the manifest.
# I keep the original manifest unchanged and add Route B-specific columns only to this copy.
work_manifest = manifest.copy()

# Store the RAS input path under a clearer name for this notebook.
# This makes it explicit that the input images are already RAS-reoriented NIfTI files.
work_manifest["input_ras_nifti_path"] = work_manifest[INPUT_PATH_COL].astype(str)

# Confirm that every RAS input file exists before creating the processing plan.
# This prevents the notebook from failing later during the expensive preprocessing loop.
work_manifest["input_ras_exists"] = work_manifest["input_ras_nifti_path"].apply(
    lambda p: Path(p).exists()
)

missing_input_files = work_manifest[~work_manifest["input_ras_exists"]]

assert len(missing_input_files) == 0, (
    f"{len(missing_input_files)} RAS NIfTI input files are missing. "
    "The pipeline should not continue until all input files exist."
)

# Create stable output stems using RID and image_id.
# This keeps each output file directly traceable to one subject and one selected ADNI MRI scan.
work_manifest["output_stem"] = work_manifest.apply(
    lambda row: f"RID{int(row[RID_COL])}_I{int(row[IMAGE_ID_COL])}",
    axis=1
)

# Create the future output path for the final MNI-space NIfTI file.
# The filename stays resolution-neutral until the MNI output grid is finalized.
work_manifest["mni_nii_path"] = work_manifest["output_stem"].apply(
    lambda stem: str(OUTPUT_NII_DIR / f"{stem}_mni_n4_affine.nii.gz")
)

# Create the future output path for the final MNI-space NumPy array.
# This will later be used as the faster model-training input.
work_manifest["mni_npy_path"] = work_manifest["output_stem"].apply(
    lambda stem: str(OUTPUT_NPY_DIR / f"{stem}_mni_n4_affine.npy")
)

# Add placeholders for MNI output metadata.
# These columns will be filled after registration, when the actual output spacing and shape are known.
work_manifest["mni_output_shape"] = None
work_manifest["mni_voxel_sizes"] = None
work_manifest["mni_template_used"] = None
work_manifest["mni_registration_type"] = "affine"
work_manifest["n4_bias_correction"] = True

# Add resume flags for existing outputs.
# These make it possible to continue safely if Colab disconnects partway through processing.
work_manifest["mni_nii_exists"] = work_manifest["mni_nii_path"].apply(
    lambda p: Path(p).exists()
)

work_manifest["mni_npy_exists"] = work_manifest["mni_npy_path"].apply(
    lambda p: Path(p).exists()
)

print("Working manifest shape:", work_manifest.shape)
print("Branch name:", BRANCH_NAME)

print("\nOutput folders:")
print("NIfTI:", OUTPUT_NII_DIR)
print("NumPy:", OUTPUT_NPY_DIR)

print("\nOutput files:")
print("Manifest:", OUTPUT_MANIFEST)
print("Report:", OUTPUT_REPORT)

print("\nExisting final MNI NIfTI outputs:")
print(work_manifest["mni_nii_exists"].value_counts(dropna=False))

print("\nExisting final MNI NumPy outputs:")
print(work_manifest["mni_npy_exists"].value_counts(dropna=False))

display(
    work_manifest[
        [
            RID_COL,
            IMAGE_ID_COL,
            LABEL_COL,
            "input_ras_nifti_path",
            "mni_nii_path",
            "mni_npy_path",
            "mni_nii_exists",
            "mni_npy_exists",
        ]
    ].head()
)

## 1.6. Check whether the current Route B cohort is compatible with a 65+ elderly template

use the `work_manifest` already created in this notebook and merge date of birth from `PTDEMOG_15Feb2026.csv`. calculate age at the selected MRI scan date using the existing `study_date` column, because the template question concerns the MRI cohort being preprocessed now.

In [ ]:
PTDEMOG_PATH = PROJECT_DIR / "PTDEMOG_15Feb2026.csv"

ptdemog = pd.read_csv(PTDEMOG_PATH)

dob_lookup = (
    ptdemog[["RID", "PTDOB"]]
    .dropna(subset=["PTDOB"])
    .drop_duplicates(subset=["RID"])
    .copy()
)

dob_lookup["date_of_birth_approx"] = pd.to_datetime(
    dob_lookup["PTDOB"],
    format="%m/%Y",
    errors="coerce"
)

work_manifest_with_age = work_manifest.merge(
    dob_lookup[["RID", "date_of_birth_approx"]],
    on="RID",
    how="left"
)

work_manifest_with_age["study_date"] = pd.to_datetime(
    work_manifest_with_age["study_date"],
    errors="coerce"
)

work_manifest_with_age["age_at_mri"] = (
    work_manifest_with_age["study_date"]
    - work_manifest_with_age["date_of_birth_approx"]
).dt.days / 365.25

missing_dob = int(work_manifest_with_age["date_of_birth_approx"].isna().sum())
missing_study_date = int(work_manifest_with_age["study_date"].isna().sum())
subjects_under_65 = int((work_manifest_with_age["age_at_mri"] < 65).sum())

print("Rows in current Route B cohort:", len(work_manifest_with_age))
print("Missing DOB:", missing_dob)
print("Missing MRI study date:", missing_study_date)
print("Minimum age at MRI:", work_manifest_with_age["age_at_mri"].min())
print("Maximum age at MRI:", work_manifest_with_age["age_at_mri"].max())
print("Subjects younger than 65:", subjects_under_65)

display(
    work_manifest_with_age
    .groupby("final_group")["age_at_mri"]
    .agg(["count", "min", "max", "mean"])
    .round(2)
)

display(
    work_manifest_with_age
    .assign(under_65=work_manifest_with_age["age_at_mri"] < 65)
    .groupby("final_group")["under_65"]
    .sum()
    .astype(int)
    .to_frame("subjects_under_65")
)

save the Route B manifest with age and a short 65+ template compatibility report. This keeps the template decision traceable without changing the preprocessing branch itself.

In [ ]:
AGE_MANIFEST = PROJECT_DIR / "manifest" / f"{BRANCH_NAME}_manifest_1063_with_age_at_mri.csv"
AGE_REPORT = PROJECT_DIR / "qc" / f"{BRANCH_NAME}_65plus_template_compatibility_report.csv"

work_manifest_with_age.to_csv(AGE_MANIFEST, index=False)

age_report = pd.DataFrame([{
    "branch_name": BRANCH_NAME,
    "n_subjects": len(work_manifest_with_age),
    "missing_dob": missing_dob,
    "missing_study_date": missing_study_date,
    "min_age_at_mri": work_manifest_with_age["age_at_mri"].min(),
    "max_age_at_mri": work_manifest_with_age["age_at_mri"].max(),
    "mean_age_at_mri": work_manifest_with_age["age_at_mri"].mean(),
    "subjects_under_65": subjects_under_65,
    "fully_compatible_with_65plus_template": (
        missing_dob == 0
        and missing_study_date == 0
        and subjects_under_65 == 0
    ),
}])

age_report.to_csv(AGE_REPORT, index=False)

display(age_report)

print("Saved age-enriched Route B manifest:")
print(AGE_MANIFEST)

print("\nSaved 65+ template compatibility report:")
print(AGE_REPORT)

## 1.7. Prepare output paths for the N4-first nonlinear MNI branch

keep using the clean RAS NIfTI files as input, but this branch will start by saving an N4-corrected version of each image before any MNI registration is applied. also prepare future output paths for the nonlinearly registered MNI T1 image and the log-Jacobian map, but the first real processing step after this will be N4 correction.

In [ ]:
# Add a separate folder for N4-corrected images before MNI registration.
# This is the first real output of the nonlinear MNI + Jacobian branch.
N4_NII_DIR = OUTPUT_NII_DIR.parent / f"{BRANCH_NAME}_n4_native_ras"

# Add a separate folder for log-Jacobian maps from the nonlinear registration.
LOGJAC_NII_DIR = OUTPUT_NII_DIR.parent / f"{BRANCH_NAME}_logjacobian"
LOGJAC_NPY_DIR = OUTPUT_NPY_DIR.parent / f"{BRANCH_NAME}_logjacobian"

for folder in [N4_NII_DIR, LOGJAC_NII_DIR, LOGJAC_NPY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# N4-corrected image in native/RAS space.
work_manifest["n4_nii_path"] = work_manifest["output_stem"].apply(
    lambda stem: str(N4_NII_DIR / f"{stem}_desc-n4_T1w.nii.gz")
)

# Log-Jacobian map in the same MNI space as the final registered T1 image.
work_manifest["logjacobian_nii_path"] = work_manifest["output_stem"].apply(
    lambda stem: str(LOGJAC_NII_DIR / f"{stem}_desc-logJacobian.nii.gz")
)

work_manifest["logjacobian_npy_path"] = work_manifest["output_stem"].apply(
    lambda stem: str(LOGJAC_NPY_DIR / f"{stem}_desc-logJacobian.npy")
)

# Update metadata to reflect the new branch logic.
work_manifest["mni_registration_type"] = "SyN"
work_manifest["n4_bias_correction"] = True
work_manifest["jacobian_saved"] = True

# Add resume flags for the new outputs.
work_manifest["n4_nii_exists"] = work_manifest["n4_nii_path"].apply(
    lambda p: Path(p).exists()
)

work_manifest["logjacobian_nii_exists"] = work_manifest["logjacobian_nii_path"].apply(
    lambda p: Path(p).exists()
)

work_manifest["logjacobian_npy_exists"] = work_manifest["logjacobian_npy_path"].apply(
    lambda p: Path(p).exists()
)

# Save the updated planning manifest.
work_manifest.to_csv(OUTPUT_MANIFEST, index=False)

print("Branch name:", BRANCH_NAME)

print("\nN4 output folder:")
print(N4_NII_DIR)

print("\nLog-Jacobian output folders:")
print(LOGJAC_NII_DIR)
print(LOGJAC_NPY_DIR)

print("\nExisting N4 outputs:")
print(work_manifest["n4_nii_exists"].value_counts(dropna=False))

print("\nExisting log-Jacobian NIfTI outputs:")
print(work_manifest["logjacobian_nii_exists"].value_counts(dropna=False))

display(
    work_manifest[
        [
            RID_COL,
            IMAGE_ID_COL,
            LABEL_COL,
            "input_ras_nifti_path",
            "n4_nii_path",
            "mni_nii_path",
            "logjacobian_nii_path",
        ]
    ].head()
)

## 1.8. Run a small N4 pilot

run N4 bias field correction on one subject from each final group before applying it to the full cohort. This checks that the RAS input files load correctly, the N4 correction runs successfully, and the corrected NIfTI files are saved in the expected paths.

In [ ]:
try:
    import ants
    print("ANTsPy is already available.")
except ModuleNotFoundError:
    print("ANTsPy is not installed in this runtime. Installing antspyx now...")
    !pip -q install antspyx
    import ants
    print("ANTsPy installed and imported successfully.")

print("ants module:", ants)

In [ ]:
required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "input_ras_nifti_path",
    "n4_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

pilot_n4_manifest = (
    work_manifest
    .groupby(LABEL_COL, group_keys=False)
    .head(1)
    .copy()
)

pilot_n4_results = []

for _, row in pilot_n4_manifest.iterrows():
    input_path = Path(row["input_ras_nifti_path"])
    output_path = Path(row["n4_nii_path"])
    output_path.parent.mkdir(parents=True, exist_ok=True)

    print(
        f"Running N4 for RID={row[RID_COL]}, "
        f"image_id={row[IMAGE_ID_COL]}, group={row[LABEL_COL]}"
    )

    if output_path.exists():
        status = "skipped_existing"
        error = ""
    else:
        try:
            img = ants.image_read(str(input_path)).clone("float")

            # The mask is used only to guide N4 correction.
            # The saved output is still the full image, not skull-stripped.
            n4_mask = ants.get_mask(img)

            n4_img = ants.n4_bias_field_correction(
                img,
                mask=n4_mask,
                shrink_factor=4,
            )

            ants.image_write(n4_img, str(output_path))

            status = "processed"
            error = ""

        except Exception as exc:
            status = "failed"
            error = repr(exc)

    pilot_n4_results.append({
        "RID": row[RID_COL],
        "image_id": row[IMAGE_ID_COL],
        "final_group": row[LABEL_COL],
        "input_ras_nifti_path": str(input_path),
        "n4_nii_path": str(output_path),
        "status": status,
        "error": error,
    })

    print(" ->", status)

pilot_n4_report = pd.DataFrame(pilot_n4_results)

PILOT_N4_REPORT = Path(OUTPUT_REPORT).parent / "pilot_n4_report.csv"
pilot_n4_report.to_csv(PILOT_N4_REPORT, index=False)

display(pilot_n4_report)

print("Saved N4 pilot report:")
print(PILOT_N4_REPORT)

## 1.9. Set CPU threading for ANTsPy

tell ANTs/ITK to use multiple CPU threads in this Colab runtime. This should be set before importing `ants`, because ANTsPy may read these settings when the library is loaded.

In [ ]:
import os

available_cpus = os.cpu_count() or 2

# Colab CPU runtimes usually have a small number of CPU cores.
# I will use all detected cores, but cap it at 4 to avoid memory pressure.
N_THREADS = min(available_cpus, 4)

os.environ["ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS"] = str(N_THREADS)
os.environ["OMP_NUM_THREADS"] = str(N_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_THREADS)
os.environ["MKL_NUM_THREADS"] = str(N_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(N_THREADS)

print("Available CPU cores:", available_cpus)
print("Thread count set for ANTs/ITK:", N_THREADS)

## 1.10. Import ANTsPy after setting CPU threads

import ANTsPy after setting the CPU thread variables. This makes sure the N4 correction uses the thread settings from this runtime.

In [ ]:
try:
    import ants
    print("ANTsPy imported successfully.")
except ModuleNotFoundError:
    print("ANTsPy is not installed in this runtime. Installing antspyx now...")
    !pip -q install antspyx
    import ants
    print("ANTsPy installed and imported successfully.")

print("ANTsPy module:", ants)

## 1.11. Another small batch

In [ ]:
BATCH_SIZE = 4

# Refresh the N4 existence flag because the pilot files now exist.
work_manifest["n4_nii_exists"] = work_manifest["n4_nii_path"].apply(
    lambda p: Path(p).exists()
)

small_n4_batch = (
    work_manifest
    .loc[~work_manifest["n4_nii_exists"]]
    .head(BATCH_SIZE)
    .copy()
)

print("Requested batch size:", BATCH_SIZE)
print("Subjects selected for this batch:", len(small_n4_batch))
print("N4 files already existing:", int(work_manifest["n4_nii_exists"].sum()))
print("N4 files still missing before this batch:", int((~work_manifest["n4_nii_exists"]).sum()))

small_n4_results = []

for batch_idx, (_, row) in enumerate(small_n4_batch.iterrows(), start=1):
    input_path = Path(row["input_ras_nifti_path"])
    output_path = Path(row["n4_nii_path"])
    output_path.parent.mkdir(parents=True, exist_ok=True)

    print(
        f"[{batch_idx}/{len(small_n4_batch)}] "
        f"Running N4 for RID={row[RID_COL]}, "
        f"image_id={row[IMAGE_ID_COL]}, group={row[LABEL_COL]}"
    )

    if output_path.exists():
        status = "skipped_existing"
        error = ""
    else:
        try:
            img = ants.image_read(str(input_path)).clone("float")

            # This mask only guides N4 correction.
            # The saved image remains a full non-skull-stripped image.
            n4_mask = ants.get_mask(img)

            n4_img = ants.n4_bias_field_correction(
                img,
                mask=n4_mask,
                shrink_factor=4,
            )

            ants.image_write(n4_img, str(output_path))

            status = "processed"
            error = ""

        except Exception as exc:
            status = "failed"
            error = repr(exc)

    small_n4_results.append({
        "RID": row[RID_COL],
        "image_id": row[IMAGE_ID_COL],
        "final_group": row[LABEL_COL],
        "input_ras_nifti_path": str(input_path),
        "n4_nii_path": str(output_path),
        "status": status,
        "error": error,
    })

    print(" ->", status)

small_n4_report = pd.DataFrame(small_n4_results)

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
SMALL_N4_REPORT = Path(OUTPUT_REPORT).parent / f"small_batch_n4_report_{timestamp}.csv"

small_n4_report.to_csv(SMALL_N4_REPORT, index=False)

# Refresh the existence flag after the batch.
work_manifest["n4_nii_exists"] = work_manifest["n4_nii_path"].apply(
    lambda p: Path(p).exists()
)

display(small_n4_report)

print("Saved small N4 batch report:")
print(SMALL_N4_REPORT)

print("\nN4 completion after this batch:")
print(work_manifest["n4_nii_exists"].value_counts(dropna=False))

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

print("Python executable:")
print(shutil.which("python") or "Not found")

print("\nCPU cores detected:")
print(os.cpu_count())

print("\nThread-related environment variables:")
for var in [
    "ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS",
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    print(f"{var}: {os.environ.get(var, 'not set')}")

print("\nMemory information:")
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    available_gb = psutil.virtual_memory().available / (1024 ** 3)
    print(f"Total RAM: {ram_gb:.2f} GB")
    print(f"Available RAM: {available_gb:.2f} GB")
except ModuleNotFoundError:
    print("psutil is not installed, so RAM could not be checked from Python.")

print("\nGPU information:")
if shutil.which("nvidia-smi") is None:
    print("nvidia-smi not found. No NVIDIA GPU is visible from this session.")
else:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    )
    print(result.stdout)

## 1.12. Run N4 for the remaining subjects

run N4 bias field correction only for subjects whose N4 output file does not already exist. This makes the batch resumable: the pilot and any already-processed images will be skipped automatically.

In [ ]:
import time

OVERWRITE_N4 = False
CHECKPOINT_EVERY = 10

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "input_ras_nifti_path",
    "n4_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

assert "ants" in globals(), "ANTsPy is not imported. Run the ANTsPy import cell first."

# Refresh the resume flag before selecting the remaining subjects.
work_manifest["n4_nii_exists"] = work_manifest["n4_nii_path"].apply(
    lambda p: Path(p).exists()
)

if OVERWRITE_N4:
    remaining_n4_manifest = work_manifest.copy()
else:
    remaining_n4_manifest = work_manifest.loc[~work_manifest["n4_nii_exists"]].copy()

print("Total subjects in Route B cohort:", len(work_manifest))
print("Existing N4 outputs before this run:", int(work_manifest["n4_nii_exists"].sum()))
print("Subjects selected for N4 now:", len(remaining_n4_manifest))

N4_REMAINING_REPORT = Path(OUTPUT_REPORT).parent / "full_n4_remaining_report.csv"

n4_results = []
start_time = time.time()

for run_idx, (_, row) in enumerate(remaining_n4_manifest.iterrows(), start=1):
    input_path = Path(row["input_ras_nifti_path"])
    output_path = Path(row["n4_nii_path"])
    output_path.parent.mkdir(parents=True, exist_ok=True)

    print(
        f"[{run_idx}/{len(remaining_n4_manifest)}] "
        f"RID={row[RID_COL]} image_id={row[IMAGE_ID_COL]} group={row[LABEL_COL]}"
    )

    if output_path.exists() and not OVERWRITE_N4:
        status = "skipped_existing"
        error = ""
        elapsed_seconds = 0.0

    else:
        subject_start = time.time()

        try:
            img = ants.image_read(str(input_path)).clone("float")

            # This mask only guides the bias-field estimation.
            # The saved N4 image is still the full non-skull-stripped image.
            n4_mask = ants.get_mask(img)

            n4_img = ants.n4_bias_field_correction(
                img,
                mask=n4_mask,
                shrink_factor=4,
            )

            ants.image_write(n4_img, str(output_path))

            status = "processed"
            error = ""

        except Exception as exc:
            status = "failed"
            error = repr(exc)

        elapsed_seconds = time.time() - subject_start

    n4_results.append({
        "RID": row[RID_COL],
        "image_id": row[IMAGE_ID_COL],
        "final_group": row[LABEL_COL],
        "input_ras_nifti_path": str(input_path),
        "n4_nii_path": str(output_path),
        "status": status,
        "elapsed_seconds": round(elapsed_seconds, 2),
        "error": error,
    })

    print(f" -> {status} ({elapsed_seconds / 60:.2f} min)")

    if run_idx % CHECKPOINT_EVERY == 0:
        pd.DataFrame(n4_results).to_csv(N4_REMAINING_REPORT, index=False)

        work_manifest["n4_nii_exists"] = work_manifest["n4_nii_path"].apply(
            lambda p: Path(p).exists()
        )
        work_manifest.to_csv(OUTPUT_MANIFEST, index=False)

        elapsed_total = time.time() - start_time
        processed_so_far = len(n4_results)
        avg_seconds = elapsed_total / processed_so_far
        remaining_subjects = len(remaining_n4_manifest) - processed_so_far
        estimated_remaining_hours = (remaining_subjects * avg_seconds) / 3600

        print(f"Checkpoint saved after {run_idx} subjects.")
        print(f"Estimated remaining time: {estimated_remaining_hours:.2f} hours")

n4_full_report = pd.DataFrame(n4_results)
n4_full_report.to_csv(N4_REMAINING_REPORT, index=False)

work_manifest["n4_nii_exists"] = work_manifest["n4_nii_path"].apply(
    lambda p: Path(p).exists()
)
work_manifest.to_csv(OUTPUT_MANIFEST, index=False)

print("\nN4 run complete.")
print("Saved N4 report:")
print(N4_REMAINING_REPORT)

print("\nCurrent N4 completion:")
print(work_manifest["n4_nii_exists"].value_counts(dropna=False))

display(n4_full_report["status"].value_counts().to_frame("count"))

failed_n4 = n4_full_report[n4_full_report["status"].eq("failed")]
print("\nFailed subjects:", len(failed_n4))

if len(failed_n4) > 0:
    display(failed_n4[[RID_COL if RID_COL in failed_n4.columns else "RID", "image_id", "final_group", "error"]].head(20))

## 1.13. Check N4 completion and file validity

check every N4 output file after the full batch. For each subject, confirm that the N4 file exists, that ANTs can load it, and that the N4 image has the same shape, spacing, orientation, and direction as the original RAS input image. This verifies that N4 corrected intensity without accidentally changing image geometry.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

assert "ants" in globals(), "ANTsPy is not imported. Run the ANTsPy import cell first."
assert "work_manifest" in globals(), "work_manifest is not available."

required_cols = ["input_ras_nifti_path", "n4_nii_path", RID_COL, IMAGE_ID_COL, LABEL_COL]
missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

n4_qc_rows = []

for idx, row in work_manifest.iterrows():
    input_path = Path(row["input_ras_nifti_path"])
    n4_path = Path(row["n4_nii_path"])

    qc_row = {
        "RID": row[RID_COL],
        "image_id": row[IMAGE_ID_COL],
        "final_group": row[LABEL_COL],
        "input_ras_nifti_path": str(input_path),
        "n4_nii_path": str(n4_path),
        "input_exists": input_path.exists(),
        "n4_exists": n4_path.exists(),
        "input_loadable": False,
        "n4_loadable": False,
        "same_shape": False,
        "same_spacing": False,
        "same_orientation": False,
        "same_direction": False,
        "n4_has_finite_values": False,
        "n4_has_nonzero_variation": False,
        "input_shape": None,
        "n4_shape": None,
        "input_spacing": None,
        "n4_spacing": None,
        "input_orientation": None,
        "n4_orientation": None,
        "n4_min": np.nan,
        "n4_max": np.nan,
        "n4_mean": np.nan,
        "error": "",
    }

    try:
        if not input_path.exists():
            raise FileNotFoundError(f"Input RAS file is missing: {input_path}")

        if not n4_path.exists():
            raise FileNotFoundError(f"N4 output file is missing: {n4_path}")

        input_img = ants.image_read(str(input_path))
        n4_img = ants.image_read(str(n4_path))

        qc_row["input_loadable"] = True
        qc_row["n4_loadable"] = True

        qc_row["input_shape"] = tuple(input_img.shape)
        qc_row["n4_shape"] = tuple(n4_img.shape)

        qc_row["input_spacing"] = tuple(round(x, 6) for x in input_img.spacing)
        qc_row["n4_spacing"] = tuple(round(x, 6) for x in n4_img.spacing)

        qc_row["input_orientation"] = ants.get_orientation(input_img)
        qc_row["n4_orientation"] = ants.get_orientation(n4_img)

        qc_row["same_shape"] = tuple(input_img.shape) == tuple(n4_img.shape)

        qc_row["same_spacing"] = np.allclose(
            np.array(input_img.spacing),
            np.array(n4_img.spacing),
            atol=1e-5,
        )

        qc_row["same_orientation"] = (
            ants.get_orientation(input_img) == ants.get_orientation(n4_img)
        )

        qc_row["same_direction"] = np.allclose(
            np.array(input_img.direction),
            np.array(n4_img.direction),
            atol=1e-5,
        )

        n4_array = n4_img.numpy()

        qc_row["n4_has_finite_values"] = bool(np.isfinite(n4_array).all())
        qc_row["n4_has_nonzero_variation"] = bool(np.nanstd(n4_array) > 0)

        qc_row["n4_min"] = float(np.nanmin(n4_array))
        qc_row["n4_max"] = float(np.nanmax(n4_array))
        qc_row["n4_mean"] = float(np.nanmean(n4_array))

    except Exception as exc:
        qc_row["error"] = repr(exc)

    n4_qc_rows.append(qc_row)

    if (idx + 1) % 100 == 0:
        print(f"Checked {idx + 1}/{len(work_manifest)} files")

n4_qc_report = pd.DataFrame(n4_qc_rows)

N4_QC_REPORT = Path(OUTPUT_REPORT).parent / "full_n4_completion_and_geometry_qc.csv"
n4_qc_report.to_csv(N4_QC_REPORT, index=False)

print("Saved N4 QC report:")
print(N4_QC_REPORT)

print("\nN4 existence:")
print(n4_qc_report["n4_exists"].value_counts(dropna=False))

print("\nN4 loadability:")
print(n4_qc_report["n4_loadable"].value_counts(dropna=False))

print("\nGeometry checks:")
for col in ["same_shape", "same_spacing", "same_orientation", "same_direction"]:
    print(f"\n{col}:")
    print(n4_qc_report[col].value_counts(dropna=False))

print("\nIntensity sanity checks:")
for col in ["n4_has_finite_values", "n4_has_nonzero_variation"]:
    print(f"\n{col}:")
    print(n4_qc_report[col].value_counts(dropna=False))

failed_n4_qc = n4_qc_report[
    (~n4_qc_report["n4_exists"])
    | (~n4_qc_report["n4_loadable"])
    | (~n4_qc_report["same_shape"])
    | (~n4_qc_report["same_spacing"])
    | (~n4_qc_report["same_orientation"])
    | (~n4_qc_report["same_direction"])
    | (~n4_qc_report["n4_has_finite_values"])
    | (~n4_qc_report["n4_has_nonzero_variation"])
]

print("\nSubjects failing any N4 QC check:", len(failed_n4_qc))

if len(failed_n4_qc) > 0:
    display(
        failed_n4_qc[
            [
                "RID",
                "image_id",
                "final_group",
                "n4_exists",
                "n4_loadable",
                "same_shape",
                "same_spacing",
                "same_orientation",
                "same_direction",
                "n4_has_finite_values",
                "n4_has_nonzero_variation",
                "error",
            ]
        ].head(30)
    )
else:
    print("All N4 outputs passed completion, loadability, geometry, and basic intensity checks.")

## 1.14. Load and inspect the MNI template

I have finished N4 correction for all 1063 images, so the next step is to load the MNI template that will define the target space for nonlinear registration. only inspect the template here: its path, shape, voxel spacing, orientation, and basic intensity range. not register any subject yet.

## 1.15. Download the MNI152NLin2009cSym T1w template

download the MNI152NLin2009cSym T1-weighted template from TemplateFlow and copy it into my project `templates` folder. This template will be used as the fixed reference image for the next nonlinear MNI registration pilot.

In [ ]:
import shutil

TEMPLATE_DIR = PROJECT_DIR / "templates"
TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)

try:
    import templateflow.api as tflow
    print("TemplateFlow is already available.")
except ModuleNotFoundError:
    print("TemplateFlow is not installed. Installing it now...")
    !pip -q install templateflow
    import templateflow.api as tflow
    print("TemplateFlow installed successfully.")

template_result = tflow.get(
    "MNI152NLin2009cSym",
    resolution=1,
    suffix="T1w",
    raise_empty=True,
)

# TemplateFlow can return either a single path or a list-like result.
if isinstance(template_result, (list, tuple)):
    if len(template_result) != 1:
        raise RuntimeError(
            f"Expected exactly one template file, but got {len(template_result)}:\n"
            f"{template_result}"
        )
    template_source_path = Path(template_result[0])
else:
    template_source_path = Path(template_result)

MNI_TEMPLATE_PATH = TEMPLATE_DIR / template_source_path.name

shutil.copy2(template_source_path, MNI_TEMPLATE_PATH)

print("Downloaded TemplateFlow source:")
print(template_source_path)

print("\nCopied template into project folder:")
print(MNI_TEMPLATE_PATH)

print("\nFile exists:", MNI_TEMPLATE_PATH.exists())
print("File size MB:", round(MNI_TEMPLATE_PATH.stat().st_size / (1024 ** 2), 2))

## 1.16. Inspect the downloaded MNI template

load the downloaded MNI152NLin2009cSym template and check its basic properties. This confirms that the template is readable and tells me the target shape, spacing, and orientation that the nonlinear MNI registration will use.

In [ ]:
assert "ants" in globals(), "ANTsPy is not imported. Run the ANTsPy import cell first."
assert "MNI_TEMPLATE_PATH" in globals(), "MNI_TEMPLATE_PATH is not defined. Run the template download cell first."
assert Path(MNI_TEMPLATE_PATH).exists(), f"Template file does not exist: {MNI_TEMPLATE_PATH}"

template_img = ants.image_read(str(MNI_TEMPLATE_PATH)).clone("float")
template_array = template_img.numpy()

print("MNI template path:")
print(MNI_TEMPLATE_PATH)

print("\nTemplate properties:")
print("Shape:", template_img.shape)
print("Spacing:", template_img.spacing)
print("Orientation:", ants.get_orientation(template_img))

print("\nTemplate intensity summary:")
print("Min:", float(np.nanmin(template_array)))
print("Max:", float(np.nanmax(template_array)))
print("Mean:", float(np.nanmean(template_array)))
print("Finite values:", bool(np.isfinite(template_array).all()))
print("Nonzero variation:", bool(np.nanstd(template_array) > 0))

## 1.17. Run one-subject nonlinear MNI registration pilot

register one N4-corrected image to the MNI152NLin2009cSym template using SyN nonlinear registration. save two outputs: the MNI-space T1 image and the log-Jacobian map from the nonlinear warp. This is only a one-subject pilot, because SyN registration is much slower than N4.

In [ ]:
assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "work_manifest" in globals(), "work_manifest is not available."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "n4_nii_path",
    "mni_nii_path",
    "logjacobian_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

# Refresh output-existence flags.
work_manifest["n4_nii_exists"] = work_manifest["n4_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["mni_nii_exists"] = work_manifest["mni_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["logjacobian_nii_exists"] = work_manifest["logjacobian_nii_path"].apply(lambda p: Path(p).exists())

pilot_row = (
    work_manifest
    .loc[
        work_manifest["n4_nii_exists"]
        & ~work_manifest["mni_nii_exists"]
        & ~work_manifest["logjacobian_nii_exists"]
    ]
    .head(1)
)

assert len(pilot_row) == 1, "No eligible subject found for the SyN pilot."

row = pilot_row.iloc[0]

n4_path = Path(row["n4_nii_path"])
mni_path = Path(row["mni_nii_path"])
logjacobian_path = Path(row["logjacobian_nii_path"])

mni_path.parent.mkdir(parents=True, exist_ok=True)
logjacobian_path.parent.mkdir(parents=True, exist_ok=True)

print("Running SyN pilot for:")
print("RID:", row[RID_COL])
print("Image ID:", row[IMAGE_ID_COL])
print("Group:", row[LABEL_COL])
print("Input N4 image:", n4_path)
print("MNI T1 output:", mni_path)
print("Log-Jacobian output:", logjacobian_path)

start_time = time.time()

try:
    moving_img = ants.image_read(str(n4_path)).clone("float")

    registration = ants.registration(
        fixed=template_img,
        moving=moving_img,
        type_of_transform="SyN",
        verbose=False,
    )

    mni_img = registration["warpedmovout"]
    ants.image_write(mni_img, str(mni_path))

    forward_warp = None
    for transform_path in registration["fwdtransforms"]:
        transform_name = Path(transform_path).name
        if "Warp" in transform_name and "Inverse" not in transform_name:
            forward_warp = transform_path
            break

    if forward_warp is None:
        raise RuntimeError(f"No forward warp found: {registration['fwdtransforms']}")

    logjacobian_img = ants.create_jacobian_determinant_image(
        domain_image=template_img,
        tx=forward_warp,
        do_log=True,
        geom=False,
    )

    ants.image_write(logjacobian_img, str(logjacobian_path))

    status = "processed"
    error = ""

except Exception as exc:
    status = "failed"
    error = repr(exc)

elapsed_minutes = (time.time() - start_time) / 60

pilot_syn_report = pd.DataFrame([{
    "RID": row[RID_COL],
    "image_id": row[IMAGE_ID_COL],
    "final_group": row[LABEL_COL],
    "n4_nii_path": str(n4_path),
    "mni_nii_path": str(mni_path),
    "logjacobian_nii_path": str(logjacobian_path),
    "status": status,
    "elapsed_minutes": round(elapsed_minutes, 2),
    "error": error,
}])

PILOT_SYN_REPORT = Path(OUTPUT_REPORT).parent / "pilot_syn_registration_jacobian_report.csv"
pilot_syn_report.to_csv(PILOT_SYN_REPORT, index=False)

display(pilot_syn_report)

print("Saved pilot SyN report:")
print(PILOT_SYN_REPORT)

## 1.18. Correct the SyN output paths before continuing

The one-subject SyN pilot worked, but the output paths still use old affine branch names. keep the completed N4 files as inputs, but redirect the MNI-registered T1 images and log-Jacobian maps into correctly named nonlinear SyN branch folders before running any more subjects.

In [ ]:
# Correct the branch name used for future nonlinear outputs.
BRANCH_NAME = "clean_90d_mni_n4_syn_jacobian"

assert "work_manifest" in globals(), "work_manifest is not available."
assert "output_stem" in work_manifest.columns, "work_manifest is missing output_stem."
assert "n4_nii_path" in work_manifest.columns, "work_manifest is missing n4_nii_path."

# Preserve the old output paths before replacing them.
if "old_mni_nii_path" not in work_manifest.columns and "mni_nii_path" in work_manifest.columns:
    work_manifest["old_mni_nii_path"] = work_manifest["mni_nii_path"]

if "old_logjacobian_nii_path" not in work_manifest.columns and "logjacobian_nii_path" in work_manifest.columns:
    work_manifest["old_logjacobian_nii_path"] = work_manifest["logjacobian_nii_path"]

# Create correctly named SyN output folders.
SYN_BRANCH_DIR = PROJECT_DIR / "processed" / BRANCH_NAME
SYN_MNI_DIR = SYN_BRANCH_DIR / "t1_mni_syn"
SYN_LOGJAC_DIR = SYN_BRANCH_DIR / "logjacobian_mni_syn"

for folder in [SYN_BRANCH_DIR, SYN_MNI_DIR, SYN_LOGJAC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Keep n4_nii_path unchanged because N4 is already complete.
# Redirect only the future nonlinear MNI T1 and log-Jacobian outputs.
work_manifest["mni_nii_path"] = work_manifest["output_stem"].apply(
    lambda stem: str(
        SYN_MNI_DIR / f"{stem}_space-MNI152NLin2009cSym_desc-n4Syn_T1w.nii.gz"
    )
)

work_manifest["logjacobian_nii_path"] = work_manifest["output_stem"].apply(
    lambda stem: str(
        SYN_LOGJAC_DIR / f"{stem}_space-MNI152NLin2009cSym_desc-logJacobian.nii.gz"
    )
)

# Refresh existence flags using the corrected paths.
work_manifest["mni_nii_exists"] = work_manifest["mni_nii_path"].apply(
    lambda p: Path(p).exists()
)

work_manifest["logjacobian_nii_exists"] = work_manifest["logjacobian_nii_path"].apply(
    lambda p: Path(p).exists()
)

# Save corrected planning manifest.
OUTPUT_MANIFEST = PROJECT_DIR / "manifest" / f"{BRANCH_NAME}_manifest_1063.csv"
OUTPUT_REPORT = PROJECT_DIR / "qc" / BRANCH_NAME / f"{BRANCH_NAME}_processing_report.csv"
OUTPUT_REPORT.parent.mkdir(parents=True, exist_ok=True)

work_manifest.to_csv(OUTPUT_MANIFEST, index=False)

print("Corrected branch name:")
print(BRANCH_NAME)

print("\nN4 input folder is unchanged:")
print(Path(work_manifest["n4_nii_path"].iloc[0]).parent)

print("\nCorrected SyN MNI output folder:")
print(SYN_MNI_DIR)

print("\nCorrected log-Jacobian output folder:")
print(SYN_LOGJAC_DIR)

print("\nExisting corrected MNI outputs:")
print(work_manifest["mni_nii_exists"].value_counts(dropna=False))

print("\nExisting corrected log-Jacobian outputs:")
print(work_manifest["logjacobian_nii_exists"].value_counts(dropna=False))

display(
    work_manifest[
        [
            RID_COL,
            IMAGE_ID_COL,
            LABEL_COL,
            "n4_nii_path",
            "mni_nii_path",
            "logjacobian_nii_path",
        ]
    ].head()
)

## 1.19. Rerun one-subject SyN pilot into the corrected branch

rerun the one-subject nonlinear registration pilot after correcting the output paths. This should save the registered MNI T1 image and log-Jacobian map into the `clean_90d_mni_n4_syn_jacobian` branch, while still using the already completed N4 image as input.

In [ ]:
import time
from pathlib import Path
import pandas as pd

assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "work_manifest" in globals(), "work_manifest is not available."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "n4_nii_path",
    "mni_nii_path",
    "logjacobian_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

# Make sure the corrected branch name is now present in the future SyN paths.
assert work_manifest["mni_nii_path"].str.contains("clean_90d_mni_n4_syn_jacobian").all(), (
    "mni_nii_path is still not pointing to the corrected SyN branch."
)

assert work_manifest["logjacobian_nii_path"].str.contains("clean_90d_mni_n4_syn_jacobian").all(), (
    "logjacobian_nii_path is still not pointing to the corrected SyN branch."
)

# Refresh existence flags using the corrected paths.
work_manifest["n4_nii_exists"] = work_manifest["n4_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["mni_nii_exists"] = work_manifest["mni_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["logjacobian_nii_exists"] = work_manifest["logjacobian_nii_path"].apply(lambda p: Path(p).exists())

pilot_row = (
    work_manifest
    .loc[
        work_manifest["n4_nii_exists"]
        & ~work_manifest["mni_nii_exists"]
        & ~work_manifest["logjacobian_nii_exists"]
    ]
    .head(1)
)

assert len(pilot_row) == 1, "No eligible subject found for the corrected SyN pilot."

row = pilot_row.iloc[0]

n4_path = Path(row["n4_nii_path"])
mni_path = Path(row["mni_nii_path"])
logjacobian_path = Path(row["logjacobian_nii_path"])

mni_path.parent.mkdir(parents=True, exist_ok=True)
logjacobian_path.parent.mkdir(parents=True, exist_ok=True)

print("Running corrected SyN pilot for:")
print("RID:", row[RID_COL])
print("Image ID:", row[IMAGE_ID_COL])
print("Group:", row[LABEL_COL])
print("Input N4 image:", n4_path)
print("Corrected MNI T1 output:", mni_path)
print("Corrected log-Jacobian output:", logjacobian_path)

start_time = time.time()

try:
    moving_img = ants.image_read(str(n4_path)).clone("float")

    registration = ants.registration(
        fixed=template_img,
        moving=moving_img,
        type_of_transform="SyN",
        verbose=False,
    )

    mni_img = registration["warpedmovout"]
    ants.image_write(mni_img, str(mni_path))

    forward_warp = None

    for transform_path in registration["fwdtransforms"]:
        transform_name = Path(transform_path).name

        if "Warp" in transform_name and "Inverse" not in transform_name:
            forward_warp = transform_path
            break

    if forward_warp is None:
        raise RuntimeError(f"No forward warp found: {registration['fwdtransforms']}")

    logjacobian_img = ants.create_jacobian_determinant_image(
        domain_image=template_img,
        tx=forward_warp,
        do_log=True,
        geom=False,
    )

    ants.image_write(logjacobian_img, str(logjacobian_path))

    status = "processed"
    error = ""

except Exception as exc:
    status = "failed"
    error = repr(exc)

elapsed_minutes = (time.time() - start_time) / 60

corrected_pilot_syn_report = pd.DataFrame([{
    "RID": row[RID_COL],
    "image_id": row[IMAGE_ID_COL],
    "final_group": row[LABEL_COL],
    "n4_nii_path": str(n4_path),
    "mni_nii_path": str(mni_path),
    "logjacobian_nii_path": str(logjacobian_path),
    "status": status,
    "elapsed_minutes": round(elapsed_minutes, 2),
    "error": error,
}])

CORRECTED_PILOT_SYN_REPORT = OUTPUT_REPORT.parent / "corrected_pilot_syn_registration_jacobian_report.csv"
corrected_pilot_syn_report.to_csv(CORRECTED_PILOT_SYN_REPORT, index=False)

display(corrected_pilot_syn_report)

print("Saved corrected pilot SyN report:")
print(CORRECTED_PILOT_SYN_REPORT)

## 1.20. QC the corrected SyN pilot

check the one-subject SyN pilot before running the full cohort. verify that the MNI-registered T1 image and log-Jacobian map are loadable, have the same shape and spacing as the MNI template, contain finite values, and visually inspect central slices.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

assert "corrected_pilot_syn_report" in globals(), "Run the corrected SyN pilot cell first."
assert "template_img" in globals(), "template_img is not loaded."

pilot_row = corrected_pilot_syn_report.iloc[0]

mni_path = Path(pilot_row["mni_nii_path"])
logjacobian_path = Path(pilot_row["logjacobian_nii_path"])

mni_img = ants.image_read(str(mni_path))
logjacobian_img = ants.image_read(str(logjacobian_path))

mni_arr = mni_img.numpy()
logjacobian_arr = logjacobian_img.numpy()
template_arr = template_img.numpy()

pilot_syn_qc = pd.DataFrame([{
    "RID": pilot_row["RID"],
    "image_id": pilot_row["image_id"],
    "final_group": pilot_row["final_group"],
    "mni_exists": mni_path.exists(),
    "logjacobian_exists": logjacobian_path.exists(),
    "mni_loadable": True,
    "logjacobian_loadable": True,
    "template_shape": tuple(template_img.shape),
    "mni_shape": tuple(mni_img.shape),
    "logjacobian_shape": tuple(logjacobian_img.shape),
    "mni_matches_template_shape": tuple(mni_img.shape) == tuple(template_img.shape),
    "logjacobian_matches_template_shape": tuple(logjacobian_img.shape) == tuple(template_img.shape),
    "template_spacing": tuple(template_img.spacing),
    "mni_spacing": tuple(mni_img.spacing),
    "logjacobian_spacing": tuple(logjacobian_img.spacing),
    "mni_matches_template_spacing": np.allclose(mni_img.spacing, template_img.spacing, atol=1e-5),
    "logjacobian_matches_template_spacing": np.allclose(logjacobian_img.spacing, template_img.spacing, atol=1e-5),
    "template_orientation": ants.get_orientation(template_img),
    "mni_orientation": ants.get_orientation(mni_img),
    "logjacobian_orientation": ants.get_orientation(logjacobian_img),
    "mni_finite": bool(np.isfinite(mni_arr).all()),
    "logjacobian_finite": bool(np.isfinite(logjacobian_arr).all()),
    "mni_nonzero_variation": bool(np.nanstd(mni_arr) > 0),
    "logjacobian_nonzero_variation": bool(np.nanstd(logjacobian_arr) > 0),
    "mni_min": float(np.nanmin(mni_arr)),
    "mni_max": float(np.nanmax(mni_arr)),
    "mni_mean": float(np.nanmean(mni_arr)),
    "logjacobian_min": float(np.nanmin(logjacobian_arr)),
    "logjacobian_max": float(np.nanmax(logjacobian_arr)),
    "logjacobian_mean": float(np.nanmean(logjacobian_arr)),
}])

PILOT_SYN_QC_REPORT = OUTPUT_REPORT.parent / "corrected_pilot_syn_qc_report.csv"
pilot_syn_qc.to_csv(PILOT_SYN_QC_REPORT, index=False)

display(pilot_syn_qc)

print("Saved corrected SyN pilot QC report:")
print(PILOT_SYN_QC_REPORT)

# Visual QC using central slices.
x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

logjac_low, logjac_high = np.nanpercentile(logjacobian_arr, [1, 99])

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

images = [
    [template_arr[x_mid, :, :], mni_arr[x_mid, :, :], logjacobian_arr[x_mid, :, :]],
    [template_arr[:, y_mid, :], mni_arr[:, y_mid, :], logjacobian_arr[:, y_mid, :]],
    [template_arr[:, :, z_mid], mni_arr[:, :, z_mid], logjacobian_arr[:, :, z_mid]],
]

titles = [
    ["Template sagittal", "Registered T1 sagittal", "Log-Jacobian sagittal"],
    ["Template coronal", "Registered T1 coronal", "Log-Jacobian coronal"],
    ["Template axial", "Registered T1 axial", "Log-Jacobian axial"],
]

for row_idx in range(3):
    for col_idx in range(3):
        ax = axes[row_idx, col_idx]

        if col_idx == 2:
            ax.imshow(
                np.rot90(images[row_idx][col_idx]),
                vmin=logjac_low,
                vmax=logjac_high,
            )
        else:
            ax.imshow(np.rot90(images[row_idx][col_idx]), cmap="gray")

        ax.set_title(titles[row_idx][col_idx])
        ax.axis("off")

plt.tight_layout()

PILOT_SYN_QC_FIGURE = OUTPUT_REPORT.parent / "corrected_pilot_syn_visual_qc.png"
plt.savefig(PILOT_SYN_QC_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved corrected SyN pilot visual QC figure:")
print(PILOT_SYN_QC_FIGURE)

## 1.21. Run a small SyN + Jacobian batch

run nonlinear MNI registration and log-Jacobian saving for a small batch of subjects whose corrected SyN outputs do not already exist. This checks that the pilot result generalizes beyond one subject before launching the full cohort.

In [ ]:
import time
from pathlib import Path
import pandas as pd

BATCH_SIZE = 8
CHECKPOINT_EVERY = 2
OVERWRITE_SYN = False

assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "work_manifest" in globals(), "work_manifest is not available."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "n4_nii_path",
    "mni_nii_path",
    "logjacobian_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

assert work_manifest["mni_nii_path"].str.contains("clean_90d_mni_n4_syn_jacobian").all(), (
    "mni_nii_path is not pointing to the corrected SyN branch."
)

assert work_manifest["logjacobian_nii_path"].str.contains("clean_90d_mni_n4_syn_jacobian").all(), (
    "logjacobian_nii_path is not pointing to the corrected SyN branch."
)

# Refresh resume flags.
work_manifest["n4_nii_exists"] = work_manifest["n4_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["mni_nii_exists"] = work_manifest["mni_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["logjacobian_nii_exists"] = work_manifest["logjacobian_nii_path"].apply(lambda p: Path(p).exists())

if OVERWRITE_SYN:
    small_syn_batch = work_manifest.loc[work_manifest["n4_nii_exists"]].head(BATCH_SIZE).copy()
else:
    small_syn_batch = (
        work_manifest
        .loc[
            work_manifest["n4_nii_exists"]
            & ~work_manifest["mni_nii_exists"]
            & ~work_manifest["logjacobian_nii_exists"]
        ]
        .head(BATCH_SIZE)
        .copy()
    )

print("Requested batch size:", BATCH_SIZE)
print("Subjects selected:", len(small_syn_batch))
print("Existing corrected MNI outputs before this batch:", int(work_manifest["mni_nii_exists"].sum()))
print("Existing corrected log-Jacobian outputs before this batch:", int(work_manifest["logjacobian_nii_exists"].sum()))

small_syn_results = []
start_time = time.time()

SMALL_SYN_REPORT = OUTPUT_REPORT.parent / "small_batch_syn_registration_jacobian_report.csv"

for run_idx, (_, row) in enumerate(small_syn_batch.iterrows(), start=1):
    n4_path = Path(row["n4_nii_path"])
    mni_path = Path(row["mni_nii_path"])
    logjacobian_path = Path(row["logjacobian_nii_path"])

    mni_path.parent.mkdir(parents=True, exist_ok=True)
    logjacobian_path.parent.mkdir(parents=True, exist_ok=True)

    print(
        f"[{run_idx}/{len(small_syn_batch)}] "
        f"RID={row[RID_COL]} image_id={row[IMAGE_ID_COL]} group={row[LABEL_COL]}"
    )

    if (
        mni_path.exists()
        and logjacobian_path.exists()
        and not OVERWRITE_SYN
    ):
        status = "skipped_existing"
        error = ""
        elapsed_minutes = 0.0

    else:
        subject_start = time.time()

        try:
            moving_img = ants.image_read(str(n4_path)).clone("float")

            registration = ants.registration(
                fixed=template_img,
                moving=moving_img,
                type_of_transform="SyN",
                verbose=False,
            )

            mni_img = registration["warpedmovout"]
            ants.image_write(mni_img, str(mni_path))

            forward_warp = None

            for transform_path in registration["fwdtransforms"]:
                transform_name = Path(transform_path).name

                if "Warp" in transform_name and "Inverse" not in transform_name:
                    forward_warp = transform_path
                    break

            if forward_warp is None:
                raise RuntimeError(f"No forward warp found: {registration['fwdtransforms']}")

            logjacobian_img = ants.create_jacobian_determinant_image(
                domain_image=template_img,
                tx=forward_warp,
                do_log=True,
                geom=False,
            )

            ants.image_write(logjacobian_img, str(logjacobian_path))

            status = "processed"
            error = ""

        except Exception as exc:
            status = "failed"
            error = repr(exc)

        elapsed_minutes = (time.time() - subject_start) / 60

    small_syn_results.append({
        "RID": row[RID_COL],
        "image_id": row[IMAGE_ID_COL],
        "final_group": row[LABEL_COL],
        "n4_nii_path": str(n4_path),
        "mni_nii_path": str(mni_path),
        "logjacobian_nii_path": str(logjacobian_path),
        "status": status,
        "elapsed_minutes": round(elapsed_minutes, 2),
        "error": error,
    })

    print(f" -> {status} ({elapsed_minutes:.2f} min)")

    if run_idx % CHECKPOINT_EVERY == 0:
        pd.DataFrame(small_syn_results).to_csv(SMALL_SYN_REPORT, index=False)
        print(f"Checkpoint saved after {run_idx} subjects.")

small_syn_report = pd.DataFrame(small_syn_results)
small_syn_report.to_csv(SMALL_SYN_REPORT, index=False)

work_manifest["mni_nii_exists"] = work_manifest["mni_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["logjacobian_nii_exists"] = work_manifest["logjacobian_nii_path"].apply(lambda p: Path(p).exists())
work_manifest.to_csv(OUTPUT_MANIFEST, index=False)

print("\nSaved small SyN batch report:")
print(SMALL_SYN_REPORT)

print("\nCorrected MNI output completion:")
print(work_manifest["mni_nii_exists"].value_counts(dropna=False))

print("\nCorrected log-Jacobian output completion:")
print(work_manifest["logjacobian_nii_exists"].value_counts(dropna=False))

display(small_syn_report)

## 1.22. Run full SyN + Jacobian batch

run nonlinear MNI registration and log-Jacobian saving for all remaining subjects. The batch is resumable: subjects with both corrected MNI T1 and log-Jacobian outputs already present will be skipped. Each output is first written to a temporary file and then moved into its final path, so an interruption during writing is less likely to leave a corrupted final file.

In [ ]:
import time
from pathlib import Path
import pandas as pd

OVERWRITE_SYN = False
CHECKPOINT_EVERY = 1

assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "work_manifest" in globals(), "work_manifest is not available."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "n4_nii_path",
    "mni_nii_path",
    "logjacobian_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

assert work_manifest["mni_nii_path"].str.contains("clean_90d_mni_n4_syn_jacobian").all(), (
    "mni_nii_path is not pointing to the corrected SyN branch."
)

assert work_manifest["logjacobian_nii_path"].str.contains("clean_90d_mni_n4_syn_jacobian").all(), (
    "logjacobian_nii_path is not pointing to the corrected SyN branch."
)

# Refresh resume flags.
work_manifest["n4_nii_exists"] = work_manifest["n4_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["mni_nii_exists"] = work_manifest["mni_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["logjacobian_nii_exists"] = work_manifest["logjacobian_nii_path"].apply(lambda p: Path(p).exists())

if OVERWRITE_SYN:
    remaining_syn_manifest = work_manifest.loc[work_manifest["n4_nii_exists"]].copy()
else:
    remaining_syn_manifest = (
        work_manifest
        .loc[
            work_manifest["n4_nii_exists"]
            & (
                ~work_manifest["mni_nii_exists"]
                | ~work_manifest["logjacobian_nii_exists"]
            )
        ]
        .copy()
    )

FULL_SYN_REPORT = OUTPUT_REPORT.parent / "full_syn_registration_jacobian_report.csv"

print("Total subjects:", len(work_manifest))
print("Existing corrected MNI outputs:", int(work_manifest["mni_nii_exists"].sum()))
print("Existing corrected log-Jacobian outputs:", int(work_manifest["logjacobian_nii_exists"].sum()))
print("Subjects selected for this run:", len(remaining_syn_manifest))
print("Full SyN report path:", FULL_SYN_REPORT)

syn_results = []
start_time = time.time()

for run_idx, (_, row) in enumerate(remaining_syn_manifest.iterrows(), start=1):
    n4_path = Path(row["n4_nii_path"])
    mni_path = Path(row["mni_nii_path"])
    logjacobian_path = Path(row["logjacobian_nii_path"])

    mni_path.parent.mkdir(parents=True, exist_ok=True)
    logjacobian_path.parent.mkdir(parents=True, exist_ok=True)

    tmp_mni_path = mni_path.with_name(mni_path.name.replace(".nii.gz", "_tmp.nii.gz"))
    tmp_logjacobian_path = logjacobian_path.with_name(
        logjacobian_path.name.replace(".nii.gz", "_tmp.nii.gz")
    )

    print(
        f"[{run_idx}/{len(remaining_syn_manifest)}] "
        f"RID={row[RID_COL]} image_id={row[IMAGE_ID_COL]} group={row[LABEL_COL]}"
    )

    if (
        mni_path.exists()
        and logjacobian_path.exists()
        and not OVERWRITE_SYN
    ):
        status = "skipped_existing"
        error = ""
        elapsed_minutes = 0.0

    else:
        subject_start = time.time()

        try:
            for tmp_path in [tmp_mni_path, tmp_logjacobian_path]:
                if tmp_path.exists():
                    tmp_path.unlink()

            moving_img = ants.image_read(str(n4_path)).clone("float")

            registration = ants.registration(
                fixed=template_img,
                moving=moving_img,
                type_of_transform="SyN",
                verbose=False,
            )

            mni_img = registration["warpedmovout"]
            ants.image_write(mni_img, str(tmp_mni_path))
            tmp_mni_path.replace(mni_path)

            forward_warp = None

            for transform_path in registration["fwdtransforms"]:
                transform_name = Path(transform_path).name

                if "Warp" in transform_name and "Inverse" not in transform_name:
                    forward_warp = transform_path
                    break

            if forward_warp is None:
                raise RuntimeError(f"No forward warp found: {registration['fwdtransforms']}")

            logjacobian_img = ants.create_jacobian_determinant_image(
                domain_image=template_img,
                tx=forward_warp,
                do_log=True,
                geom=False,
            )

            ants.image_write(logjacobian_img, str(tmp_logjacobian_path))
            tmp_logjacobian_path.replace(logjacobian_path)

            status = "processed"
            error = ""

        except Exception as exc:
            status = "failed"
            error = repr(exc)

            for tmp_path in [tmp_mni_path, tmp_logjacobian_path]:
                if tmp_path.exists():
                    tmp_path.unlink()

        elapsed_minutes = (time.time() - subject_start) / 60

    syn_results.append({
        "RID": row[RID_COL],
        "image_id": row[IMAGE_ID_COL],
        "final_group": row[LABEL_COL],
        "n4_nii_path": str(n4_path),
        "mni_nii_path": str(mni_path),
        "logjacobian_nii_path": str(logjacobian_path),
        "status": status,
        "elapsed_minutes": round(elapsed_minutes, 2),
        "error": error,
    })

    print(f" -> {status} ({elapsed_minutes:.2f} min)")

    if run_idx % CHECKPOINT_EVERY == 0:
        full_syn_report = pd.DataFrame(syn_results)
        full_syn_report.to_csv(FULL_SYN_REPORT, index=False)

        work_manifest["mni_nii_exists"] = work_manifest["mni_nii_path"].apply(lambda p: Path(p).exists())
        work_manifest["logjacobian_nii_exists"] = work_manifest["logjacobian_nii_path"].apply(lambda p: Path(p).exists())
        work_manifest.to_csv(OUTPUT_MANIFEST, index=False)

        elapsed_total = time.time() - start_time
        avg_minutes = (elapsed_total / 60) / len(syn_results)
        remaining_subjects = len(remaining_syn_manifest) - len(syn_results)
        estimated_remaining_hours = (remaining_subjects * avg_minutes) / 60

        print("Checkpoint saved.")
        print(f"Average time per subject: {avg_minutes:.2f} min")
        print(f"Estimated remaining time: {estimated_remaining_hours:.2f} hours")

full_syn_report = pd.DataFrame(syn_results)
full_syn_report.to_csv(FULL_SYN_REPORT, index=False)

work_manifest["mni_nii_exists"] = work_manifest["mni_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["logjacobian_nii_exists"] = work_manifest["logjacobian_nii_path"].apply(lambda p: Path(p).exists())
work_manifest.to_csv(OUTPUT_MANIFEST, index=False)

print("\nFull SyN run finished or stopped naturally.")
print("Saved full SyN report:")
print(FULL_SYN_REPORT)

print("\nCorrected MNI output completion:")
print(work_manifest["mni_nii_exists"].value_counts(dropna=False))

print("\nCorrected log-Jacobian output completion:")
print(work_manifest["logjacobian_nii_exists"].value_counts(dropna=False))

display(full_syn_report["status"].value_counts().to_frame("count"))

failed_syn = full_syn_report[full_syn_report["status"].eq("failed")]
print("\nFailed subjects in this run:", len(failed_syn))

if len(failed_syn) > 0:
    display(failed_syn[["RID", "image_id", "final_group", "error"]].head(30))

## 1.23. QC the full SyN registration and Jacobian outputs

check the full nonlinear registration outputs before moving to masking or intensity normalization. verify that every registered MNI T1 image and every log-Jacobian map exists, can be loaded, matches the MNI template shape and spacing, has finite values, and contains nonzero variation.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import time

assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "work_manifest" in globals(), "work_manifest is not available."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "n4_nii_path",
    "mni_nii_path",
    "logjacobian_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

FULL_SYN_QC_REPORT = OUTPUT_REPORT.parent / "full_syn_registration_jacobian_qc_report.csv"

template_shape = tuple(template_img.shape)
template_spacing = tuple(template_img.spacing)
template_orientation = ants.get_orientation(template_img)
template_direction = np.array(template_img.direction)

qc_rows = []
start_time = time.time()

print("Starting full SyN + Jacobian QC")
print("Subjects to check:", len(work_manifest))
print("Template shape:", template_shape)
print("Template spacing:", template_spacing)
print("Template orientation:", template_orientation)
print("QC report will be saved to:")
print(FULL_SYN_QC_REPORT)

for idx, (_, row) in enumerate(work_manifest.iterrows(), start=1):
    rid = row[RID_COL]
    image_id = row[IMAGE_ID_COL]
    final_group = row[LABEL_COL]

    mni_path = Path(row["mni_nii_path"])
    logjacobian_path = Path(row["logjacobian_nii_path"])

    mni_exists = mni_path.exists()
    logjacobian_exists = logjacobian_path.exists()

    qc_record = {
        "RID": rid,
        "image_id": image_id,
        "final_group": final_group,
        "mni_nii_path": str(mni_path),
        "logjacobian_nii_path": str(logjacobian_path),
        "mni_exists": mni_exists,
        "logjacobian_exists": logjacobian_exists,
        "mni_loadable": False,
        "logjacobian_loadable": False,
        "mni_shape": None,
        "logjacobian_shape": None,
        "mni_spacing": None,
        "logjacobian_spacing": None,
        "mni_orientation": None,
        "logjacobian_orientation": None,
        "mni_matches_template_shape": False,
        "logjacobian_matches_template_shape": False,
        "mni_matches_template_spacing": False,
        "logjacobian_matches_template_spacing": False,
        "mni_matches_template_orientation": False,
        "logjacobian_matches_template_orientation": False,
        "mni_matches_template_direction": False,
        "logjacobian_matches_template_direction": False,
        "mni_finite": False,
        "logjacobian_finite": False,
        "mni_nonzero_variation": False,
        "logjacobian_nonzero_variation": False,
        "mni_min": np.nan,
        "mni_max": np.nan,
        "mni_mean": np.nan,
        "mni_std": np.nan,
        "logjacobian_min": np.nan,
        "logjacobian_max": np.nan,
        "logjacobian_mean": np.nan,
        "logjacobian_std": np.nan,
        "qc_pass": False,
        "error": "",
    }

    try:
        if mni_exists:
            mni_img = ants.image_read(str(mni_path))
            mni_arr = mni_img.numpy()

            qc_record["mni_loadable"] = True
            qc_record["mni_shape"] = tuple(mni_img.shape)
            qc_record["mni_spacing"] = tuple(mni_img.spacing)
            qc_record["mni_orientation"] = ants.get_orientation(mni_img)
            qc_record["mni_matches_template_shape"] = tuple(mni_img.shape) == template_shape
            qc_record["mni_matches_template_spacing"] = np.allclose(mni_img.spacing, template_spacing, atol=1e-5)
            qc_record["mni_matches_template_orientation"] = ants.get_orientation(mni_img) == template_orientation
            qc_record["mni_matches_template_direction"] = np.allclose(
                np.array(mni_img.direction),
                template_direction,
                atol=1e-5,
            )
            qc_record["mni_finite"] = bool(np.isfinite(mni_arr).all())
            qc_record["mni_nonzero_variation"] = bool(np.nanstd(mni_arr) > 0)
            qc_record["mni_min"] = float(np.nanmin(mni_arr))
            qc_record["mni_max"] = float(np.nanmax(mni_arr))
            qc_record["mni_mean"] = float(np.nanmean(mni_arr))
            qc_record["mni_std"] = float(np.nanstd(mni_arr))

        if logjacobian_exists:
            logjacobian_img = ants.image_read(str(logjacobian_path))
            logjacobian_arr = logjacobian_img.numpy()

            qc_record["logjacobian_loadable"] = True
            qc_record["logjacobian_shape"] = tuple(logjacobian_img.shape)
            qc_record["logjacobian_spacing"] = tuple(logjacobian_img.spacing)
            qc_record["logjacobian_orientation"] = ants.get_orientation(logjacobian_img)
            qc_record["logjacobian_matches_template_shape"] = tuple(logjacobian_img.shape) == template_shape
            qc_record["logjacobian_matches_template_spacing"] = np.allclose(
                logjacobian_img.spacing,
                template_spacing,
                atol=1e-5,
            )
            qc_record["logjacobian_matches_template_orientation"] = (
                ants.get_orientation(logjacobian_img) == template_orientation
            )
            qc_record["logjacobian_matches_template_direction"] = np.allclose(
                np.array(logjacobian_img.direction),
                template_direction,
                atol=1e-5,
            )
            qc_record["logjacobian_finite"] = bool(np.isfinite(logjacobian_arr).all())
            qc_record["logjacobian_nonzero_variation"] = bool(np.nanstd(logjacobian_arr) > 0)
            qc_record["logjacobian_min"] = float(np.nanmin(logjacobian_arr))
            qc_record["logjacobian_max"] = float(np.nanmax(logjacobian_arr))
            qc_record["logjacobian_mean"] = float(np.nanmean(logjacobian_arr))
            qc_record["logjacobian_std"] = float(np.nanstd(logjacobian_arr))

        qc_record["qc_pass"] = all([
            qc_record["mni_exists"],
            qc_record["logjacobian_exists"],
            qc_record["mni_loadable"],
            qc_record["logjacobian_loadable"],
            qc_record["mni_matches_template_shape"],
            qc_record["logjacobian_matches_template_shape"],
            qc_record["mni_matches_template_spacing"],
            qc_record["logjacobian_matches_template_spacing"],
            qc_record["mni_matches_template_orientation"],
            qc_record["logjacobian_matches_template_orientation"],
            qc_record["mni_matches_template_direction"],
            qc_record["logjacobian_matches_template_direction"],
            qc_record["mni_finite"],
            qc_record["logjacobian_finite"],
            qc_record["mni_nonzero_variation"],
            qc_record["logjacobian_nonzero_variation"],
        ])

    except Exception as exc:
        qc_record["error"] = repr(exc)

    qc_rows.append(qc_record)

    if idx % 25 == 0 or idx == len(work_manifest):
        qc_report_partial = pd.DataFrame(qc_rows)
        qc_report_partial.to_csv(FULL_SYN_QC_REPORT, index=False)

        elapsed_minutes = (time.time() - start_time) / 60
        print(
            f"Checked {idx}/{len(work_manifest)} subjects "
            f"({elapsed_minutes:.1f} min elapsed)"
        )

full_syn_qc_report = pd.DataFrame(qc_rows)
full_syn_qc_report.to_csv(FULL_SYN_QC_REPORT, index=False)

work_manifest["mni_nii_exists"] = work_manifest["mni_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["logjacobian_nii_exists"] = work_manifest["logjacobian_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["syn_qc_pass"] = full_syn_qc_report["qc_pass"].values
work_manifest.to_csv(OUTPUT_MANIFEST, index=False)

print("\nSaved full SyN + Jacobian QC report:")
print(FULL_SYN_QC_REPORT)

print("\nOverall QC pass counts:")
display(full_syn_qc_report["qc_pass"].value_counts(dropna=False).to_frame("count"))

print("\nOutput existence counts:")
display(
    full_syn_qc_report[
        ["mni_exists", "logjacobian_exists", "mni_loadable", "logjacobian_loadable"]
    ].sum().to_frame("count")
)

print("\nShape/spacing/orientation match counts:")
display(
    full_syn_qc_report[
        [
            "mni_matches_template_shape",
            "logjacobian_matches_template_shape",
            "mni_matches_template_spacing",
            "logjacobian_matches_template_spacing",
            "mni_matches_template_orientation",
            "logjacobian_matches_template_orientation",
            "mni_matches_template_direction",
            "logjacobian_matches_template_direction",
        ]
    ].sum().to_frame("count")
)

print("\nFinite and nonzero-variation counts:")
display(
    full_syn_qc_report[
        [
            "mni_finite",
            "logjacobian_finite",
            "mni_nonzero_variation",
            "logjacobian_nonzero_variation",
        ]
    ].sum().to_frame("count")
)

print("\nQC pass by final group:")
display(
    full_syn_qc_report
    .groupby("final_group")["qc_pass"]
    .agg(["count", "sum"])
    .rename(columns={"count": "subjects", "sum": "passed"})
)

failed_syn_qc = full_syn_qc_report.loc[~full_syn_qc_report["qc_pass"]].copy()

print("\nFailed QC subjects:", len(failed_syn_qc))

if len(failed_syn_qc) > 0:
    display(
        failed_syn_qc[
            [
                "RID",
                "image_id",
                "final_group",
                "mni_exists",
                "logjacobian_exists",
                "mni_loadable",
                "logjacobian_loadable",
                "mni_matches_template_shape",
                "logjacobian_matches_template_shape",
                "mni_finite",
                "logjacobian_finite",
                "error",
            ]
        ].head(50)
    )

## 1.24. Inspect the one failed SyN QC subject

inspect the single subject that failed the full SyN QC. Since the file exists, loads, and matches the MNI template geometry, check whether the registered T1 image is blank or constant and visually compare it with the template and its log-Jacobian map.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

assert "full_syn_qc_report" in globals(), "Run the full SyN QC cell first."
assert "template_img" in globals(), "template_img is not loaded."
assert "ants" in globals(), "ANTsPy is not imported."

failed_syn_qc = full_syn_qc_report.loc[~full_syn_qc_report["qc_pass"]].copy()

print("Number of failed QC subjects:", len(failed_syn_qc))
display(failed_syn_qc)

assert len(failed_syn_qc) > 0, "There are no failed QC subjects to inspect."

failed_row = failed_syn_qc.iloc[0]

mni_path = Path(failed_row["mni_nii_path"])
logjacobian_path = Path(failed_row["logjacobian_nii_path"])

print("Inspecting failed subject:")
print("RID:", failed_row["RID"])
print("Image ID:", failed_row["image_id"])
print("Group:", failed_row["final_group"])
print("MNI T1 path:", mni_path)
print("Log-Jacobian path:", logjacobian_path)

mni_img = ants.image_read(str(mni_path))
logjacobian_img = ants.image_read(str(logjacobian_path))

mni_arr = mni_img.numpy()
logjacobian_arr = logjacobian_img.numpy()
template_arr = template_img.numpy()

summary = pd.DataFrame([{
    "RID": failed_row["RID"],
    "image_id": failed_row["image_id"],
    "final_group": failed_row["final_group"],
    "mni_shape": tuple(mni_img.shape),
    "logjacobian_shape": tuple(logjacobian_img.shape),
    "mni_spacing": tuple(mni_img.spacing),
    "logjacobian_spacing": tuple(logjacobian_img.spacing),
    "mni_orientation": ants.get_orientation(mni_img),
    "logjacobian_orientation": ants.get_orientation(logjacobian_img),
    "mni_min": float(np.nanmin(mni_arr)),
    "mni_max": float(np.nanmax(mni_arr)),
    "mni_mean": float(np.nanmean(mni_arr)),
    "mni_std": float(np.nanstd(mni_arr)),
    "mni_unique_values_first_20": str(np.unique(mni_arr)[:20]),
    "logjacobian_min": float(np.nanmin(logjacobian_arr)),
    "logjacobian_max": float(np.nanmax(logjacobian_arr)),
    "logjacobian_mean": float(np.nanmean(logjacobian_arr)),
    "logjacobian_std": float(np.nanstd(logjacobian_arr)),
}])

display(summary)

# Visual inspection using central slices.
x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

logjac_low, logjac_high = np.nanpercentile(logjacobian_arr, [1, 99])

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

images = [
    [template_arr[x_mid, :, :], mni_arr[x_mid, :, :], logjacobian_arr[x_mid, :, :]],
    [template_arr[:, y_mid, :], mni_arr[:, y_mid, :], logjacobian_arr[:, y_mid, :]],
    [template_arr[:, :, z_mid], mni_arr[:, :, z_mid], logjacobian_arr[:, :, z_mid]],
]

titles = [
    ["Template sagittal", "Failed subject MNI T1 sagittal", "Failed subject log-Jacobian sagittal"],
    ["Template coronal", "Failed subject MNI T1 coronal", "Failed subject log-Jacobian coronal"],
    ["Template axial", "Failed subject MNI T1 axial", "Failed subject log-Jacobian axial"],
]

for row_idx in range(3):
    for col_idx in range(3):
        ax = axes[row_idx, col_idx]

        if col_idx == 2:
            ax.imshow(
                np.rot90(images[row_idx][col_idx]),
                vmin=logjac_low,
                vmax=logjac_high,
            )
        else:
            ax.imshow(np.rot90(images[row_idx][col_idx]), cmap="gray")

        ax.set_title(titles[row_idx][col_idx])
        ax.axis("off")

plt.tight_layout()

FAILED_SUBJECT_QC_FIGURE = OUTPUT_REPORT.parent / "failed_syn_qc_subject_visual_inspection.png"
plt.savefig(FAILED_SUBJECT_QC_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved failed-subject visual inspection figure:")
print(FAILED_SUBJECT_QC_FIGURE)

## 1.25. Rerun SyN registration for the one failed subject

rerun nonlinear MNI registration and log-Jacobian generation only for the failed subject. overwrite the corrupted registered T1 output and regenerate the matching log-Jacobian map from the same new registration.

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd

assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "work_manifest" in globals(), "work_manifest is not available."
assert "full_syn_qc_report" in globals(), "full_syn_qc_report is not available."

failed_syn_qc = full_syn_qc_report.loc[~full_syn_qc_report["qc_pass"]].copy()

assert len(failed_syn_qc) == 1, (
    f"Expected exactly one failed subject, but found {len(failed_syn_qc)}."
)

failed_row = failed_syn_qc.iloc[0]

target_rid = failed_row["RID"]
target_image_id = failed_row["image_id"]

target_manifest_row = work_manifest.loc[
    (work_manifest[RID_COL].astype(str) == str(target_rid))
    & (work_manifest[IMAGE_ID_COL].astype(str) == str(target_image_id))
]

assert len(target_manifest_row) == 1, (
    "Could not uniquely match the failed QC subject back to work_manifest."
)

row = target_manifest_row.iloc[0]

n4_path = Path(row["n4_nii_path"])
mni_path = Path(row["mni_nii_path"])
logjacobian_path = Path(row["logjacobian_nii_path"])

tmp_mni_path = mni_path.with_name(mni_path.name.replace(".nii.gz", "_rerun_tmp.nii.gz"))
tmp_logjacobian_path = logjacobian_path.with_name(
    logjacobian_path.name.replace(".nii.gz", "_rerun_tmp.nii.gz")
)

print("Rerunning failed SyN subject:")
print("RID:", row[RID_COL])
print("Image ID:", row[IMAGE_ID_COL])
print("Group:", row[LABEL_COL])
print("Input N4 image:", n4_path)
print("MNI T1 output to replace:", mni_path)
print("Log-Jacobian output to replace:", logjacobian_path)

assert n4_path.exists(), f"N4 input does not exist: {n4_path}"

start_time = time.time()

for tmp_path in [tmp_mni_path, tmp_logjacobian_path]:
    if tmp_path.exists():
        tmp_path.unlink()

try:
    moving_img = ants.image_read(str(n4_path)).clone("float")

    registration = ants.registration(
        fixed=template_img,
        moving=moving_img,
        type_of_transform="SyN",
        verbose=False,
    )

    mni_img = registration["warpedmovout"]
    ants.image_write(mni_img, str(tmp_mni_path))

    forward_warp = None

    for transform_path in registration["fwdtransforms"]:
        transform_name = Path(transform_path).name

        if "Warp" in transform_name and "Inverse" not in transform_name:
            forward_warp = transform_path
            break

    if forward_warp is None:
        raise RuntimeError(f"No forward warp found: {registration['fwdtransforms']}")

    logjacobian_img = ants.create_jacobian_determinant_image(
        domain_image=template_img,
        tx=forward_warp,
        do_log=True,
        geom=False,
    )

    ants.image_write(logjacobian_img, str(tmp_logjacobian_path))

    # Replace the corrupted outputs only after both new files have been written.
    tmp_mni_path.replace(mni_path)
    tmp_logjacobian_path.replace(logjacobian_path)

    rerun_status = "processed"
    rerun_error = ""

except Exception as exc:
    rerun_status = "failed"
    rerun_error = repr(exc)

    for tmp_path in [tmp_mni_path, tmp_logjacobian_path]:
        if tmp_path.exists():
            tmp_path.unlink()

elapsed_minutes = (time.time() - start_time) / 60

rerun_report = pd.DataFrame([{
    "RID": row[RID_COL],
    "image_id": row[IMAGE_ID_COL],
    "final_group": row[LABEL_COL],
    "n4_nii_path": str(n4_path),
    "mni_nii_path": str(mni_path),
    "logjacobian_nii_path": str(logjacobian_path),
    "status": rerun_status,
    "elapsed_minutes": round(elapsed_minutes, 2),
    "error": rerun_error,
}])

FAILED_RERUN_REPORT = OUTPUT_REPORT.parent / "rerun_failed_syn_subject_report.csv"
rerun_report.to_csv(FAILED_RERUN_REPORT, index=False)

display(rerun_report)

print("Saved failed-subject rerun report:")
print(FAILED_RERUN_REPORT)

# Immediate technical check of the rerun output.
if rerun_status == "processed":
    rerun_mni_img = ants.image_read(str(mni_path))
    rerun_logjacobian_img = ants.image_read(str(logjacobian_path))

    rerun_mni_arr = rerun_mni_img.numpy()
    rerun_logjacobian_arr = rerun_logjacobian_img.numpy()

    rerun_qc = pd.DataFrame([{
        "RID": row[RID_COL],
        "image_id": row[IMAGE_ID_COL],
        "final_group": row[LABEL_COL],
        "mni_shape": tuple(rerun_mni_img.shape),
        "logjacobian_shape": tuple(rerun_logjacobian_img.shape),
        "mni_spacing": tuple(rerun_mni_img.spacing),
        "logjacobian_spacing": tuple(rerun_logjacobian_img.spacing),
        "mni_orientation": ants.get_orientation(rerun_mni_img),
        "logjacobian_orientation": ants.get_orientation(rerun_logjacobian_img),
        "mni_matches_template_shape": tuple(rerun_mni_img.shape) == tuple(template_img.shape),
        "logjacobian_matches_template_shape": tuple(rerun_logjacobian_img.shape) == tuple(template_img.shape),
        "mni_matches_template_spacing": np.allclose(rerun_mni_img.spacing, template_img.spacing, atol=1e-5),
        "logjacobian_matches_template_spacing": np.allclose(
            rerun_logjacobian_img.spacing,
            template_img.spacing,
            atol=1e-5,
        ),
        "mni_finite": bool(np.isfinite(rerun_mni_arr).all()),
        "logjacobian_finite": bool(np.isfinite(rerun_logjacobian_arr).all()),
        "mni_nonzero_variation": bool(np.nanstd(rerun_mni_arr) > 0),
        "logjacobian_nonzero_variation": bool(np.nanstd(rerun_logjacobian_arr) > 0),
        "mni_min": float(np.nanmin(rerun_mni_arr)),
        "mni_max": float(np.nanmax(rerun_mni_arr)),
        "mni_mean": float(np.nanmean(rerun_mni_arr)),
        "mni_std": float(np.nanstd(rerun_mni_arr)),
        "logjacobian_min": float(np.nanmin(rerun_logjacobian_arr)),
        "logjacobian_max": float(np.nanmax(rerun_logjacobian_arr)),
        "logjacobian_mean": float(np.nanmean(rerun_logjacobian_arr)),
        "logjacobian_std": float(np.nanstd(rerun_logjacobian_arr)),
    }])

    RERUN_FAILED_SUBJECT_QC_REPORT = OUTPUT_REPORT.parent / "rerun_failed_syn_subject_qc_report.csv"
    rerun_qc.to_csv(RERUN_FAILED_SUBJECT_QC_REPORT, index=False)

    display(rerun_qc)

    print("Saved rerun failed-subject QC report:")
    print(RERUN_FAILED_SUBJECT_QC_REPORT)

## 1.26. Visually inspect the rerun SyN subject

visually inspect the subject that was rerun before updating the final QC report. compare the MNI template, the corrected registered T1 image, and the corrected log-Jacobian map in sagittal, coronal, and axial views.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "rerun_report" in globals(), "rerun_report is not available."

rerun_row = rerun_report.iloc[0]

mni_path = Path(rerun_row["mni_nii_path"])
logjacobian_path = Path(rerun_row["logjacobian_nii_path"])

print("Visually inspecting rerun subject:")
print("RID:", rerun_row["RID"])
print("Image ID:", rerun_row["image_id"])
print("Group:", rerun_row["final_group"])
print("MNI T1 path:", mni_path)
print("Log-Jacobian path:", logjacobian_path)

assert mni_path.exists(), f"Corrected MNI T1 output does not exist: {mni_path}"
assert logjacobian_path.exists(), f"Corrected log-Jacobian output does not exist: {logjacobian_path}"

mni_img = ants.image_read(str(mni_path))
logjacobian_img = ants.image_read(str(logjacobian_path))

template_arr = template_img.numpy()
mni_arr = mni_img.numpy()
logjacobian_arr = logjacobian_img.numpy()

summary = pd.DataFrame([{
    "RID": rerun_row["RID"],
    "image_id": rerun_row["image_id"],
    "final_group": rerun_row["final_group"],
    "mni_shape": tuple(mni_img.shape),
    "logjacobian_shape": tuple(logjacobian_img.shape),
    "mni_spacing": tuple(mni_img.spacing),
    "logjacobian_spacing": tuple(logjacobian_img.spacing),
    "mni_orientation": ants.get_orientation(mni_img),
    "logjacobian_orientation": ants.get_orientation(logjacobian_img),
    "mni_min": float(np.nanmin(mni_arr)),
    "mni_max": float(np.nanmax(mni_arr)),
    "mni_mean": float(np.nanmean(mni_arr)),
    "mni_std": float(np.nanstd(mni_arr)),
    "logjacobian_min": float(np.nanmin(logjacobian_arr)),
    "logjacobian_max": float(np.nanmax(logjacobian_arr)),
    "logjacobian_mean": float(np.nanmean(logjacobian_arr)),
    "logjacobian_std": float(np.nanstd(logjacobian_arr)),
}])

display(summary)

# Use central slices in template space.
x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

# Robust display limits make the corrected T1 easier to inspect visually.
mni_low, mni_high = np.nanpercentile(mni_arr, [1, 99])
logjac_low, logjac_high = np.nanpercentile(logjacobian_arr, [1, 99])

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

images = [
    [template_arr[x_mid, :, :], mni_arr[x_mid, :, :], logjacobian_arr[x_mid, :, :]],
    [template_arr[:, y_mid, :], mni_arr[:, y_mid, :], logjacobian_arr[:, y_mid, :]],
    [template_arr[:, :, z_mid], mni_arr[:, :, z_mid], logjacobian_arr[:, :, z_mid]],
]

titles = [
    ["Template sagittal", "Corrected rerun T1 sagittal", "Corrected rerun log-Jacobian sagittal"],
    ["Template coronal", "Corrected rerun T1 coronal", "Corrected rerun log-Jacobian coronal"],
    ["Template axial", "Corrected rerun T1 axial", "Corrected rerun log-Jacobian axial"],
]

for row_idx in range(3):
    for col_idx in range(3):
        ax = axes[row_idx, col_idx]

        if col_idx == 0:
            ax.imshow(np.rot90(images[row_idx][col_idx]), cmap="gray")

        elif col_idx == 1:
            ax.imshow(
                np.rot90(images[row_idx][col_idx]),
                cmap="gray",
                vmin=mni_low,
                vmax=mni_high,
            )

        else:
            ax.imshow(
                np.rot90(images[row_idx][col_idx]),
                vmin=logjac_low,
                vmax=logjac_high,
            )

        ax.set_title(titles[row_idx][col_idx])
        ax.axis("off")

plt.tight_layout()

RERUN_VISUAL_QC_FIGURE = OUTPUT_REPORT.parent / "rerun_failed_subject_visual_qc_after_fix.png"
plt.savefig(RERUN_VISUAL_QC_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved rerun visual QC figure:")
print(RERUN_VISUAL_QC_FIGURE)

## 1.27. Update final SyN QC after rerunning the failed subject

update the full SyN QC report after rerunning the one failed subject. replace the old failed QC row with the new corrected measurements, add a stricter sanity check for impossible intensity values, and save a final QC report and updated manifest.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "work_manifest" in globals(), "work_manifest is not available."
assert "rerun_report" in globals(), "rerun_report is not available."

# Reload the last saved full QC report to avoid any partially modified in-memory state.
ORIGINAL_FULL_SYN_QC_REPORT = OUTPUT_REPORT.parent / "full_syn_registration_jacobian_qc_report.csv"

assert ORIGINAL_FULL_SYN_QC_REPORT.exists(), (
    f"Original full SyN QC report not found: {ORIGINAL_FULL_SYN_QC_REPORT}"
)

full_syn_qc_report = pd.read_csv(ORIGINAL_FULL_SYN_QC_REPORT)

target_rid = str(rerun_report.iloc[0]["RID"])
target_image_id = str(rerun_report.iloc[0]["image_id"])

target_manifest_row = work_manifest.loc[
    (work_manifest[RID_COL].astype(str) == target_rid)
    & (work_manifest[IMAGE_ID_COL].astype(str) == target_image_id)
]

assert len(target_manifest_row) == 1, "Could not uniquely find the rerun subject in work_manifest."

row = target_manifest_row.iloc[0]

mni_path = Path(row["mni_nii_path"])
logjacobian_path = Path(row["logjacobian_nii_path"])

template_shape = tuple(template_img.shape)
template_spacing = tuple(template_img.spacing)
template_orientation = ants.get_orientation(template_img)
template_direction = np.array(template_img.direction)

mni_img = ants.image_read(str(mni_path))
logjacobian_img = ants.image_read(str(logjacobian_path))

mni_arr = mni_img.numpy().astype(np.float64)
logjacobian_arr = logjacobian_img.numpy().astype(np.float64)

updated_record = {
    "RID": row[RID_COL],
    "image_id": row[IMAGE_ID_COL],
    "final_group": row[LABEL_COL],
    "mni_nii_path": str(mni_path),
    "logjacobian_nii_path": str(logjacobian_path),
    "mni_exists": mni_path.exists(),
    "logjacobian_exists": logjacobian_path.exists(),
    "mni_loadable": True,
    "logjacobian_loadable": True,
    "mni_shape": str(tuple(mni_img.shape)),
    "logjacobian_shape": str(tuple(logjacobian_img.shape)),
    "mni_spacing": str(tuple(mni_img.spacing)),
    "logjacobian_spacing": str(tuple(logjacobian_img.spacing)),
    "mni_orientation": ants.get_orientation(mni_img),
    "logjacobian_orientation": ants.get_orientation(logjacobian_img),
    "mni_matches_template_shape": tuple(mni_img.shape) == template_shape,
    "logjacobian_matches_template_shape": tuple(logjacobian_img.shape) == template_shape,
    "mni_matches_template_spacing": np.allclose(mni_img.spacing, template_spacing, atol=1e-5),
    "logjacobian_matches_template_spacing": np.allclose(logjacobian_img.spacing, template_spacing, atol=1e-5),
    "mni_matches_template_orientation": ants.get_orientation(mni_img) == template_orientation,
    "logjacobian_matches_template_orientation": ants.get_orientation(logjacobian_img) == template_orientation,
    "mni_matches_template_direction": np.allclose(np.array(mni_img.direction), template_direction, atol=1e-5),
    "logjacobian_matches_template_direction": np.allclose(
        np.array(logjacobian_img.direction),
        template_direction,
        atol=1e-5,
    ),
    "mni_finite": bool(np.isfinite(mni_arr).all()),
    "logjacobian_finite": bool(np.isfinite(logjacobian_arr).all()),
    "mni_nonzero_variation": bool(np.nanstd(mni_arr) > 0),
    "logjacobian_nonzero_variation": bool(np.nanstd(logjacobian_arr) > 0),
    "mni_min": float(np.nanmin(mni_arr)),
    "mni_max": float(np.nanmax(mni_arr)),
    "mni_mean": float(np.nanmean(mni_arr)),
    "mni_std": float(np.nanstd(mni_arr)),
    "logjacobian_min": float(np.nanmin(logjacobian_arr)),
    "logjacobian_max": float(np.nanmax(logjacobian_arr)),
    "logjacobian_mean": float(np.nanmean(logjacobian_arr)),
    "logjacobian_std": float(np.nanstd(logjacobian_arr)),
    "error": "",
}

updated_record["mni_summary_stats_finite"] = bool(
    np.isfinite(updated_record["mni_min"])
    and np.isfinite(updated_record["mni_max"])
    and np.isfinite(updated_record["mni_mean"])
    and np.isfinite(updated_record["mni_std"])
)

updated_record["logjacobian_summary_stats_finite"] = bool(
    np.isfinite(updated_record["logjacobian_min"])
    and np.isfinite(updated_record["logjacobian_max"])
    and np.isfinite(updated_record["logjacobian_mean"])
    and np.isfinite(updated_record["logjacobian_std"])
)

updated_record["mni_abs_max_reasonable"] = bool(
    max(abs(updated_record["mni_min"]), abs(updated_record["mni_max"])) < 1_000_000
)

updated_record["qc_pass"] = all([
    updated_record["mni_exists"],
    updated_record["logjacobian_exists"],
    updated_record["mni_loadable"],
    updated_record["logjacobian_loadable"],
    updated_record["mni_matches_template_shape"],
    updated_record["logjacobian_matches_template_shape"],
    updated_record["mni_matches_template_spacing"],
    updated_record["logjacobian_matches_template_spacing"],
    updated_record["mni_matches_template_orientation"],
    updated_record["logjacobian_matches_template_orientation"],
    updated_record["mni_matches_template_direction"],
    updated_record["logjacobian_matches_template_direction"],
    updated_record["mni_finite"],
    updated_record["logjacobian_finite"],
    updated_record["mni_nonzero_variation"],
    updated_record["logjacobian_nonzero_variation"],
    updated_record["mni_summary_stats_finite"],
    updated_record["logjacobian_summary_stats_finite"],
    updated_record["mni_abs_max_reasonable"],
])

target_mask = (
    (full_syn_qc_report["RID"].astype(str) == target_rid)
    & (full_syn_qc_report["image_id"].astype(str) == target_image_id)
)

assert target_mask.sum() == 1, "Could not uniquely find the rerun subject in full_syn_qc_report."

target_index = full_syn_qc_report.index[target_mask][0]

# Use .at with the exact row index to avoid pandas treating tuples/lists as row-length arrays.
for key, value in updated_record.items():
    if key not in full_syn_qc_report.columns:
        full_syn_qc_report[key] = np.nan

    full_syn_qc_report.at[target_index, key] = value

# Convert numeric summary columns safely.
numeric_cols = [
    "mni_min",
    "mni_max",
    "mni_mean",
    "mni_std",
    "logjacobian_min",
    "logjacobian_max",
    "logjacobian_mean",
    "logjacobian_std",
]

for col in numeric_cols:
    full_syn_qc_report[col] = pd.to_numeric(full_syn_qc_report[col], errors="coerce")

# Convert QC columns safely to boolean.
bool_cols = [
    "qc_pass",
    "mni_summary_stats_finite",
    "logjacobian_summary_stats_finite",
    "mni_abs_max_reasonable",
]

for col in bool_cols:
    if col not in full_syn_qc_report.columns:
        full_syn_qc_report[col] = False

    full_syn_qc_report[col] = full_syn_qc_report[col].astype(str).str.lower().map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }).fillna(False)

# Add stricter sanity columns to all rows using already-saved summary statistics.
full_syn_qc_report["mni_summary_stats_finite"] = (
    np.isfinite(full_syn_qc_report["mni_min"])
    & np.isfinite(full_syn_qc_report["mni_max"])
    & np.isfinite(full_syn_qc_report["mni_mean"])
    & np.isfinite(full_syn_qc_report["mni_std"])
)

full_syn_qc_report["logjacobian_summary_stats_finite"] = (
    np.isfinite(full_syn_qc_report["logjacobian_min"])
    & np.isfinite(full_syn_qc_report["logjacobian_max"])
    & np.isfinite(full_syn_qc_report["logjacobian_mean"])
    & np.isfinite(full_syn_qc_report["logjacobian_std"])
)

full_syn_qc_report["mni_abs_max_reasonable"] = (
    full_syn_qc_report[["mni_min", "mni_max"]]
    .abs()
    .max(axis=1)
    .lt(1_000_000)
)

full_syn_qc_report["final_syn_qc_pass"] = (
    full_syn_qc_report["qc_pass"]
    & full_syn_qc_report["mni_summary_stats_finite"]
    & full_syn_qc_report["logjacobian_summary_stats_finite"]
    & full_syn_qc_report["mni_abs_max_reasonable"]
)

FINAL_SYN_QC_REPORT = OUTPUT_REPORT.parent / "full_syn_registration_jacobian_qc_report_final_after_rerun.csv"
full_syn_qc_report.to_csv(FINAL_SYN_QC_REPORT, index=False)

# Update work_manifest with the final QC status.
qc_lookup = (
    full_syn_qc_report
    .assign(
        RID_key=full_syn_qc_report["RID"].astype(str),
        image_id_key=full_syn_qc_report["image_id"].astype(str),
    )
    .set_index(["RID_key", "image_id_key"])["final_syn_qc_pass"]
    .to_dict()
)

work_manifest["syn_qc_pass"] = work_manifest.apply(
    lambda r: bool(qc_lookup.get((str(r[RID_COL]), str(r[IMAGE_ID_COL])), False)),
    axis=1,
)

work_manifest.to_csv(OUTPUT_MANIFEST, index=False)

print("Saved final SyN QC report:")
print(FINAL_SYN_QC_REPORT)

print("\nFinal SyN QC pass counts:")
display(full_syn_qc_report["final_syn_qc_pass"].value_counts(dropna=False).to_frame("count"))

print("\nFinal SyN QC pass by group:")
display(
    full_syn_qc_report
    .groupby("final_group")["final_syn_qc_pass"]
    .agg(["count", "sum"])
    .rename(columns={"count": "subjects", "sum": "passed"})
)

print("\nManifest syn_qc_pass counts:")
display(work_manifest["syn_qc_pass"].value_counts(dropna=False).to_frame("count"))

failed_final_syn_qc = full_syn_qc_report.loc[~full_syn_qc_report["final_syn_qc_pass"]].copy()

print("\nRemaining failed final SyN QC subjects:", len(failed_final_syn_qc))

if len(failed_final_syn_qc) > 0:
    display(
        failed_final_syn_qc[
            [
                "RID",
                "image_id",
                "final_group",
                "qc_pass",
                "mni_summary_stats_finite",
                "logjacobian_summary_stats_finite",
                "mni_abs_max_reasonable",
                "mni_min",
                "mni_max",
                "mni_mean",
                "mni_std",
                "error",
            ]
        ]
    )

## 1.28. Visually sample registered T1 and Jacobian outputs across groups

visually inspect a small balanced sample from AD, CN, pMCI, and sMCI. This helps me check that the MNI-registered T1 images look anatomically aligned and that the log-Jacobian maps look structured rather than corrupted. save both the sampled-subject table and the visual QC figure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "work_manifest" in globals(), "work_manifest is not available."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "mni_nii_path",
    "logjacobian_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

VISUAL_QC_DIR = OUTPUT_REPORT.parent / "visual_qc"
VISUAL_QC_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES_PER_GROUP = 2
RANDOM_SEED = 42

# Use only subjects that passed SyN QC if that column is available.
if "syn_qc_pass" in work_manifest.columns:
    visual_pool = work_manifest.loc[work_manifest["syn_qc_pass"].astype(bool)].copy()
else:
    visual_pool = work_manifest.copy()

visual_pool["mni_nii_exists"] = visual_pool["mni_nii_path"].apply(lambda p: Path(p).exists())
visual_pool["logjacobian_nii_exists"] = visual_pool["logjacobian_nii_path"].apply(lambda p: Path(p).exists())

visual_pool = visual_pool.loc[
    visual_pool["mni_nii_exists"]
    & visual_pool["logjacobian_nii_exists"]
].copy()

print("Subjects available for visual QC:", len(visual_pool))
print("Available subjects by group:")
display(visual_pool[LABEL_COL].value_counts().to_frame("count"))

sampled_rows = []

for group_name, group_df in visual_pool.groupby(LABEL_COL):
    n_to_sample = min(SAMPLES_PER_GROUP, len(group_df))

    sampled_group = group_df.sample(
        n=n_to_sample,
        random_state=RANDOM_SEED,
    )

    sampled_rows.append(sampled_group)

visual_qc_sample = pd.concat(sampled_rows, axis=0).reset_index(drop=True)

# Sort into a stable group order if all groups are present.
preferred_group_order = ["CN", "AD", "pMCI", "sMCI"]
visual_qc_sample[LABEL_COL] = pd.Categorical(
    visual_qc_sample[LABEL_COL],
    categories=preferred_group_order,
    ordered=True,
)

visual_qc_sample = (
    visual_qc_sample
    .sort_values([LABEL_COL, RID_COL])
    .reset_index(drop=True)
)

VISUAL_QC_SAMPLE_MANIFEST = VISUAL_QC_DIR / "visual_qc_sample_subjects.csv"
visual_qc_sample.to_csv(VISUAL_QC_SAMPLE_MANIFEST, index=False)

print("Visual QC sample:")
display(
    visual_qc_sample[
        [
            RID_COL,
            IMAGE_ID_COL,
            LABEL_COL,
            "mni_nii_path",
            "logjacobian_nii_path",
        ]
    ]
)

print("Saved visual QC sample manifest:")
print(VISUAL_QC_SAMPLE_MANIFEST)

template_arr = template_img.numpy()

x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

# First show the template slices so I can compare the sampled subjects mentally.
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

template_slices = [
    template_arr[x_mid, :, :],
    template_arr[:, y_mid, :],
    template_arr[:, :, z_mid],
]

template_titles = [
    "Template sagittal",
    "Template coronal",
    "Template axial",
]

for ax, image_slice, title in zip(axes, template_slices, template_titles):
    ax.imshow(np.rot90(image_slice), cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()

TEMPLATE_VISUAL_QC_FIGURE = VISUAL_QC_DIR / "visual_qc_template_reference.png"
plt.savefig(TEMPLATE_VISUAL_QC_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved template visual QC figure:")
print(TEMPLATE_VISUAL_QC_FIGURE)

# Then show sampled subjects. Each row is one subject.
fig, axes = plt.subplots(
    len(visual_qc_sample),
    4,
    figsize=(16, 3.2 * len(visual_qc_sample)),
)

if len(visual_qc_sample) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, (_, row) in enumerate(visual_qc_sample.iterrows()):
    mni_path = Path(row["mni_nii_path"])
    logjacobian_path = Path(row["logjacobian_nii_path"])

    mni_img = ants.image_read(str(mni_path))
    logjacobian_img = ants.image_read(str(logjacobian_path))

    mni_arr = mni_img.numpy()
    logjacobian_arr = logjacobian_img.numpy()

    mni_low, mni_high = np.nanpercentile(mni_arr, [1, 99])
    logjac_low, logjac_high = np.nanpercentile(logjacobian_arr, [1, 99])

    subject_label = (
        f"{row[LABEL_COL]} | RID {row[RID_COL]} | I{row[IMAGE_ID_COL]}"
    )

    images = [
        mni_arr[x_mid, :, :],
        mni_arr[:, y_mid, :],
        mni_arr[:, :, z_mid],
        logjacobian_arr[:, :, z_mid],
    ]

    titles = [
        "T1 sagittal",
        "T1 coronal",
        "T1 axial",
        "Log-Jacobian axial",
    ]

    for col_idx in range(4):
        ax = axes[row_idx, col_idx]

        if col_idx < 3:
            ax.imshow(
                np.rot90(images[col_idx]),
                cmap="gray",
                vmin=mni_low,
                vmax=mni_high,
            )
        else:
            ax.imshow(
                np.rot90(images[col_idx]),
                vmin=logjac_low,
                vmax=logjac_high,
            )

        if col_idx == 0:
            ax.set_ylabel(subject_label, fontsize=10)

        ax.set_title(titles[col_idx])
        ax.axis("off")

plt.tight_layout()

GROUP_VISUAL_QC_FIGURE = VISUAL_QC_DIR / "visual_qc_sample_registered_t1_and_logjacobian.png"
plt.savefig(GROUP_VISUAL_QC_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved group visual QC figure:")
print(GROUP_VISUAL_QC_FIGURE)

## 1.29. Create and inspect a shared MNI-space analysis mask

create one shared mask in MNI template space. This is not subject-specific skull stripping. The same mask will be used for every subject to define the common crop region and later compute normalization statistics consistently. only create and inspect the mask in this cell; not apply it to all MRI files yet.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."

MASK_DIR = PROJECT_DIR / "templates" / "masks"
MASK_DIR.mkdir(parents=True, exist_ok=True)

COMMON_MASK_PATH = MASK_DIR / "tpl-MNI152NLin2009cSym_res-1_desc-sharedAnalysis_mask.nii.gz"
COMMON_MASK_QC_REPORT = OUTPUT_REPORT.parent / "shared_mni_analysis_mask_qc_report.csv"
COMMON_MASK_QC_FIGURE = OUTPUT_REPORT.parent / "shared_mni_analysis_mask_visual_qc.png"

template_arr = template_img.numpy().astype(np.float32)

# This creates one shared foreground mask from the MNI template.
# It removes only the zero-valued empty background of the template.
# It does not create subject-specific brain outlines.
mask_arr = template_arr > 0

common_mask_img = ants.from_numpy(
    mask_arr.astype(np.uint8),
    origin=template_img.origin,
    spacing=template_img.spacing,
    direction=template_img.direction,
)

ants.image_write(common_mask_img, str(COMMON_MASK_PATH))

mask_qc = pd.DataFrame([{
    "mask_path": str(COMMON_MASK_PATH),
    "template_shape": tuple(template_img.shape),
    "mask_shape": tuple(common_mask_img.shape),
    "template_spacing": tuple(template_img.spacing),
    "mask_spacing": tuple(common_mask_img.spacing),
    "template_orientation": ants.get_orientation(template_img),
    "mask_orientation": ants.get_orientation(common_mask_img),
    "mask_voxels": int(mask_arr.sum()),
    "total_voxels": int(mask_arr.size),
    "mask_fraction": float(mask_arr.sum() / mask_arr.size),
    "mask_unique_values": str(np.unique(mask_arr.astype(np.uint8))),
}])

mask_qc.to_csv(COMMON_MASK_QC_REPORT, index=False)

display(mask_qc)

print("Saved shared MNI-space analysis mask:")
print(COMMON_MASK_PATH)

print("Saved mask QC report:")
print(COMMON_MASK_QC_REPORT)

# Visual inspection: template, mask, and overlay.
x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

slices = [
    ("Sagittal", template_arr[x_mid, :, :], mask_arr[x_mid, :, :]),
    ("Coronal", template_arr[:, y_mid, :], mask_arr[:, y_mid, :]),
    ("Axial", template_arr[:, :, z_mid], mask_arr[:, :, z_mid]),
]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for row_idx, (view_name, template_slice, mask_slice) in enumerate(slices):
    axes[row_idx, 0].imshow(np.rot90(template_slice), cmap="gray")
    axes[row_idx, 0].set_title(f"Template {view_name.lower()}")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(np.rot90(mask_slice), cmap="gray")
    axes[row_idx, 1].set_title(f"Shared mask {view_name.lower()}")
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(np.rot90(template_slice), cmap="gray")
    axes[row_idx, 2].imshow(np.rot90(mask_slice), alpha=0.35)
    axes[row_idx, 2].set_title(f"Overlay {view_name.lower()}")
    axes[row_idx, 2].axis("off")

plt.tight_layout()
plt.savefig(COMMON_MASK_QC_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved shared mask visual QC figure:")
print(COMMON_MASK_QC_FIGURE)

## 1.30. Compare candidate shared MNI masks

The first shared mask was too broad because it used every nonzero template voxel. compare several stronger template-intensity thresholds. only inspect candidate masks in this cell, not apply any mask to the subject images yet.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

assert "template_img" in globals(), "template_img is not loaded."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."

template_arr = template_img.numpy().astype(np.float32)

MASK_CANDIDATE_DIR = OUTPUT_REPORT.parent / "mask_candidates"
MASK_CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

# Use fractions of the template maximum intensity.
# These are candidate shared masks only.
template_max = float(np.nanmax(template_arr))

threshold_fractions = [0.01, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]
threshold_values = [template_max * frac for frac in threshold_fractions]

x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

candidate_rows = []

fig, axes = plt.subplots(
    len(threshold_values),
    3,
    figsize=(12, 3.2 * len(threshold_values)),
)

for row_idx, (frac, threshold_value) in enumerate(zip(threshold_fractions, threshold_values)):
    candidate_mask = template_arr > threshold_value

    mask_voxels = int(candidate_mask.sum())
    total_voxels = int(candidate_mask.size)
    mask_fraction = mask_voxels / total_voxels

    if mask_voxels > 0:
        coords = np.argwhere(candidate_mask)
        crop_start = coords.min(axis=0)
        crop_end = coords.max(axis=0) + 1
        crop_shape = crop_end - crop_start
    else:
        crop_start = np.array([np.nan, np.nan, np.nan])
        crop_end = np.array([np.nan, np.nan, np.nan])
        crop_shape = np.array([np.nan, np.nan, np.nan])

    candidate_rows.append({
        "threshold_fraction_of_template_max": frac,
        "threshold_value": threshold_value,
        "mask_voxels": mask_voxels,
        "total_voxels": total_voxels,
        "mask_fraction": mask_fraction,
        "x_start": crop_start[0],
        "x_end": crop_end[0],
        "y_start": crop_start[1],
        "y_end": crop_end[1],
        "z_start": crop_start[2],
        "z_end": crop_end[2],
        "crop_shape_without_margin": tuple(crop_shape),
    })

    sagittal_template = template_arr[x_mid, :, :]
    coronal_template = template_arr[:, y_mid, :]
    axial_template = template_arr[:, :, z_mid]

    sagittal_mask = candidate_mask[x_mid, :, :]
    coronal_mask = candidate_mask[:, y_mid, :]
    axial_mask = candidate_mask[:, :, z_mid]

    views = [
        ("Sagittal", sagittal_template, sagittal_mask),
        ("Coronal", coronal_template, coronal_mask),
        ("Axial", axial_template, axial_mask),
    ]

    for col_idx, (view_name, template_slice, mask_slice) in enumerate(views):
        ax = axes[row_idx, col_idx]

        ax.imshow(np.rot90(template_slice), cmap="gray")
        ax.imshow(np.rot90(mask_slice), alpha=0.35)

        ax.set_title(
            f"{view_name}: threshold {frac:.0%} of max\n"
            f"mask fraction {mask_fraction:.2f}"
        )
        ax.axis("off")

plt.tight_layout()

MASK_CANDIDATE_FIGURE = MASK_CANDIDATE_DIR / "shared_mni_mask_threshold_candidates.png"
plt.savefig(MASK_CANDIDATE_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

mask_candidate_report = pd.DataFrame(candidate_rows)

MASK_CANDIDATE_REPORT = MASK_CANDIDATE_DIR / "shared_mni_mask_threshold_candidates_report.csv"
mask_candidate_report.to_csv(MASK_CANDIDATE_REPORT, index=False)

print("Template maximum intensity:", template_max)

print("\nSaved mask candidate report:")
print(MASK_CANDIDATE_REPORT)

print("\nSaved mask candidate figure:")
print(MASK_CANDIDATE_FIGURE)

display(mask_candidate_report)

## 1.31. Define a broad shared MNI crop box without applying skull stripping

define one shared rectangular crop box in MNI template space. I am deliberately using a broad 5% template-intensity threshold only as a guide to find the outer useful image region, not as a hard mask to apply to the MRI scans.

This distinction matters because the threshold mask can contain holes inside the brain, especially in dark T1 regions such as ventricles, sulci, and CSF spaces. If I applied that mask directly, it could remove real anatomy. I do not want that. Instead, only use the mask to find the outer bounding box, then add a safety margin and crop every subject with the exact same rectangular crop.

This is also different from subject-specific skull stripping. I am not creating a different brain outline for each subject. The crop box is fixed for the whole cohort, so it cannot encode individual skull-stripping contours or subject-specific brain-mask volume. The goal here is only to remove unnecessary empty background and reduce the final model input size before normalization and array saving.

In this cell, only define and visually inspect the shared crop box on the MNI template. not crop all subject images yet.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

assert "template_arr" in globals(), "template_arr is missing. Rerun the crop-planning cell up to the failed point."
assert "crop_plan" in globals(), "crop_plan is missing. Rerun the crop-planning cell up to the failed point."
assert "CROP_PLAN_JSON" in globals(), "CROP_PLAN_JSON is missing."
assert "CROP_PLAN_REPORT" in globals(), "CROP_PLAN_REPORT is missing."
assert "CROP_DIR" in globals(), "CROP_DIR is missing."
assert "crop_reference_mask" in globals(), "crop_reference_mask is missing."
assert "crop_box_mask" in globals(), "crop_box_mask is missing."
assert "crop_shape" in globals(), "crop_shape is missing."

# Convert all NumPy scalar values into normal Python values before JSON saving.
crop_params = {
    "crop_threshold_fraction_of_template_max": float(CROP_THRESHOLD_FRACTION),
    "crop_threshold_value": float(crop_threshold_value),
    "crop_margin_voxels": int(CROP_MARGIN_VOXELS),
    "x_start": int(x_start),
    "x_end": int(x_end),
    "y_start": int(y_start),
    "y_end": int(y_end),
    "z_start": int(z_start),
    "z_end": int(z_end),
    "full_template_shape": [int(v) for v in template_arr.shape],
    "crop_shape": [int(v) for v in crop_shape],
    "important_note": (
        "This threshold mask is used only to define one shared rectangular crop box. "
        "It is not used as subject-specific skull stripping and is not applied as a hard mask."
    ),
}

with open(CROP_PLAN_JSON, "w") as f:
    json.dump(crop_params, f, indent=2)

crop_plan.to_csv(CROP_PLAN_REPORT, index=False)

display(crop_plan)

print("Saved shared crop plan CSV:")
print(CROP_PLAN_REPORT)

print("\nSaved shared crop plan JSON:")
print(CROP_PLAN_JSON)

# Recreate the visual QC figure.
x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

views = [
    (
        "Sagittal",
        template_arr[x_mid, :, :],
        crop_reference_mask[x_mid, :, :],
        crop_box_mask[x_mid, :, :],
    ),
    (
        "Coronal",
        template_arr[:, y_mid, :],
        crop_reference_mask[:, y_mid, :],
        crop_box_mask[:, y_mid, :],
    ),
    (
        "Axial",
        template_arr[:, :, z_mid],
        crop_reference_mask[:, :, z_mid],
        crop_box_mask[:, :, z_mid],
    ),
]

for row_idx, (view_name, template_slice, reference_mask_slice, crop_box_slice) in enumerate(views):
    axes[row_idx, 0].imshow(np.rot90(template_slice), cmap="gray")
    axes[row_idx, 0].set_title(f"Template {view_name.lower()}")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(np.rot90(template_slice), cmap="gray")
    axes[row_idx, 1].imshow(np.rot90(reference_mask_slice), alpha=0.35)
    axes[row_idx, 1].set_title(
        f"5% reference mask {view_name.lower()}\n"
        "for crop-box estimation only"
    )
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(np.rot90(template_slice), cmap="gray")
    axes[row_idx, 2].imshow(np.rot90(crop_box_slice), alpha=0.25)
    axes[row_idx, 2].set_title(
        f"Shared rectangular crop {view_name.lower()}\n"
        f"shape {tuple(int(v) for v in crop_shape)}"
    )
    axes[row_idx, 2].axis("off")

plt.tight_layout()

CROP_BOX_FIGURE = CROP_DIR / "shared_mni_crop_box_5pct_margin12_visual_qc.png"
plt.savefig(CROP_BOX_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved shared crop-box visual QC figure:")
print(CROP_BOX_FIGURE)

## 1.32. Define a tighter shared MNI crop box using the 10% template threshold

The 5% template threshold was anatomically safe, but it did not actually reduce the image size: the resulting crop box was still the full MNI template shape, 193 × 229 × 193. So it would not help with model memory or storage.

use the 10% template threshold instead, but only to estimate the outer rectangular crop box. I am not applying the 10% threshold mask as a hard mask to the subject images. This is important because the threshold mask can contain holes inside the brain, especially in darker T1 regions such as ventricles and CSF spaces. Those holes do not matter for crop-box estimation, because I only use the outer bounding box around the mask.

This is still not subject-specific skull stripping. The crop box will be one fixed rectangular region in MNI space, shared by every subject and applied identically to both the registered T1 channel and the log-Jacobian channel. The goal is only to remove unnecessary outer background and reduce the final model input size while keeping the same anatomical coordinate system for everyone.

In this cell, define and visually inspect the 10% shared crop box. not crop all subject images yet.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json

assert "template_img" in globals(), "template_img is not loaded."
assert "work_manifest" in globals(), "work_manifest is not available."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."

template_arr = template_img.numpy().astype(np.float32)

CROP_DIR = OUTPUT_REPORT.parent / "crop_planning"
CROP_DIR.mkdir(parents=True, exist_ok=True)

# I use this threshold only to estimate the outer crop box.
# I do not use this threshold mask as a hard mask on the subject images.
CROP_THRESHOLD_FRACTION = 0.10
CROP_MARGIN_VOXELS = 12

template_max = float(np.nanmax(template_arr))
crop_threshold_value = template_max * CROP_THRESHOLD_FRACTION

crop_reference_mask = template_arr > crop_threshold_value

coords = np.argwhere(crop_reference_mask)

crop_start = coords.min(axis=0)
crop_end = coords.max(axis=0) + 1

crop_start_with_margin = np.maximum(crop_start - CROP_MARGIN_VOXELS, 0)
crop_end_with_margin = np.minimum(
    crop_end + CROP_MARGIN_VOXELS,
    np.array(template_arr.shape),
)

x_start, y_start, z_start = [int(v) for v in crop_start_with_margin]
x_end, y_end, z_end = [int(v) for v in crop_end_with_margin]

crop_shape = crop_end_with_margin - crop_start_with_margin
crop_shape_tuple = tuple(int(v) for v in crop_shape)

# This rectangular mask is only for visualizing the crop region.
# It represents the region that would be kept after cropping.
crop_box_mask = np.zeros_like(template_arr, dtype=bool)
crop_box_mask[x_start:x_end, y_start:y_end, z_start:z_end] = True

voxels_per_channel = int(np.prod(crop_shape))
full_voxels_per_channel = int(np.prod(template_arr.shape))

estimated_gb_t1_only_float32 = (
    voxels_per_channel
    * len(work_manifest)
    * 4
    / (1024 ** 3)
)

estimated_gb_t1_plus_jacobian_float32 = (
    voxels_per_channel
    * len(work_manifest)
    * 2
    * 4
    / (1024 ** 3)
)

estimated_full_gb_t1_plus_jacobian_float32 = (
    full_voxels_per_channel
    * len(work_manifest)
    * 2
    * 4
    / (1024 ** 3)
)

crop_plan = pd.DataFrame([{
    "crop_threshold_fraction_of_template_max": CROP_THRESHOLD_FRACTION,
    "crop_threshold_value": crop_threshold_value,
    "crop_margin_voxels": CROP_MARGIN_VOXELS,
    "x_start": x_start,
    "x_end": x_end,
    "y_start": y_start,
    "y_end": y_end,
    "z_start": z_start,
    "z_end": z_end,
    "full_template_shape": tuple(template_arr.shape),
    "crop_shape": crop_shape_tuple,
    "voxels_per_channel_after_crop": voxels_per_channel,
    "full_voxels_per_channel": full_voxels_per_channel,
    "voxel_reduction_fraction": round(1 - (voxels_per_channel / full_voxels_per_channel), 4),
    "estimated_GB_T1_only_float32_after_crop": round(estimated_gb_t1_only_float32, 2),
    "estimated_GB_T1_plus_Jacobian_float32_after_crop": round(estimated_gb_t1_plus_jacobian_float32, 2),
    "estimated_GB_T1_plus_Jacobian_float32_full_grid": round(estimated_full_gb_t1_plus_jacobian_float32, 2),
}])

CROP_PLAN_REPORT = CROP_DIR / "shared_mni_crop_box_plan_10pct_margin12.csv"
CROP_PLAN_JSON = CROP_DIR / "shared_mni_crop_box_plan_10pct_margin12.json"

crop_plan.to_csv(CROP_PLAN_REPORT, index=False)

crop_params = {
    "crop_threshold_fraction_of_template_max": float(CROP_THRESHOLD_FRACTION),
    "crop_threshold_value": float(crop_threshold_value),
    "crop_margin_voxels": int(CROP_MARGIN_VOXELS),
    "x_start": int(x_start),
    "x_end": int(x_end),
    "y_start": int(y_start),
    "y_end": int(y_end),
    "z_start": int(z_start),
    "z_end": int(z_end),
    "full_template_shape": [int(v) for v in template_arr.shape],
    "crop_shape": [int(v) for v in crop_shape],
    "important_note": (
        "The 10% threshold mask is used only to define one shared rectangular crop box. "
        "It is not used as subject-specific skull stripping and is not applied as a hard mask."
    ),
}

with open(CROP_PLAN_JSON, "w") as f:
    json.dump(crop_params, f, indent=2)

display(crop_plan)

print("Saved shared crop plan CSV:")
print(CROP_PLAN_REPORT)

print("\nSaved shared crop plan JSON:")
print(CROP_PLAN_JSON)

# Visual QC on central template slices.
x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

views = [
    (
        "Sagittal",
        template_arr[x_mid, :, :],
        crop_reference_mask[x_mid, :, :],
        crop_box_mask[x_mid, :, :],
    ),
    (
        "Coronal",
        template_arr[:, y_mid, :],
        crop_reference_mask[:, y_mid, :],
        crop_box_mask[:, y_mid, :],
    ),
    (
        "Axial",
        template_arr[:, :, z_mid],
        crop_reference_mask[:, :, z_mid],
        crop_box_mask[:, :, z_mid],
    ),
]

for row_idx, (view_name, template_slice, reference_mask_slice, crop_box_slice) in enumerate(views):
    axes[row_idx, 0].imshow(np.rot90(template_slice), cmap="gray")
    axes[row_idx, 0].set_title(f"Template {view_name.lower()}")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(np.rot90(template_slice), cmap="gray")
    axes[row_idx, 1].imshow(np.rot90(reference_mask_slice), alpha=0.35)
    axes[row_idx, 1].set_title(
        f"10% reference mask {view_name.lower()}\n"
        "for crop-box estimation only"
    )
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(np.rot90(template_slice), cmap="gray")
    axes[row_idx, 2].imshow(np.rot90(crop_box_slice), alpha=0.25)
    axes[row_idx, 2].set_title(
        f"Shared rectangular crop {view_name.lower()}\n"
        f"shape {crop_shape_tuple}"
    )
    axes[row_idx, 2].axis("off")

plt.tight_layout()

CROP_BOX_FIGURE = CROP_DIR / "shared_mni_crop_box_10pct_margin12_visual_qc.png"
plt.savefig(CROP_BOX_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved shared crop-box visual QC figure:")
print(CROP_BOX_FIGURE)

## 1.33. Load and inspect a proper shared MNI brain mask

The threshold-based masks were not useful for cropping. A 5% threshold was too broad and produced the full MNI shape, while stricter thresholds created holes inside darker but real anatomical regions such as ventricles and CSF. therefore stop using template-intensity threshold masks as the crop strategy.

The mask is still useful, but for a different purpose: controlling the final model input, especially the log-Jacobian channel. The Jacobian maps can contain strong deformation values around the skull/head boundary and outer background. Those regions may reflect registration boundary effects rather than brain atrophy. To reduce that risk, use one proper shared MNI-space brain mask, not a subject-specific skull-stripping mask.

This is different from the skull-stripping problem described by Tinauer et al. because I am not creating a different brain outline for every subject. The mask will be fixed in MNI space and identical for all subjects. inspect it first and will not apply it to all images in this cell.

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."

MASK_DIR = PROJECT_DIR / "templates" / "masks"
MASK_DIR.mkdir(parents=True, exist_ok=True)

SHARED_BRAIN_MASK_PATH = MASK_DIR / "tpl-MNI152NLin2009cSym_res-1_desc-brain_mask.nii.gz"
SHARED_BRAIN_MASK_QC_REPORT = OUTPUT_REPORT.parent / "shared_mni_brain_mask_qc_report.csv"
SHARED_BRAIN_MASK_QC_FIGURE = OUTPUT_REPORT.parent / "shared_mni_brain_mask_visual_qc.png"

# Try to get the official TemplateFlow brain mask for the same template.
try:
    from templateflow.api import get as get_template

    downloaded_mask = get_template(
        "MNI152NLin2009cSym",
        resolution=1,
        desc="brain",
        suffix="mask",
    )

    if isinstance(downloaded_mask, (list, tuple)):
        downloaded_mask = downloaded_mask[0]

    downloaded_mask = Path(downloaded_mask)

    assert downloaded_mask.exists(), f"TemplateFlow returned a missing mask path: {downloaded_mask}"

    shutil.copyfile(downloaded_mask, SHARED_BRAIN_MASK_PATH)

    print("Copied TemplateFlow brain mask from:")
    print(downloaded_mask)

except Exception as exc:
    raise RuntimeError(
        "Could not load the TemplateFlow brain mask. "
        "This is better to fix than falling back to a threshold mask."
    ) from exc

shared_brain_mask_img = ants.image_read(str(SHARED_BRAIN_MASK_PATH))

template_arr = template_img.numpy().astype(np.float32)
mask_arr = shared_brain_mask_img.numpy()

# Convert possible soft/probabilistic mask values to binary for inspection.
mask_binary = mask_arr > 0

mask_qc = pd.DataFrame([{
    "shared_brain_mask_path": str(SHARED_BRAIN_MASK_PATH),
    "template_shape": tuple(template_img.shape),
    "mask_shape": tuple(shared_brain_mask_img.shape),
    "template_spacing": tuple(template_img.spacing),
    "mask_spacing": tuple(shared_brain_mask_img.spacing),
    "template_orientation": ants.get_orientation(template_img),
    "mask_orientation": ants.get_orientation(shared_brain_mask_img),
    "mask_unique_values_first_20": str(np.unique(mask_arr)[:20]),
    "mask_voxels": int(mask_binary.sum()),
    "total_voxels": int(mask_binary.size),
    "mask_fraction": float(mask_binary.sum() / mask_binary.size),
    "matches_template_shape": tuple(shared_brain_mask_img.shape) == tuple(template_img.shape),
    "matches_template_spacing": np.allclose(shared_brain_mask_img.spacing, template_img.spacing, atol=1e-5),
    "matches_template_orientation": ants.get_orientation(shared_brain_mask_img) == ants.get_orientation(template_img),
}])

mask_qc.to_csv(SHARED_BRAIN_MASK_QC_REPORT, index=False)

display(mask_qc)

print("Saved shared MNI brain mask:")
print(SHARED_BRAIN_MASK_PATH)

print("\nSaved shared brain mask QC report:")
print(SHARED_BRAIN_MASK_QC_REPORT)

# Visual QC: template, shared brain mask, and overlay.
x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

views = [
    ("Sagittal", template_arr[x_mid, :, :], mask_binary[x_mid, :, :]),
    ("Coronal", template_arr[:, y_mid, :], mask_binary[:, y_mid, :]),
    ("Axial", template_arr[:, :, z_mid], mask_binary[:, :, z_mid]),
]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for row_idx, (view_name, template_slice, mask_slice) in enumerate(views):
    axes[row_idx, 0].imshow(np.rot90(template_slice), cmap="gray")
    axes[row_idx, 0].set_title(f"Template {view_name.lower()}")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(np.rot90(mask_slice), cmap="gray")
    axes[row_idx, 1].set_title(f"Shared brain mask {view_name.lower()}")
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(np.rot90(template_slice), cmap="gray")
    axes[row_idx, 2].imshow(np.rot90(mask_slice), alpha=0.35)
    axes[row_idx, 2].set_title(f"Overlay {view_name.lower()}")
    axes[row_idx, 2].axis("off")

plt.tight_layout()
plt.savefig(SHARED_BRAIN_MASK_QC_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved shared MNI brain mask visual QC figure:")
print(SHARED_BRAIN_MASK_QC_FIGURE)

## 1.34. Restore the final SyN QC pass flag from saved QC reports

restore the final SyN registration QC status in the working manifest using the saved QC reports on Drive.

The original full SyN QC report contains the cohort-level registration and Jacobian checks. In that report, one subject had a corrupted registered T1 output. I already reran that subject successfully and saved its rerun QC report separately. In this step, use the original full QC report as the base, then apply the saved rerun result for that one subject.

This step does not rerun registration, does not modify the registered T1 files, and does not modify the log-Jacobian files. It only rebuilds the `syn_qc_pass` column in `work_manifest` so that later model-grid preprocessing uses the correct set of QC-passed subjects.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

assert "work_manifest" in globals(), "work_manifest is not available."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."

ORIGINAL_SYN_QC_REPORT = OUTPUT_REPORT.parent / "full_syn_registration_jacobian_qc_report.csv"
RERUN_FAILED_SUBJECT_QC_REPORT = OUTPUT_REPORT.parent / "rerun_failed_syn_subject_qc_report.csv"
CLEANED_SYN_QC_REPORT = OUTPUT_REPORT.parent / "full_syn_registration_jacobian_qc_report_cleaned_for_manifest.csv"

assert ORIGINAL_SYN_QC_REPORT.exists(), f"Original SyN QC report not found: {ORIGINAL_SYN_QC_REPORT}"
assert RERUN_FAILED_SUBJECT_QC_REPORT.exists(), f"Rerun subject QC report not found: {RERUN_FAILED_SUBJECT_QC_REPORT}"

def parse_bool(value):
    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    return str(value).strip().lower() in ["true", "1", "yes"]

def clean_key(value):
    if pd.isna(value):
        return ""

    try:
        return str(int(float(value)))
    except Exception:
        return str(value).strip()

# Load the original full QC report. This was the reliable report with only one failed subject.
syn_qc_report = pd.read_csv(ORIGINAL_SYN_QC_REPORT)

assert "qc_pass" in syn_qc_report.columns, "Original SyN QC report does not contain qc_pass."

syn_qc_report["RID_key"] = syn_qc_report["RID"].apply(clean_key)
syn_qc_report["image_id_key"] = syn_qc_report["image_id"].apply(clean_key)
syn_qc_report["syn_qc_pass_clean"] = syn_qc_report["qc_pass"].apply(parse_bool)

print("Base QC pass counts from original full QC report:")
display(syn_qc_report["syn_qc_pass_clean"].value_counts(dropna=False).to_frame("count"))

# Load the saved QC report for the one rerun subject.
rerun_qc = pd.read_csv(RERUN_FAILED_SUBJECT_QC_REPORT)

required_rerun_qc_cols = [
    "RID",
    "image_id",
    "mni_matches_template_shape",
    "logjacobian_matches_template_shape",
    "mni_matches_template_spacing",
    "logjacobian_matches_template_spacing",
    "mni_finite",
    "logjacobian_finite",
    "mni_nonzero_variation",
    "logjacobian_nonzero_variation",
]

missing_rerun_cols = [col for col in required_rerun_qc_cols if col not in rerun_qc.columns]
assert not missing_rerun_cols, f"Missing columns in rerun QC report: {missing_rerun_cols}"

rerun_qc["rerun_core_pass"] = True

for col in required_rerun_qc_cols[2:]:
    rerun_qc["rerun_core_pass"] = rerun_qc["rerun_core_pass"] & rerun_qc[col].apply(parse_bool)

print("\nRerun subject QC:")
display(
    rerun_qc[
        [
            "RID",
            "image_id",
            "final_group",
            "rerun_core_pass",
            "mni_min",
            "mni_max",
            "mni_mean",
            "mni_std",
            "logjacobian_min",
            "logjacobian_max",
            "logjacobian_mean",
            "logjacobian_std",
        ]
    ]
)

assert rerun_qc["rerun_core_pass"].all(), "The rerun subject QC did not pass."

# Mark the rerun subject as passed in the cleaned QC report.
for _, rerun_row in rerun_qc.iterrows():
    rerun_rid_key = clean_key(rerun_row["RID"])
    rerun_image_id_key = clean_key(rerun_row["image_id"])

    match_mask = (
        syn_qc_report["RID_key"].eq(rerun_rid_key)
        & syn_qc_report["image_id_key"].eq(rerun_image_id_key)
    )

    assert match_mask.sum() == 1, (
        f"Could not uniquely match rerun subject RID={rerun_rid_key}, "
        f"image_id={rerun_image_id_key} in the full QC report."
    )

    syn_qc_report.loc[match_mask, "syn_qc_pass_clean"] = True

print("\nCleaned QC pass counts after applying rerun fix:")
display(syn_qc_report["syn_qc_pass_clean"].value_counts(dropna=False).to_frame("count"))

# Save a cleaned QC report so future cells do not depend on confusing older columns.
syn_qc_report.to_csv(CLEANED_SYN_QC_REPORT, index=False)

print("\nSaved cleaned SyN QC report:")
print(CLEANED_SYN_QC_REPORT)

# Merge cleaned QC status back into work_manifest.
qc_lookup = (
    syn_qc_report
    .set_index(["RID_key", "image_id_key"])["syn_qc_pass_clean"]
    .to_dict()
)

work_manifest["syn_qc_pass"] = work_manifest.apply(
    lambda row: bool(
        qc_lookup.get(
            (clean_key(row[RID_COL]), clean_key(row[IMAGE_ID_COL])),
            False,
        )
    ),
    axis=1,
)

work_manifest["mni_nii_exists"] = work_manifest["mni_nii_path"].apply(lambda p: Path(p).exists())
work_manifest["logjacobian_nii_exists"] = work_manifest["logjacobian_nii_path"].apply(lambda p: Path(p).exists())

if "OUTPUT_MANIFEST" in globals():
    work_manifest.to_csv(OUTPUT_MANIFEST, index=False)

    print("\nSaved corrected working manifest:")
    print(OUTPUT_MANIFEST)

print("\nFinal restored syn_qc_pass counts:")
display(work_manifest["syn_qc_pass"].value_counts(dropna=False).to_frame("count"))

print("\nFinal restored syn_qc_pass by group:")
display(
    work_manifest
    .groupby(LABEL_COL)["syn_qc_pass"]
    .agg(["count", "sum"])
    .rename(columns={"count": "subjects", "sum": "passed"})
)

print("\nExisting MNI/log-Jacobian output counts:")
display(
    work_manifest[
        ["mni_nii_exists", "logjacobian_nii_exists"]
    ].sum().to_frame("count")
)

remaining_false = work_manifest.loc[~work_manifest["syn_qc_pass"]].copy()

print("\nRemaining subjects marked as failed:", len(remaining_false))

if len(remaining_false) > 0:
    display(
        remaining_false[
            [
                RID_COL,
                IMAGE_ID_COL,
                LABEL_COL,
                "mni_nii_exists",
                "logjacobian_nii_exists",
                "mni_nii_path",
                "logjacobian_nii_path",
            ]
        ].head(30)
    )

## 1.35. Pilot TriFormer-style and interaction pathway-style final image sizes

All SyN-registered T1 images and log-Jacobian maps are available for the cohort, so compare two candidate final model-input sizes before processing all subjects.

test a TriFormer-style cubic grid of 128 × 128 × 128 and a interaction pathway-style grid of 144 × 176 × 144. Both options will be applied to the same MRI channels: the registered T1 image and the log-Jacobian map. I am not applying a hard brain mask here, because that would make the T1 input resemble skull stripping.

This pilot is only for visual and numerical comparison. use a small balanced sample with one subject from each final group so I can check whether the smaller TriFormer-style grid loses visible anatomical detail compared with the larger interaction pathway-style grid.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

assert "ants" in globals(), "ANTsPy is not imported."
assert "work_manifest" in globals(), "work_manifest is not available."
assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."
assert "output_stem" in work_manifest.columns, "work_manifest is missing output_stem."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "mni_nii_path",
    "logjacobian_nii_path",
    "output_stem",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

FINAL_GRID_PILOT_DIR = PROJECT_DIR / "processed" / "clean_90d_mni_n4_syn_jacobian_final_grid_pilots"
FINAL_GRID_QC_DIR = OUTPUT_REPORT.parent / "final_grid_pilot_qc"

for folder in [FINAL_GRID_PILOT_DIR, FINAL_GRID_QC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

grid_options = {
    "triformer_128": {
        "shape": (128, 128, 128),
        "description": "TriFormer-style cubic 128 grid",
    },
    "threeMT_144x176x144": {
        "shape": (144, 176, 144),
        "description": "3MT-style 144 x 176 x 144 grid",
    },
}

pilot_pool = work_manifest.copy()

pilot_pool["mni_nii_exists"] = pilot_pool["mni_nii_path"].apply(lambda p: Path(p).exists())
pilot_pool["logjacobian_nii_exists"] = pilot_pool["logjacobian_nii_path"].apply(lambda p: Path(p).exists())

pilot_pool = pilot_pool.loc[
    pilot_pool["mni_nii_exists"]
    & pilot_pool["logjacobian_nii_exists"]
].copy()

assert len(pilot_pool) > 0, "No subjects with both MNI T1 and log-Jacobian files were found."

# I use one subject per final group so the visual check is not based on only one diagnostic category.
pilot_rows = (
    pilot_pool
    .sort_values([LABEL_COL, RID_COL, IMAGE_ID_COL])
    .groupby(LABEL_COL, group_keys=False)
    .head(1)
    .copy()
)

print("Pilot subjects selected:")
display(pilot_rows[[RID_COL, IMAGE_ID_COL, LABEL_COL, "mni_nii_path", "logjacobian_nii_path"]])

summary_rows = []
visual_records = []

for _, subject_row in pilot_rows.iterrows():
    input_t1_path = Path(subject_row["mni_nii_path"])
    input_logjacobian_path = Path(subject_row["logjacobian_nii_path"])
    subject_stem = subject_row["output_stem"]

    t1_img = ants.image_read(str(input_t1_path))
    logjacobian_img = ants.image_read(str(input_logjacobian_path))

    t1_arr = t1_img.numpy().astype(np.float32)
    logjacobian_arr = logjacobian_img.numpy().astype(np.float32)

    visual_records.append({
        "label": f"Original T1 | {subject_row[LABEL_COL]} RID {subject_row[RID_COL]}",
        "arr": t1_arr,
        "channel": "t1",
    })
    visual_records.append({
        "label": f"Original log-Jac | {subject_row[LABEL_COL]} RID {subject_row[RID_COL]}",
        "arr": logjacobian_arr,
        "channel": "logjacobian",
    })

    for grid_name, grid_info in grid_options.items():
        target_shape = grid_info["shape"]

        grid_dir = FINAL_GRID_PILOT_DIR / grid_name
        t1_dir = grid_dir / "t1"
        logjacobian_dir = grid_dir / "logjacobian"

        for folder in [grid_dir, t1_dir, logjacobian_dir]:
            folder.mkdir(parents=True, exist_ok=True)

        output_t1_path = t1_dir / f"{subject_stem}_space-MNI152NLin2009cSym_desc-n4Syn_grid-{grid_name}_T1w.nii.gz"
        output_logjacobian_path = logjacobian_dir / f"{subject_stem}_space-MNI152NLin2009cSym_desc-logJacobian_grid-{grid_name}.nii.gz"

        # Linear interpolation is used because both channels are continuous scalar images.
        t1_resampled_img = ants.resample_image(
            t1_img,
            target_shape,
            use_voxels=True,
            interp_type=0,
        )

        logjacobian_resampled_img = ants.resample_image(
            logjacobian_img,
            target_shape,
            use_voxels=True,
            interp_type=0,
        )

        ants.image_write(t1_resampled_img, str(output_t1_path))
        ants.image_write(logjacobian_resampled_img, str(output_logjacobian_path))

        t1_resampled_arr = t1_resampled_img.numpy().astype(np.float32)
        logjacobian_resampled_arr = logjacobian_resampled_img.numpy().astype(np.float32)

        voxels_per_channel = int(np.prod(target_shape))

        estimated_gb_t1_only_float32 = (
            voxels_per_channel
            * len(work_manifest)
            * 4
            / (1024 ** 3)
        )

        estimated_gb_t1_plus_logjacobian_float32 = (
            voxels_per_channel
            * len(work_manifest)
            * 2
            * 4
            / (1024 ** 3)
        )

        summary_rows.append({
            "RID": subject_row[RID_COL],
            "image_id": subject_row[IMAGE_ID_COL],
            "final_group": subject_row[LABEL_COL],
            "grid_name": grid_name,
            "grid_description": grid_info["description"],
            "target_shape": target_shape,
            "output_t1_shape": tuple(t1_resampled_img.shape),
            "output_logjacobian_shape": tuple(logjacobian_resampled_img.shape),
            "output_t1_spacing": tuple(t1_resampled_img.spacing),
            "output_logjacobian_spacing": tuple(logjacobian_resampled_img.spacing),
            "voxels_per_channel": voxels_per_channel,
            "estimated_GB_T1_only_float32_full_cohort": round(estimated_gb_t1_only_float32, 2),
            "estimated_GB_T1_plus_logjacobian_float32_full_cohort": round(estimated_gb_t1_plus_logjacobian_float32, 2),
            "output_t1_min": float(np.nanmin(t1_resampled_arr)),
            "output_t1_max": float(np.nanmax(t1_resampled_arr)),
            "output_t1_mean": float(np.nanmean(t1_resampled_arr)),
            "output_t1_std": float(np.nanstd(t1_resampled_arr)),
            "output_logjacobian_min": float(np.nanmin(logjacobian_resampled_arr)),
            "output_logjacobian_max": float(np.nanmax(logjacobian_resampled_arr)),
            "output_logjacobian_mean": float(np.nanmean(logjacobian_resampled_arr)),
            "output_logjacobian_std": float(np.nanstd(logjacobian_resampled_arr)),
            "output_t1_finite": bool(np.isfinite(t1_resampled_arr).all()),
            "output_logjacobian_finite": bool(np.isfinite(logjacobian_resampled_arr).all()),
            "output_t1_nonzero_variation": bool(np.nanstd(t1_resampled_arr) > 0),
            "output_logjacobian_nonzero_variation": bool(np.nanstd(logjacobian_resampled_arr) > 0),
            "output_t1_path": str(output_t1_path),
            "output_logjacobian_path": str(output_logjacobian_path),
        })

        visual_records.append({
            "label": f"{grid_name} T1 | {subject_row[LABEL_COL]} RID {subject_row[RID_COL]}",
            "arr": t1_resampled_arr,
            "channel": "t1",
        })
        visual_records.append({
            "label": f"{grid_name} log-Jac | {subject_row[LABEL_COL]} RID {subject_row[RID_COL]}",
            "arr": logjacobian_resampled_arr,
            "channel": "logjacobian",
        })

final_grid_pilot_report = pd.DataFrame(summary_rows)

FINAL_GRID_PILOT_REPORT = FINAL_GRID_QC_DIR / "pilot_triformer128_vs_3mt144x176x144_resampling_report.csv"
final_grid_pilot_report.to_csv(FINAL_GRID_PILOT_REPORT, index=False)

print("Saved final-grid pilot report:")
print(FINAL_GRID_PILOT_REPORT)

display(final_grid_pilot_report)

# Visual QC: central sagittal, coronal, and axial slices for every pilot output.
fig, axes = plt.subplots(
    len(visual_records),
    3,
    figsize=(12, 2.8 * len(visual_records)),
)

if len(visual_records) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, record in enumerate(visual_records):
    arr = record["arr"]

    x_mid = arr.shape[0] // 2
    y_mid = arr.shape[1] // 2
    z_mid = arr.shape[2] // 2

    display_slices = [
        arr[x_mid, :, :],
        arr[:, y_mid, :],
        arr[:, :, z_mid],
    ]

    col_titles = ["Sagittal", "Coronal", "Axial"]

    low, high = np.nanpercentile(arr, [1, 99])

    for col_idx, image_slice in enumerate(display_slices):
        ax = axes[row_idx, col_idx]

        if record["channel"] == "t1":
            ax.imshow(
                np.rot90(image_slice),
                cmap="gray",
                vmin=low,
                vmax=high,
            )
        else:
            ax.imshow(
                np.rot90(image_slice),
                vmin=low,
                vmax=high,
            )

        ax.set_title(f"{record['label']} | {col_titles[col_idx]}")
        ax.axis("off")

plt.tight_layout()

FINAL_GRID_PILOT_FIGURE = FINAL_GRID_QC_DIR / "pilot_triformer128_vs_3mt144x176x144_visual_qc.png"
plt.savefig(FINAL_GRID_PILOT_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved final-grid pilot visual QC figure:")
print(FINAL_GRID_PILOT_FIGURE)

## 1.36. Estimate a shared crop box from my own registered MRI cohort

Instead of choosing a final image size only because it was used by TriFormer or interaction pathway, estimate a shared crop box from my own MNI-registered ADNI cohort.

The goal is to measure where meaningful T1 image content actually appears across the 1063 registered subjects. calculate a simple foreground bounding box for each registered T1 image, then summarize those bounding boxes across the cohort. From these statistics, I can compare conservative and slightly tighter shared crop boxes.

This does not apply subject-specific cropping to the model inputs. The per-subject foreground boxes are used only to estimate one shared crop region in MNI space. Later, the same final crop box would be applied identically to every subject and to both MRI channels: the registered T1 image and the log-Jacobian map.

I am not applying a brain mask here, and I am not skull-stripping the images. This step only studies the cohort-level spatial extent of useful image content so I can make a data-driven decision about the final model input size.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
import time

assert "ants" in globals(), "ANTsPy is not imported."
assert "work_manifest" in globals(), "work_manifest is not available."
assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."
assert "template_img" in globals(), "template_img is not loaded."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "mni_nii_path",
    "logjacobian_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

DATA_CROP_DIR = OUTPUT_REPORT.parent / "data_driven_crop_planning"
DATA_CROP_DIR.mkdir(parents=True, exist_ok=True)

SUBJECT_BBOX_REPORT = DATA_CROP_DIR / "subject_t1_foreground_bounding_boxes.csv"
CANDIDATE_CROP_REPORT = DATA_CROP_DIR / "shared_crop_box_candidates_from_cohort.csv"
CANDIDATE_CLIP_REPORT = DATA_CROP_DIR / "shared_crop_box_candidate_clip_counts.csv"
CANDIDATE_CROP_JSON = DATA_CROP_DIR / "shared_crop_box_candidates_from_cohort.json"
CANDIDATE_CROP_FIGURE = DATA_CROP_DIR / "shared_crop_box_candidates_visual_qc.png"

# I use the registered T1 images to estimate where real image content exists.
# The resulting shared crop box will later be applied to both T1 and log-Jacobian.
crop_pool = work_manifest.copy()

crop_pool["mni_nii_exists"] = crop_pool["mni_nii_path"].apply(lambda p: Path(p).exists())
crop_pool["logjacobian_nii_exists"] = crop_pool["logjacobian_nii_path"].apply(lambda p: Path(p).exists())

crop_pool = crop_pool.loc[
    crop_pool["mni_nii_exists"]
    & crop_pool["logjacobian_nii_exists"]
].copy()

assert len(crop_pool) == len(work_manifest), (
    f"Expected all {len(work_manifest)} subjects to have both outputs, "
    f"but found {len(crop_pool)} complete subjects."
)

# This threshold is only for estimating foreground extent.
# It is not used as a mask for model input.
FOREGROUND_POSITIVE_PERCENTILE = 1.0
MIN_FOREGROUND_THRESHOLD = 1e-3

bbox_rows = []

start_time = time.time()

for row_idx, (_, row) in enumerate(crop_pool.iterrows(), start=1):
    t1_path = Path(row["mni_nii_path"])

    t1_img = ants.image_read(str(t1_path))
    t1_arr = t1_img.numpy().astype(np.float32)

    finite_mask = np.isfinite(t1_arr)
    positive_values = t1_arr[finite_mask & (t1_arr > MIN_FOREGROUND_THRESHOLD)]

    assert positive_values.size > 0, f"No positive foreground values found for: {t1_path}"

    # The percentile threshold removes tiny interpolation halos while keeping the real head/brain extent.
    foreground_threshold = max(
        float(np.percentile(positive_values, FOREGROUND_POSITIVE_PERCENTILE)),
        MIN_FOREGROUND_THRESHOLD,
    )

    foreground_mask = finite_mask & (t1_arr > foreground_threshold)

    coords = np.argwhere(foreground_mask)

    assert coords.size > 0, f"Foreground mask is empty for: {t1_path}"

    bbox_start = coords.min(axis=0)
    bbox_end = coords.max(axis=0) + 1

    bbox_rows.append({
        "RID": row[RID_COL],
        "image_id": row[IMAGE_ID_COL],
        "final_group": row[LABEL_COL],
        "mni_nii_path": str(t1_path),
        "foreground_threshold": foreground_threshold,
        "x_start": int(bbox_start[0]),
        "x_end": int(bbox_end[0]),
        "y_start": int(bbox_start[1]),
        "y_end": int(bbox_end[1]),
        "z_start": int(bbox_start[2]),
        "z_end": int(bbox_end[2]),
        "x_size": int(bbox_end[0] - bbox_start[0]),
        "y_size": int(bbox_end[1] - bbox_start[1]),
        "z_size": int(bbox_end[2] - bbox_start[2]),
    })

    if row_idx % 50 == 0 or row_idx == len(crop_pool):
        elapsed_minutes = (time.time() - start_time) / 60
        print(
            f"Processed {row_idx}/{len(crop_pool)} subjects "
            f"for crop statistics in {elapsed_minutes:.1f} minutes."
        )

subject_bbox_stats = pd.DataFrame(bbox_rows)
subject_bbox_stats.to_csv(SUBJECT_BBOX_REPORT, index=False)

print("Saved subject foreground bounding boxes:")
print(SUBJECT_BBOX_REPORT)

display(subject_bbox_stats.head())

# Build several shared crop-box candidates from cohort statistics.
# 100% union is safest but may crop less.
# 99% and 95% candidates are included only for comparison and must be checked for clipping.
template_shape = np.array(template_img.shape, dtype=int)
n_subjects = len(subject_bbox_stats)

candidate_specs = [
    {
        "candidate_name": "union_100pct_margin8",
        "start_quantile": 0.00,
        "end_quantile": 1.00,
        "margin_voxels": 8,
        "description": "Safest shared crop: contains all measured T1 foreground boxes plus margin.",
    },
    {
        "candidate_name": "quantile_99pct_margin12",
        "start_quantile": 0.01,
        "end_quantile": 0.99,
        "margin_voxels": 12,
        "description": "Very conservative quantile crop: may remove rare outlier extent, with generous margin.",
    },
    {
        "candidate_name": "quantile_95pct_margin12",
        "start_quantile": 0.05,
        "end_quantile": 0.95,
        "margin_voxels": 12,
        "description": "More aggressive comparison crop: useful only if clipping counts remain acceptable.",
    },
]

candidate_rows = []
clip_rows = []

starts = subject_bbox_stats[["x_start", "y_start", "z_start"]].to_numpy()
ends = subject_bbox_stats[["x_end", "y_end", "z_end"]].to_numpy()

for spec in candidate_specs:
    margin = spec["margin_voxels"]

    raw_start = np.floor(
        np.quantile(starts, spec["start_quantile"], axis=0)
    ).astype(int)

    raw_end = np.ceil(
        np.quantile(ends, spec["end_quantile"], axis=0)
    ).astype(int)

    candidate_start = np.maximum(raw_start - margin, 0)
    candidate_end = np.minimum(raw_end + margin, template_shape)

    x_start, y_start, z_start = [int(v) for v in candidate_start]
    x_end, y_end, z_end = [int(v) for v in candidate_end]

    crop_shape = candidate_end - candidate_start
    voxels_per_channel = int(np.prod(crop_shape))
    full_voxels_per_channel = int(np.prod(template_shape))

    clipped_mask = (
        (subject_bbox_stats["x_start"] < x_start)
        | (subject_bbox_stats["x_end"] > x_end)
        | (subject_bbox_stats["y_start"] < y_start)
        | (subject_bbox_stats["y_end"] > y_end)
        | (subject_bbox_stats["z_start"] < z_start)
        | (subject_bbox_stats["z_end"] > z_end)
    )

    clipped_subjects = subject_bbox_stats.loc[clipped_mask].copy()

    estimated_gb_two_channels_float32 = (
        voxels_per_channel
        * n_subjects
        * 2
        * 4
        / (1024 ** 3)
    )

    estimated_gb_full_grid_two_channels_float32 = (
        full_voxels_per_channel
        * n_subjects
        * 2
        * 4
        / (1024 ** 3)
    )

    candidate_rows.append({
        "candidate_name": spec["candidate_name"],
        "description": spec["description"],
        "start_quantile": spec["start_quantile"],
        "end_quantile": spec["end_quantile"],
        "margin_voxels": margin,
        "x_start": x_start,
        "x_end": x_end,
        "y_start": y_start,
        "y_end": y_end,
        "z_start": z_start,
        "z_end": z_end,
        "full_template_shape": tuple(int(v) for v in template_shape),
        "crop_shape": tuple(int(v) for v in crop_shape),
        "voxels_per_channel": voxels_per_channel,
        "full_voxels_per_channel": full_voxels_per_channel,
        "voxel_reduction_fraction": round(1 - (voxels_per_channel / full_voxels_per_channel), 4),
        "estimated_GB_T1_plus_logJac_float32_after_crop": round(estimated_gb_two_channels_float32, 2),
        "estimated_GB_T1_plus_logJac_float32_full_grid": round(estimated_gb_full_grid_two_channels_float32, 2),
        "clipped_subject_count": int(len(clipped_subjects)),
        "clipped_subject_fraction": round(len(clipped_subjects) / n_subjects, 4),
    })

    if len(clipped_subjects) > 0:
        clipped_by_group = (
            clipped_subjects
            .groupby("final_group")
            .size()
            .reset_index(name="clipped_subject_count")
        )

        for _, group_row in clipped_by_group.iterrows():
            clip_rows.append({
                "candidate_name": spec["candidate_name"],
                "final_group": group_row["final_group"],
                "clipped_subject_count": int(group_row["clipped_subject_count"]),
            })
    else:
        clip_rows.append({
            "candidate_name": spec["candidate_name"],
            "final_group": "ALL",
            "clipped_subject_count": 0,
        })

candidate_crop_boxes = pd.DataFrame(candidate_rows)
candidate_clip_counts = pd.DataFrame(clip_rows)

candidate_crop_boxes.to_csv(CANDIDATE_CROP_REPORT, index=False)
candidate_clip_counts.to_csv(CANDIDATE_CLIP_REPORT, index=False)

candidate_json = candidate_crop_boxes.to_dict(orient="records")

with open(CANDIDATE_CROP_JSON, "w") as f:
    json.dump(candidate_json, f, indent=2)

print("\nSaved shared crop-box candidates:")
print(CANDIDATE_CROP_REPORT)

print("\nSaved candidate clip counts:")
print(CANDIDATE_CLIP_REPORT)

print("\nSaved shared crop-box candidate JSON:")
print(CANDIDATE_CROP_JSON)

display(candidate_crop_boxes)
display(candidate_clip_counts)

# Visual QC of candidate crop boxes on the template.
template_arr = template_img.numpy().astype(np.float32)

x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

fig, axes = plt.subplots(
    len(candidate_crop_boxes),
    3,
    figsize=(12, 3.5 * len(candidate_crop_boxes)),
)

if len(candidate_crop_boxes) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, candidate in candidate_crop_boxes.iterrows():
    crop_box_mask = np.zeros_like(template_arr, dtype=bool)

    crop_box_mask[
        int(candidate["x_start"]):int(candidate["x_end"]),
        int(candidate["y_start"]):int(candidate["y_end"]),
        int(candidate["z_start"]):int(candidate["z_end"]),
    ] = True

    views = [
        (
            "Sagittal",
            template_arr[x_mid, :, :],
            crop_box_mask[x_mid, :, :],
        ),
        (
            "Coronal",
            template_arr[:, y_mid, :],
            crop_box_mask[:, y_mid, :],
        ),
        (
            "Axial",
            template_arr[:, :, z_mid],
            crop_box_mask[:, :, z_mid],
        ),
    ]

    for col_idx, (view_name, template_slice, crop_slice) in enumerate(views):
        ax = axes[row_idx, col_idx]

        ax.imshow(np.rot90(template_slice), cmap="gray")
        ax.imshow(np.rot90(crop_slice), alpha=0.25)

        ax.set_title(
            f"{candidate['candidate_name']} | {view_name}\n"
            f"shape {candidate['crop_shape']} | clipped {candidate['clipped_subject_count']}/{n_subjects}"
        )
        ax.axis("off")

plt.tight_layout()
plt.savefig(CANDIDATE_CROP_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved candidate crop-box visual QC figure:")
print(CANDIDATE_CROP_FIGURE)

## 1.37. Result of data-driven crop-box planning from cohort foreground statistics

This run estimated foreground bounding boxes from all 1063 MNI-registered T1 images and used those measurements to test shared crop-box candidates. The run completed for the full cohort and saved the subject-level bounding boxes, candidate crop-box table, clipping-count table, and visual QC figure.

The result shows that this foreground-box method did not produce a useful crop. All tested shared crop candidates kept the full MNI grid:

`193 × 229 × 193`

This happened for all three candidates: `union_100pct_margin8`, `quantile_99pct_margin12`, and `quantile_95pct_margin12`. Their voxel reduction was `0.0`, and their estimated storage remained the same as the full grid. The clipping count was also `0/1063`, but this is only because the proposed crop boxes were still the full image volume.

The subject-level bounding boxes explain why this happened. The detected foreground often extended to the image boundaries, with starts at `0` and ends at the full template size along x, y, and z. This likely reflects low-level nonzero signal, interpolation halo, padding effects, or head/background signal near the volume borders after registration. Therefore, the method was too conservative and interpreted the useful image extent as the whole MNI field of view.

The conclusion from this run is that conservative data-driven rectangular cropping is not justified using the current foreground definition. At this stage, the safest interpretation is that the registered cohort should preserve the full MNI field of view, and any size reduction should be handled by full-field resampling rather than by applying this crop-box method.

## 1.38. Estimate a fixed rectangular MNI crop from the shared template brain mask

The previous data-driven crop attempt used voxelwise foreground detection on each registered T1 image. That method was too sensitive to tiny nonzero interpolation/background values near the image borders, so the estimated foreground boxes often touched the full MNI volume boundary.

Here use the shared MNI brain mask only to estimate a rectangular crop box. I am not applying the mask to the subject images. The mask is used only to find a central anatomical region in MNI coordinates, then I expand that box with generous margins so that the final crop still preserves surrounding skull/scalp/background context. The same rectangular crop would later be applied identically to every subject and to both channels: registered T1 and log-Jacobian.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "ants" in globals(), "ANTsPy is not imported."
assert "template_img" in globals(), "template_img is not loaded."
assert "work_manifest" in globals(), "work_manifest is not available."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."
assert "SHARED_BRAIN_MASK_PATH" in globals(), "SHARED_BRAIN_MASK_PATH is not defined."

RECT_CROP_DIR = OUTPUT_REPORT.parent / "fixed_rectangular_mni_crop_planning"
RECT_CROP_DIR.mkdir(parents=True, exist_ok=True)

RECT_CROP_REPORT = RECT_CROP_DIR / "mni_mask_based_rectangular_crop_candidates.csv"
RECT_CROP_JSON = RECT_CROP_DIR / "mni_mask_based_rectangular_crop_candidates.json"
RECT_CROP_FIGURE = RECT_CROP_DIR / "mni_mask_based_rectangular_crop_candidates_visual_qc.png"

shared_mask_img = ants.image_read(str(SHARED_BRAIN_MASK_PATH))

assert tuple(shared_mask_img.shape) == tuple(template_img.shape), (
    f"Mask shape {shared_mask_img.shape} does not match template shape {template_img.shape}."
)

mask_arr = shared_mask_img.numpy() > 0
template_arr = template_img.numpy().astype(np.float32)

mask_coords = np.argwhere(mask_arr)

brain_box_start = mask_coords.min(axis=0)
brain_box_end = mask_coords.max(axis=0) + 1

template_shape = np.array(template_img.shape, dtype=int)
full_voxels_per_channel = int(np.prod(template_shape))
n_subjects = len(work_manifest)

# I test several margins because the brain mask itself is too tight.
# The final crop must include generous surrounding tissue/context and not behave like skull stripping.
margin_options = [16, 24, 32, 40, 48]

candidate_rows = []

for margin in margin_options:
    crop_start = np.maximum(brain_box_start - margin, 0)
    crop_end = np.minimum(brain_box_end + margin, template_shape)

    crop_shape = crop_end - crop_start
    voxels_per_channel = int(np.prod(crop_shape))

    estimated_gb_two_channels_float32 = (
        voxels_per_channel
        * n_subjects
        * 2
        * 4
        / (1024 ** 3)
    )

    estimated_gb_full_grid_two_channels_float32 = (
        full_voxels_per_channel
        * n_subjects
        * 2
        * 4
        / (1024 ** 3)
    )

    candidate_rows.append({
        "candidate_name": f"mni_brainbox_margin{margin}",
        "margin_voxels": margin,
        "x_start": int(crop_start[0]),
        "x_end": int(crop_end[0]),
        "y_start": int(crop_start[1]),
        "y_end": int(crop_end[1]),
        "z_start": int(crop_start[2]),
        "z_end": int(crop_end[2]),
        "full_template_shape": tuple(int(v) for v in template_shape),
        "crop_shape": tuple(int(v) for v in crop_shape),
        "voxels_per_channel": voxels_per_channel,
        "full_voxels_per_channel": full_voxels_per_channel,
        "voxel_reduction_fraction": round(1 - (voxels_per_channel / full_voxels_per_channel), 4),
        "estimated_GB_T1_plus_logJac_float32_after_crop": round(estimated_gb_two_channels_float32, 2),
        "estimated_GB_T1_plus_logJac_float32_full_grid": round(estimated_gb_full_grid_two_channels_float32, 2),
    })

crop_candidates = pd.DataFrame(candidate_rows)

crop_candidates.to_csv(RECT_CROP_REPORT, index=False)

with open(RECT_CROP_JSON, "w") as f:
    json.dump(crop_candidates.to_dict(orient="records"), f, indent=2)

print("Saved fixed rectangular crop candidates:")
print(RECT_CROP_REPORT)

display(crop_candidates)

# Visual QC: overlay each rectangular crop candidate on the template.
x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

fig, axes = plt.subplots(
    len(crop_candidates),
    3,
    figsize=(12, 3.2 * len(crop_candidates)),
)

if len(crop_candidates) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, candidate in crop_candidates.iterrows():
    crop_box_mask = np.zeros_like(template_arr, dtype=bool)

    crop_box_mask[
        int(candidate["x_start"]):int(candidate["x_end"]),
        int(candidate["y_start"]):int(candidate["y_end"]),
        int(candidate["z_start"]):int(candidate["z_end"]),
    ] = True

    views = [
        ("Sagittal", template_arr[x_mid, :, :], crop_box_mask[x_mid, :, :]),
        ("Coronal", template_arr[:, y_mid, :], crop_box_mask[:, y_mid, :]),
        ("Axial", template_arr[:, :, z_mid], crop_box_mask[:, :, z_mid]),
    ]

    for col_idx, (view_name, template_slice, crop_slice) in enumerate(views):
        ax = axes[row_idx, col_idx]

        ax.imshow(np.rot90(template_slice), cmap="gray")
        ax.imshow(np.rot90(crop_slice), alpha=0.20)

        ax.set_title(
            f"{candidate['candidate_name']} | {view_name}\n"
            f"crop shape {candidate['crop_shape']}"
        )
        ax.axis("off")

plt.tight_layout()
plt.savefig(RECT_CROP_FIGURE, dpi=150, bbox_inches="tight")
plt.show()

print("Saved fixed rectangular crop visual QC figure:")
print(RECT_CROP_FIGURE)

## 1.39. Preview a 177 × 213 × 183 fixed rectangular crop

After previewing the `177 × 213 × 177` crop, I noticed that the third dimension was still too tight and slightly cropped the superior scalp/head boundary. Because of this visual QC finding, I decided to increase only the third dimension from `177` to `183` while keeping the first two dimensions unchanged.

This preview checks whether the adjusted `177 × 213 × 183` rectangular crop preserves the full head boundary better while still reducing unused outer MNI padding. This is only a visual inspection step. I am not applying the crop to subject images yet, and I am not using a brain-shaped or scalp-shaped mask. The crop remains a fixed rectangular box in MNI space that would be applied identically to all subjects if accepted.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

assert "template_img" in globals(), "template_img is not loaded."
assert "crop_candidates" in globals(), "crop_candidates is not available."

template_arr = template_img.numpy().astype(np.float32)
template_shape = np.array(template_arr.shape, dtype=int)

# I use the margin-16 crop as the starting point because it was the only candidate
# that gave meaningful size reduction without immediately returning to the full MNI grid.
margin16_row = crop_candidates.loc[
    crop_candidates["candidate_name"] == "mni_brainbox_margin16"
].iloc[0]

base_start = np.array([
    int(margin16_row["x_start"]),
    int(margin16_row["y_start"]),
    int(margin16_row["z_start"]),
], dtype=int)

base_end = np.array([
    int(margin16_row["x_end"]),
    int(margin16_row["y_end"]),
    int(margin16_row["z_end"]),
], dtype=int)

base_shape = base_end - base_start

# I preview the crop that visually preserved the superior head boundary better.
preview_shape = np.array([177, 213, 183], dtype=int)
preview_shape_label = "177×213×183"

extra_voxels = preview_shape - base_shape

assert np.all(extra_voxels >= 0), (
    f"The preview shape {preview_shape.tolist()} is smaller than the base margin-16 shape "
    f"{base_shape.tolist()} in at least one dimension."
)

extra_before = extra_voxels // 2
extra_after = extra_voxels - extra_before

preview_start = base_start - extra_before
preview_end = base_end + extra_after

# I keep the preview crop inside the image boundaries.
underflow = np.maximum(-preview_start, 0)
preview_start += underflow
preview_end += underflow

overflow = np.maximum(preview_end - template_shape, 0)
preview_start -= overflow
preview_end -= overflow

preview_start = np.maximum(preview_start, 0)
preview_end = np.minimum(preview_end, template_shape)

actual_preview_shape = preview_end - preview_start

assert np.array_equal(actual_preview_shape, preview_shape), (
    f"Expected preview shape {preview_shape.tolist()}, "
    f"but got {actual_preview_shape.tolist()}."
)

print("Base margin-16 crop:")
print("start:", base_start.tolist())
print("end:  ", base_end.tolist())
print("shape:", base_shape.tolist())

print("\nPreview crop:")
print("start:", preview_start.tolist())
print("end:  ", preview_end.tolist())
print("shape:", actual_preview_shape.tolist())

# I create only a temporary rectangular overlay for visualization.
# This is not a brain mask and is not applied to the subject images.
preview_box = np.zeros_like(template_arr, dtype=bool)
preview_box[
    preview_start[0]:preview_end[0],
    preview_start[1]:preview_end[1],
    preview_start[2]:preview_end[2],
] = True

x_mid = template_arr.shape[0] // 2
y_mid = template_arr.shape[1] // 2
z_mid = template_arr.shape[2] // 2

views = [
    ("Sagittal", template_arr[x_mid, :, :], preview_box[x_mid, :, :]),
    ("Coronal", template_arr[:, y_mid, :], preview_box[:, y_mid, :]),
    ("Axial", template_arr[:, :, z_mid], preview_box[:, :, z_mid]),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (view_name, template_slice, crop_slice) in zip(axes, views):
    ax.imshow(np.rot90(template_slice), cmap="gray")
    ax.imshow(np.rot90(crop_slice), alpha=0.20)
    ax.set_title(f"Preview {preview_shape_label} | {view_name}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 1.40. Save the accepted fixed rectangular MNI crop

After visually previewing the fixed rectangular crop candidates, I selected a `177 × 213 × 183` crop. The original margin-16 candidate had shape `177 × 213 × 177`, but visual QC showed that the third dimension was slightly too tight and cropped the superior scalp/head boundary. I therefore increased only the third dimension from `177` to `183` while keeping the first two dimensions unchanged.

This cell saves the accepted crop coordinates for later preprocessing. The crop is a fixed rectangular box in MNI space, not a brain mask and not a subject-specific crop. If used later, the same crop coordinates will be applied identically to every subject and to both MRI channels: registered T1 and log-Jacobian.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

assert "template_img" in globals(), "template_img is not loaded."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."

RECT_CROP_DIR = OUTPUT_REPORT.parent / "fixed_rectangular_mni_crop_planning"
RECT_CROP_REPORT = RECT_CROP_DIR / "mni_mask_based_rectangular_crop_candidates.csv"

assert RECT_CROP_REPORT.exists(), f"Crop candidate report not found: {RECT_CROP_REPORT}"

crop_candidates = pd.read_csv(RECT_CROP_REPORT)

# I remove the previously saved incorrect crop definition, if it exists.
WRONG_CROP_JSON = RECT_CROP_DIR / "custom_mni_rectcrop_177x215x177.json"

if WRONG_CROP_JSON.exists():
    WRONG_CROP_JSON.unlink()
    print("Deleted incorrect crop JSON:")
    print(WRONG_CROP_JSON)
else:
    print("No incorrect crop JSON found to delete:")
    print(WRONG_CROP_JSON)

margin16_row = crop_candidates.loc[
    crop_candidates["candidate_name"] == "mni_brainbox_margin16"
].iloc[0]

template_shape = np.array(template_img.shape, dtype=int)

# I now use the visually accepted crop shape.
# The first two dimensions stay as the margin-16 crop.
# The third dimension is increased from 177 to 183 to avoid cropping the superior scalp/head boundary.
target_shape = np.array([177, 213, 183], dtype=int)

crop_start = np.array([
    int(margin16_row["x_start"]),
    int(margin16_row["y_start"]),
    int(margin16_row["z_start"]),
], dtype=int)

crop_end = np.array([
    int(margin16_row["x_end"]),
    int(margin16_row["y_end"]),
    int(margin16_row["z_end"]),
], dtype=int)

current_shape = crop_end - crop_start

print("\nMargin-16 crop:")
print("start:", crop_start.tolist())
print("end:  ", crop_end.tolist())
print("shape:", current_shape.tolist())

extra_voxels = target_shape - current_shape

assert np.all(extra_voxels >= 0), (
    f"The target shape {target_shape.tolist()} is smaller than the base crop "
    f"{current_shape.tolist()} in at least one dimension."
)

extra_before = extra_voxels // 2
extra_after = extra_voxels - extra_before

custom_start = crop_start - extra_before
custom_end = crop_end + extra_after

# I keep the corrected crop inside the template boundaries.
underflow = np.maximum(-custom_start, 0)
custom_start += underflow
custom_end += underflow

overflow = np.maximum(custom_end - template_shape, 0)
custom_start -= overflow
custom_end -= overflow

custom_start = np.maximum(custom_start, 0)
custom_end = np.minimum(custom_end, template_shape)

custom_shape = custom_end - custom_start

assert np.array_equal(custom_shape, target_shape), (
    f"Expected custom crop shape {target_shape.tolist()}, "
    f"but got {custom_shape.tolist()}."
)

custom_crop = {
    "candidate_name": "custom_mni_rectcrop_177x213x183",
    "description": (
        "Fixed rectangular MNI crop derived from the margin-16 MNI brain-mask "
        "bounding box. The third dimension was increased from 177 to 183 after "
        "visual QC showed that 177 slightly cropped the superior scalp/head boundary. "
        "This is not a mask and will be applied identically to all subjects."
    ),
    "x_start": int(custom_start[0]),
    "x_end": int(custom_end[0]),
    "y_start": int(custom_start[1]),
    "y_end": int(custom_end[1]),
    "z_start": int(custom_start[2]),
    "z_end": int(custom_end[2]),
    "crop_shape": [int(v) for v in custom_shape],
    "full_template_shape": [int(v) for v in template_shape],
    "voxels_per_channel": int(np.prod(custom_shape)),
    "full_voxels_per_channel": int(np.prod(template_shape)),
    "voxel_reduction_fraction": round(1 - (np.prod(custom_shape) / np.prod(template_shape)), 4),
}

CORRECT_CROP_JSON = RECT_CROP_DIR / "custom_mni_rectcrop_177x213x183.json"

with open(CORRECT_CROP_JSON, "w") as f:
    json.dump(custom_crop, f, indent=2)

print("\nSaved corrected custom fixed crop:")
print(CORRECT_CROP_JSON)

display(pd.DataFrame([custom_crop]))

## 1.41. Preview the accepted crop on real registered subject images

After selecting the `177 × 213 × 183` fixed rectangular crop on the MNI template, I need to check whether the same crop also looks safe on actual registered ADNI images. The template preview confirms that the crop is reasonable in MNI space, but real registered subjects may still show slightly different scalp extent, interpolation halos, or registration boundary effects.

Here preview the crop on representative real subjects from each diagnostic group. This is still only a QC step. I am not applying the crop to the full cohort yet and I am not saving cropped subject images in this cell.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "ants" in globals(), "ANTsPy is not imported."
assert "work_manifest" in globals(), "work_manifest is not available."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."
assert "RID_COL" in globals(), "RID_COL is not defined."
assert "IMAGE_ID_COL" in globals(), "IMAGE_ID_COL is not defined."
assert "LABEL_COL" in globals(), "LABEL_COL is not defined."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "mni_nii_path",
    "logjacobian_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

RECT_CROP_DIR = OUTPUT_REPORT.parent / "fixed_rectangular_mni_crop_planning"
CUSTOM_CROP_JSON = RECT_CROP_DIR / "custom_mni_rectcrop_177x213x183.json"

assert CUSTOM_CROP_JSON.exists(), f"Accepted crop JSON not found: {CUSTOM_CROP_JSON}"

with open(CUSTOM_CROP_JSON, "r") as f:
    custom_crop = json.load(f)

x_start = int(custom_crop["x_start"])
x_end = int(custom_crop["x_end"])
y_start = int(custom_crop["y_start"])
y_end = int(custom_crop["y_end"])
z_start = int(custom_crop["z_start"])
z_end = int(custom_crop["z_end"])

crop_shape = (
    x_end - x_start,
    y_end - y_start,
    z_end - z_start,
)

print("Loaded accepted crop:")
print(CUSTOM_CROP_JSON)
print("Crop coordinates:")
print(f"x: {x_start}:{x_end}")
print(f"y: {y_start}:{y_end}")
print(f"z: {z_start}:{z_end}")
print("Crop shape:", crop_shape)

qc_pool = work_manifest.copy()

qc_pool["mni_nii_exists"] = qc_pool["mni_nii_path"].apply(lambda p: Path(p).exists())
qc_pool["logjacobian_nii_exists"] = qc_pool["logjacobian_nii_path"].apply(lambda p: Path(p).exists())

qc_pool = qc_pool.loc[
    qc_pool["mni_nii_exists"]
    & qc_pool["logjacobian_nii_exists"]
].copy()

assert len(qc_pool) > 0, "No subjects with both MNI T1 and log-Jacobian files were found."

# I select one real registered subject per diagnostic group.
# This makes the visual check more representative than inspecting only one subject.
qc_subjects = (
    qc_pool
    .sort_values([LABEL_COL, RID_COL, IMAGE_ID_COL])
    .groupby(LABEL_COL, group_keys=False)
    .head(1)
    .copy()
)

print("\nSelected subjects for crop QC:")
display(qc_subjects[[RID_COL, IMAGE_ID_COL, LABEL_COL, "mni_nii_path", "logjacobian_nii_path"]])

n_rows = len(qc_subjects)
fig, axes = plt.subplots(n_rows, 3, figsize=(12, 4 * n_rows))

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for row_idx, (_, row) in enumerate(qc_subjects.iterrows()):
    t1_img = ants.image_read(str(row["mni_nii_path"]))
    t1_arr = t1_img.numpy().astype(np.float32)

    assert t1_arr.ndim == 3, f"Expected 3D T1 image, got shape {t1_arr.shape}"
    assert x_end <= t1_arr.shape[0] and y_end <= t1_arr.shape[1] and z_end <= t1_arr.shape[2], (
        f"Crop {crop_shape} with end coordinates {(x_end, y_end, z_end)} "
        f"does not fit image shape {t1_arr.shape}."
    )

    crop_overlay = np.zeros_like(t1_arr, dtype=bool)
    crop_overlay[x_start:x_end, y_start:y_end, z_start:z_end] = True

    # I use the center of the accepted crop rather than the center of the full image.
    # This makes the displayed slices pass through the crop itself.
    x_mid = (x_start + x_end) // 2
    y_mid = (y_start + y_end) // 2
    z_mid = (z_start + z_end) // 2

    views = [
        ("Sagittal", t1_arr[x_mid, :, :], crop_overlay[x_mid, :, :]),
        ("Coronal", t1_arr[:, y_mid, :], crop_overlay[:, y_mid, :]),
        ("Axial", t1_arr[:, :, z_mid], crop_overlay[:, :, z_mid]),
    ]

    subject_label = f"{row[LABEL_COL]} | RID {row[RID_COL]} | image {row[IMAGE_ID_COL]}"

    for col_idx, (view_name, image_slice, overlay_slice) in enumerate(views):
        ax = axes[row_idx, col_idx]

        ax.imshow(np.rot90(image_slice), cmap="gray")
        ax.imshow(np.rot90(overlay_slice), alpha=0.20)

        ax.set_title(f"{subject_label}\n{view_name} crop overlay")
        ax.axis("off")

plt.tight_layout()
plt.show()

## 1.42. Prepare a helper function for group-wise crop QC on real subjects

I already checked the accepted `177 × 213 × 183` crop on one representative subject per diagnostic group. Now I want a broader visual QC check using 10 subjects from each group. define a helper function that loads real registered MNI T1 images, overlays the fixed rectangular crop, and displays sagittal, coronal, and axial views.

This is still only a visual QC step. I am not cropping or saving the full cohort here.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "ants" in globals(), "ANTsPy is not imported."
assert "work_manifest" in globals(), "work_manifest is not available."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."
assert "RID_COL" in globals(), "RID_COL is not defined."
assert "IMAGE_ID_COL" in globals(), "IMAGE_ID_COL is not defined."
assert "LABEL_COL" in globals(), "LABEL_COL is not defined."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "mni_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

RECT_CROP_DIR = OUTPUT_REPORT.parent / "fixed_rectangular_mni_crop_planning"
CUSTOM_CROP_JSON = RECT_CROP_DIR / "custom_mni_rectcrop_177x213x183.json"

assert CUSTOM_CROP_JSON.exists(), f"Accepted crop JSON not found: {CUSTOM_CROP_JSON}"

with open(CUSTOM_CROP_JSON, "r") as f:
    custom_crop = json.load(f)

x_start = int(custom_crop["x_start"])
x_end = int(custom_crop["x_end"])
y_start = int(custom_crop["y_start"])
y_end = int(custom_crop["y_end"])
z_start = int(custom_crop["z_start"])
z_end = int(custom_crop["z_end"])

crop_shape = (
    x_end - x_start,
    y_end - y_start,
    z_end - z_start,
)

print("Loaded accepted crop:")
print(CUSTOM_CROP_JSON)
print("Crop coordinates:")
print(f"x: {x_start}:{x_end}")
print(f"y: {y_start}:{y_end}")
print(f"z: {z_start}:{z_end}")
print("Crop shape:", crop_shape)

qc_pool = work_manifest.copy()
qc_pool["mni_nii_exists"] = qc_pool["mni_nii_path"].apply(lambda p: Path(p).exists())

qc_pool = qc_pool.loc[qc_pool["mni_nii_exists"]].copy()

assert len(qc_pool) > 0, "No subjects with registered MNI T1 files were found."

# If the earlier one-subject-per-group QC table exists, I avoid repeating those images.
if "qc_subjects" in globals():
    previously_seen_image_ids = set(qc_subjects[IMAGE_ID_COL].astype(str))
else:
    previously_seen_image_ids = set()

print("Previously seen QC image IDs that will be avoided if possible:")
print(sorted(previously_seen_image_ids))


def plot_crop_overlay_for_group(group_name, n_subjects=10, random_state=42):
    group_pool = qc_pool.loc[
        qc_pool[LABEL_COL].astype(str) == str(group_name)
    ].copy()

    assert len(group_pool) > 0, f"No subjects found for group: {group_name}"

    group_pool["image_id_as_string"] = group_pool[IMAGE_ID_COL].astype(str)

    unseen_pool = group_pool.loc[
        ~group_pool["image_id_as_string"].isin(previously_seen_image_ids)
    ].copy()

    if len(unseen_pool) >= n_subjects:
        selected_subjects = unseen_pool.sample(
            n=n_subjects,
            random_state=random_state,
            replace=False,
        ).sort_values(RID_COL)
    else:
        print(
            f"Only {len(unseen_pool)} unseen subjects were available for {group_name}. "
            f"Sampling from all available subjects instead."
        )
        selected_subjects = group_pool.sample(
            n=min(n_subjects, len(group_pool)),
            random_state=random_state,
            replace=False,
        ).sort_values(RID_COL)

    print(f"\nSelected {len(selected_subjects)} subjects for {group_name} crop QC:")
    display(selected_subjects[[RID_COL, IMAGE_ID_COL, LABEL_COL, "mni_nii_path"]])

    n_rows = len(selected_subjects)
    fig, axes = plt.subplots(n_rows, 3, figsize=(12, 3.2 * n_rows))

    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, (_, row) in enumerate(selected_subjects.iterrows()):
        t1_img = ants.image_read(str(row["mni_nii_path"]))
        t1_arr = t1_img.numpy().astype(np.float32)

        assert t1_arr.ndim == 3, f"Expected 3D T1 image, got shape {t1_arr.shape}"
        assert x_end <= t1_arr.shape[0] and y_end <= t1_arr.shape[1] and z_end <= t1_arr.shape[2], (
            f"Crop end coordinates {(x_end, y_end, z_end)} do not fit image shape {t1_arr.shape}."
        )

        crop_overlay = np.zeros_like(t1_arr, dtype=bool)
        crop_overlay[x_start:x_end, y_start:y_end, z_start:z_end] = True

        x_mid = (x_start + x_end) // 2
        y_mid = (y_start + y_end) // 2
        z_mid = (z_start + z_end) // 2

        views = [
            ("Sagittal", t1_arr[x_mid, :, :], crop_overlay[x_mid, :, :]),
            ("Coronal", t1_arr[:, y_mid, :], crop_overlay[:, y_mid, :]),
            ("Axial", t1_arr[:, :, z_mid], crop_overlay[:, :, z_mid]),
        ]

        subject_label = f"{row[LABEL_COL]} | RID {row[RID_COL]} | image {row[IMAGE_ID_COL]}"

        for col_idx, (view_name, image_slice, overlay_slice) in enumerate(views):
            ax = axes[row_idx, col_idx]

            ax.imshow(np.rot90(image_slice), cmap="gray")
            ax.imshow(np.rot90(overlay_slice), alpha=0.20)

            ax.set_title(f"{subject_label}\n{view_name} crop overlay")
            ax.axis("off")

    plt.tight_layout()
    plt.show()

    return selected_subjects

## 1.43. Check the accepted crop on 10 AD subjects

visually inspect the fixed rectangular crop on 10 AD subjects. This helps confirm that the crop does not cut the superior scalp/head boundary or remove relevant anatomy in the AD group.

In [ ]:
ad_crop_qc_subjects = plot_crop_overlay_for_group(
    group_name="AD",
    n_subjects=10,
    random_state=101,
)

## 1.44. Check the accepted crop on 10 CN subjects

visually inspect the fixed rectangular crop on 10 CN subjects. This helps confirm that the crop is not only safe for disease cases but also works for cognitively normal controls.

In [ ]:
cn_crop_qc_subjects = plot_crop_overlay_for_group(
    group_name="CN",
    n_subjects=10,
    random_state=102,
)

## 1.45. Check the accepted crop on 10 pMCI subjects

visually inspect the fixed rectangular crop on 10 pMCI subjects. Since pMCI is one of the main prognosis groups, this check helps confirm that the crop remains safe for the progressive MCI subgroup.

In [ ]:
pmci_crop_qc_subjects = plot_crop_overlay_for_group(
    group_name="pMCI",
    n_subjects=10,
    random_state=103,
)

## 1.46. Check the accepted crop on 10 sMCI subjects

visually inspect the fixed rectangular crop on 10 sMCI subjects. Since sMCI is the comparison prognosis group for pMCI, this check helps confirm that the crop is equally safe for the stable MCI subgroup.

In [ ]:
smci_crop_qc_subjects = plot_crop_overlay_for_group(
    group_name="sMCI",
    n_subjects=10,
    random_state=104,
)

## 1.47. Programmatically check whether the accepted crop may clip real subject signal

The earlier data-driven foreground crop attempt was too sensitive to low-level boundary signal, so I should not use its bounding boxes as anatomical truth. However, before applying the accepted `177 × 213 × 183` crop to the full cohort, I still want a cohort-wide safety check.

Here run a new clipping-risk QC on the original full MNI T1 images. For each subject, I estimate a more robust high-signal anatomical mask using a subject-specific percentile threshold from positive T1 intensities. I then count how much of this robust signal lies outside the accepted fixed rectangular crop. This does not apply any mask to the model inputs; it is only a QC check to identify whether the crop may remove meaningful subject signal.

The output will be a CSV report and group-level summaries. If any subjects look suspicious, visually inspect those cases before cropping the full cohort.

## 1.48. Apply the accepted fixed rectangular crop to the full cohort

The accepted `177 × 213 × 183` crop has now been checked on the MNI template and on multiple real registered ADNI subjects from each diagnostic group. The visual QC showed that the crop preserves the head/scalp boundary while removing unused outer MNI padding.

apply this same fixed rectangular crop to every subject and to both MRI channels: the registered T1 image and the log-Jacobian map. This is not skull stripping, not a brain mask, and not subject-specific cropping. The original full MNI images will remain unchanged. This cell only creates cropped copies for the next model-preparation stage.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

assert "ants" in globals(), "ANTsPy is not imported."
assert "work_manifest" in globals(), "work_manifest is not available."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."
assert "RID_COL" in globals(), "RID_COL is not defined."
assert "IMAGE_ID_COL" in globals(), "IMAGE_ID_COL is not defined."
assert "LABEL_COL" in globals(), "LABEL_COL is not defined."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "mni_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

RECT_CROP_DIR = OUTPUT_REPORT.parent / "fixed_rectangular_mni_crop_planning"
CUSTOM_CROP_JSON = RECT_CROP_DIR / "custom_mni_rectcrop_177x213x183.json"

assert CUSTOM_CROP_JSON.exists(), f"Accepted crop JSON not found: {CUSTOM_CROP_JSON}"

with open(CUSTOM_CROP_JSON, "r") as f:
    custom_crop = json.load(f)

x_start = int(custom_crop["x_start"])
x_end = int(custom_crop["x_end"])
y_start = int(custom_crop["y_start"])
y_end = int(custom_crop["y_end"])
z_start = int(custom_crop["z_start"])
z_end = int(custom_crop["z_end"])

expected_crop_shape = (
    x_end - x_start,
    y_end - y_start,
    z_end - z_start,
)

assert expected_crop_shape == (177, 213, 183), (
    f"Expected crop shape (177, 213, 183), but got {expected_crop_shape}."
)

print("Using accepted crop:")
print(CUSTOM_CROP_JSON)
print(f"x: {x_start}:{x_end}")
print(f"y: {y_start}:{y_end}")
print(f"z: {z_start}:{z_end}")
print("shape:", expected_crop_shape)

CROP_QC_DIR = OUTPUT_REPORT.parent / "fixed_rectangular_mni_crop_planning"
CROP_QC_DIR.mkdir(parents=True, exist_ok=True)

CROP_CLIP_QC_REPORT = CROP_QC_DIR / "custom_mni_rectcrop_177x213x183_full_cohort_clipping_qc.csv"
CROP_CLIP_QC_SUMMARY = CROP_QC_DIR / "custom_mni_rectcrop_177x213x183_full_cohort_clipping_qc_by_group.csv"

qc_pool = work_manifest.copy()
qc_pool["mni_nii_exists"] = qc_pool["mni_nii_path"].apply(lambda p: Path(p).exists())
qc_pool = qc_pool.loc[qc_pool["mni_nii_exists"]].copy()

assert len(qc_pool) == len(work_manifest), (
    f"Expected all {len(work_manifest)} subjects to have MNI T1 files, "
    f"but found {len(qc_pool)}."
)

# I use a robust percentile threshold, not the very low threshold from the failed crop-box attempt.
# p30 is deliberately more conservative than using only the brightest tissue, but avoids tiny edge halos.
ROBUST_SIGNAL_PERCENTILE = 30.0

# Small isolated noise outside the crop is not automatically treated as anatomical clipping.
# These thresholds are only QC flags to identify cases that deserve visual inspection.
OUTSIDE_VOXEL_WARNING_COUNT = 500
OUTSIDE_SIGNAL_FRACTION_WARNING = 0.001

qc_rows = []

start_time = time.time()

for row_number, (_, row) in enumerate(qc_pool.iterrows(), start=1):
    t1_path = Path(row["mni_nii_path"])
    t1_img = ants.image_read(str(t1_path))
    t1_arr = t1_img.numpy().astype(np.float32)

    assert t1_arr.ndim == 3, f"Expected 3D T1 image, got shape {t1_arr.shape} for {t1_path}"
    assert x_end <= t1_arr.shape[0] and y_end <= t1_arr.shape[1] and z_end <= t1_arr.shape[2], (
        f"Crop end coordinates {(x_end, y_end, z_end)} do not fit image shape {t1_arr.shape}."
    )

    finite_arr = t1_arr[np.isfinite(t1_arr)]
    positive_values = finite_arr[finite_arr > 0]

    if positive_values.size == 0:
        threshold = np.nan
        robust_mask = np.zeros_like(t1_arr, dtype=bool)
    else:
        threshold = float(np.percentile(positive_values, ROBUST_SIGNAL_PERCENTILE))
        robust_mask = t1_arr > threshold

    total_robust_voxels = int(robust_mask.sum())

    inside_mask = np.zeros_like(robust_mask, dtype=bool)
    inside_mask[x_start:x_end, y_start:y_end, z_start:z_end] = True

    outside_robust_voxels = int((robust_mask & ~inside_mask).sum())
    outside_robust_fraction = (
        outside_robust_voxels / total_robust_voxels
        if total_robust_voxels > 0
        else 0.0
    )

    # I also count which side the outside robust signal is on.
    # These are not all mutually exclusive at corners, but they are useful for QC.
    x_low_outside = int(robust_mask[:x_start, :, :].sum())
    x_high_outside = int(robust_mask[x_end:, :, :].sum())
    y_low_outside = int(robust_mask[:, :y_start, :].sum())
    y_high_outside = int(robust_mask[:, y_end:, :].sum())
    z_low_outside = int(robust_mask[:, :, :z_start].sum())
    z_high_outside = int(robust_mask[:, :, z_end:].sum())

    potential_clip_flag = (
        outside_robust_voxels > OUTSIDE_VOXEL_WARNING_COUNT
        and outside_robust_fraction > OUTSIDE_SIGNAL_FRACTION_WARNING
    )

    qc_rows.append({
        RID_COL: row[RID_COL],
        IMAGE_ID_COL: row[IMAGE_ID_COL],
        LABEL_COL: row[LABEL_COL],
        "mni_nii_path": str(t1_path),
        "image_shape": str(tuple(t1_arr.shape)),
        "crop_shape": str(expected_crop_shape),
        "robust_signal_percentile": ROBUST_SIGNAL_PERCENTILE,
        "robust_signal_threshold": threshold,
        "total_robust_voxels": total_robust_voxels,
        "outside_robust_voxels": outside_robust_voxels,
        "outside_robust_fraction": outside_robust_fraction,
        "x_low_outside_robust_voxels": x_low_outside,
        "x_high_outside_robust_voxels": x_high_outside,
        "y_low_outside_robust_voxels": y_low_outside,
        "y_high_outside_robust_voxels": y_high_outside,
        "z_low_outside_robust_voxels": z_low_outside,
        "z_high_outside_robust_voxels": z_high_outside,
        "potential_clip_flag": potential_clip_flag,
    })

    if row_number % 50 == 0 or row_number == len(qc_pool):
        elapsed_minutes = (time.time() - start_time) / 60
        print(
            f"Checked {row_number}/{len(qc_pool)} subjects "
            f"for crop clipping risk in {elapsed_minutes:.1f} minutes."
        )

crop_clipping_qc = pd.DataFrame(qc_rows)

crop_clipping_qc.to_csv(CROP_CLIP_QC_REPORT, index=False)

group_summary = (
    crop_clipping_qc
    .groupby(LABEL_COL)
    .agg(
        n_subjects=(RID_COL, "count"),
        flagged_subjects=("potential_clip_flag", "sum"),
        max_outside_robust_voxels=("outside_robust_voxels", "max"),
        median_outside_robust_voxels=("outside_robust_voxels", "median"),
        max_outside_robust_fraction=("outside_robust_fraction", "max"),
        median_outside_robust_fraction=("outside_robust_fraction", "median"),
    )
    .reset_index()
)

group_summary.to_csv(CROP_CLIP_QC_SUMMARY, index=False)

print("\nSaved full-cohort crop clipping QC report:")
print(CROP_CLIP_QC_REPORT)

print("\nSaved group-level crop clipping QC summary:")
print(CROP_CLIP_QC_SUMMARY)

print("\nGroup-level summary:")
display(group_summary)

print("\nPotential clipping flags:")
display(crop_clipping_qc["potential_clip_flag"].value_counts().to_frame("count"))

print("\nTop 20 subjects by outside robust signal:")
display(
    crop_clipping_qc
    .sort_values("outside_robust_voxels", ascending=False)
    [
        [
            RID_COL,
            IMAGE_ID_COL,
            LABEL_COL,
            "outside_robust_voxels",
            "outside_robust_fraction",
            "x_low_outside_robust_voxels",
            "x_high_outside_robust_voxels",
            "y_low_outside_robust_voxels",
            "y_high_outside_robust_voxels",
            "z_low_outside_robust_voxels",
            "z_high_outside_robust_voxels",
            "potential_clip_flag",
            "mni_nii_path",
        ]
    ]
    .head(20)
)

## 1.49. Visually inspect the 20 highest-risk crop QC cases

The full-cohort clipping-risk QC flagged many subjects, so inspect the 20 cases with the largest amount of robust T1 signal outside the accepted crop. These are the most conservative/highest-risk cases according to the automated QC. If these cases still look visually safe, then the crop is likely acceptable and the automated flag is probably over-sensitive to skull/scalp signal, interpolation halos, or non-anatomical boundary signal.

In [ ]:
N_HIGHEST_RISK_CASES_TO_VIEW = 20

top_20_highest_risk_cases = (
    crop_clipping_qc
    .sort_values("outside_robust_voxels", ascending=False)
    .head(N_HIGHEST_RISK_CASES_TO_VIEW)
    .copy()
)

print(f"Inspecting the top {len(top_20_highest_risk_cases)} highest-risk cases by outside robust voxels.")

display(
    top_20_highest_risk_cases[
        [
            RID_COL,
            IMAGE_ID_COL,
            LABEL_COL,
            "outside_robust_voxels",
            "outside_robust_fraction",
            "x_low_outside_robust_voxels",
            "x_high_outside_robust_voxels",
            "y_low_outside_robust_voxels",
            "y_high_outside_robust_voxels",
            "z_low_outside_robust_voxels",
            "z_high_outside_robust_voxels",
            "potential_clip_flag",
            "mni_nii_path",
        ]
    ]
)

plot_crop_overlay_for_qc_rows(
    qc_rows=top_20_highest_risk_cases,
    title_prefix="Top 20 highest-risk crop QC case",
)

## 1.50. Visual review of the highest-risk crop QC cases

The full-cohort clipping-risk QC flagged many subjects, which suggested that the automated robust-signal criterion was overly conservative rather than necessarily indicating true anatomical clipping. To evaluate this, I visually inspected the 20 subjects with the largest amount of robust T1 signal outside the accepted crop.

These 20 cases were treated as the highest-risk examples because they had the largest `outside_robust_voxels` values in the QC report. For each case, I overlaid the accepted fixed rectangular crop on the original full MNI-registered T1 image in sagittal, coronal, and axial views.

The visual inspection showed that the relevant head/brain anatomy remained inside the crop boundaries in these highest-risk cases. The signal counted outside the crop appeared to correspond mainly to peripheral skull/scalp, interpolation halo, or non-anatomical boundary signal rather than visibly clipped brain anatomy. Therefore, I accepted the `177 × 213 × 183` fixed rectangular crop for full-cohort preprocessing.

This crop remains a shared rectangular MNI-space crop. It is not skull stripping, not a brain-shaped mask, and not subject-specific cropping. The same coordinates will be applied identically to every subject and to both MRI channels: registered T1 and log-Jacobian.

## 1.51. Apply the accepted fixed rectangular crop to the full cohort

The accepted `177 × 213 × 183` crop has now been checked on the MNI template and on multiple real registered ADNI subjects from each diagnostic group. The visual QC showed that the crop preserves the head/scalp boundary while removing unused outer MNI padding.

apply this same fixed rectangular crop to every subject and to both MRI channels: the registered T1 image and the log-Jacobian map. This is not skull stripping, not a brain mask, and not subject-specific cropping. The original full MNI images will remain unchanged. This cell only creates cropped copies for the next model-preparation stage.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

assert "ants" in globals(), "ANTsPy is not imported."
assert "work_manifest" in globals(), "work_manifest is not available."
assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "OUTPUT_REPORT" in globals(), "OUTPUT_REPORT is not defined."
assert "RID_COL" in globals(), "RID_COL is not defined."
assert "IMAGE_ID_COL" in globals(), "IMAGE_ID_COL is not defined."
assert "LABEL_COL" in globals(), "LABEL_COL is not defined."

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "mni_nii_path",
    "logjacobian_nii_path",
]

missing_cols = [col for col in required_cols if col not in work_manifest.columns]
assert not missing_cols, f"Missing required columns in work_manifest: {missing_cols}"

RECT_CROP_DIR = OUTPUT_REPORT.parent / "fixed_rectangular_mni_crop_planning"
CUSTOM_CROP_JSON = RECT_CROP_DIR / "custom_mni_rectcrop_177x213x183.json"

assert CUSTOM_CROP_JSON.exists(), f"Accepted crop JSON not found: {CUSTOM_CROP_JSON}"

with open(CUSTOM_CROP_JSON, "r") as f:
    custom_crop = json.load(f)

crop_start = np.array([
    int(custom_crop["x_start"]),
    int(custom_crop["y_start"]),
    int(custom_crop["z_start"]),
], dtype=int)

crop_end = np.array([
    int(custom_crop["x_end"]),
    int(custom_crop["y_end"]),
    int(custom_crop["z_end"]),
], dtype=int)

expected_crop_shape = tuple((crop_end - crop_start).tolist())

assert expected_crop_shape == (177, 213, 183), (
    f"Expected crop shape (177, 213, 183), but got {expected_crop_shape}."
)

print("Using accepted fixed rectangular crop:")
print(CUSTOM_CROP_JSON)
print("crop start:", crop_start.tolist())
print("crop end:  ", crop_end.tolist())
print("crop shape:", expected_crop_shape)

CROP_BRANCH_NAME = "mni_n4_syn_rectcrop_177x213x183"

CROPPED_ROOT = PROJECT_DIR / "processed" / CROP_BRANCH_NAME
CROPPED_T1_NII_DIR = CROPPED_ROOT / "nii_t1"
CROPPED_LOGJAC_NII_DIR = CROPPED_ROOT / "nii_logjacobian"
CROPPED_T1_NPY_DIR = CROPPED_ROOT / "npy_t1"
CROPPED_LOGJAC_NPY_DIR = CROPPED_ROOT / "npy_logjacobian"

for folder in [
    CROPPED_ROOT,
    CROPPED_T1_NII_DIR,
    CROPPED_LOGJAC_NII_DIR,
    CROPPED_T1_NPY_DIR,
    CROPPED_LOGJAC_NPY_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

CROPPED_QC_DIR = OUTPUT_REPORT.parent / CROP_BRANCH_NAME
CROPPED_QC_DIR.mkdir(parents=True, exist_ok=True)

CROPPED_MANIFEST_PATH = (
    PROJECT_DIR
    / "manifest"
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_manifest_1063.csv"
)

CROPPED_REPORT_PATH = (
    CROPPED_QC_DIR
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_report.csv"
)

crop_pool = work_manifest.copy()

crop_pool["mni_nii_exists"] = crop_pool["mni_nii_path"].apply(lambda p: Path(p).exists())
crop_pool["logjacobian_nii_exists"] = crop_pool["logjacobian_nii_path"].apply(lambda p: Path(p).exists())

crop_pool = crop_pool.loc[
    crop_pool["mni_nii_exists"]
    & crop_pool["logjacobian_nii_exists"]
].copy()

assert len(crop_pool) == len(work_manifest), (
    f"Expected all {len(work_manifest)} subjects to have both MNI T1 and log-Jacobian files, "
    f"but found {len(crop_pool)} complete subjects."
)

if "output_stem" not in crop_pool.columns:
    crop_pool["output_stem"] = crop_pool.apply(
        lambda row: f"RID-{int(row[RID_COL])}_image-{int(row[IMAGE_ID_COL])}",
        axis=1,
    )

FORCE_RERUN_CROP = False


def crop_ants_image_with_shape_check(image, start_indices, end_indices, expected_shape):
    """
    I crop an ANTs image using index coordinates and verify the output shape.

    ANTsPy versions can differ in whether the upper crop index behaves as
    inclusive or exclusive, so I try the inclusive interpretation first and
    then the exclusive interpretation if needed.
    """
    start_indices = np.array(start_indices, dtype=int)
    end_indices = np.array(end_indices, dtype=int)

    upper_index_candidates = [
        end_indices - 1,
        end_indices,
    ]

    crop_errors = []

    for upper_indices in upper_index_candidates:
        try:
            cropped_image = ants.crop_indices(
                image,
                start_indices.tolist(),
                upper_indices.tolist(),
            )

            if tuple(cropped_image.shape) == tuple(expected_shape):
                return cropped_image

            crop_errors.append(
                f"upper={upper_indices.tolist()} gave shape {tuple(cropped_image.shape)}"
            )

        except Exception as exc:
            crop_errors.append(
                f"upper={upper_indices.tolist()} failed with error: {repr(exc)}"
            )

    raise RuntimeError(
        "Could not crop image to the expected shape. "
        f"Expected {expected_shape}. Attempts: {crop_errors}"
    )


report_rows = []

start_time = time.time()

for row_number, (_, row) in enumerate(crop_pool.iterrows(), start=1):
    subject_stem = str(row["output_stem"])

    input_t1_path = Path(row["mni_nii_path"])
    input_logjacobian_path = Path(row["logjacobian_nii_path"])

    output_t1_nii_path = (
        CROPPED_T1_NII_DIR
        / f"{subject_stem}_space-MNI152NLin2009cSym_desc-n4Syn_rectcrop-177x213x183_T1w.nii.gz"
    )

    output_logjacobian_nii_path = (
        CROPPED_LOGJAC_NII_DIR
        / f"{subject_stem}_space-MNI152NLin2009cSym_desc-logJacobian_rectcrop-177x213x183.nii.gz"
    )

    output_t1_npy_path = (
        CROPPED_T1_NPY_DIR
        / f"{subject_stem}_space-MNI152NLin2009cSym_desc-n4Syn_rectcrop-177x213x183_T1w.npy"
    )

    output_logjacobian_npy_path = (
        CROPPED_LOGJAC_NPY_DIR
        / f"{subject_stem}_space-MNI152NLin2009cSym_desc-logJacobian_rectcrop-177x213x183.npy"
    )

    outputs_exist = (
    output_t1_nii_path.exists()
    and output_logjacobian_nii_path.exists()
    and output_t1_npy_path.exists()
    and output_logjacobian_npy_path.exists()
    )

    existing_outputs_valid = False

    if outputs_exist and not FORCE_RERUN_CROP:
        try:
            existing_t1_shape = tuple(np.load(output_t1_npy_path, mmap_mode="r").shape)
            existing_logjacobian_shape = tuple(np.load(output_logjacobian_npy_path, mmap_mode="r").shape)

            existing_outputs_valid = (
                existing_t1_shape == expected_crop_shape
                and existing_logjacobian_shape == expected_crop_shape
            )

            if not existing_outputs_valid:
                print(
                    f"Existing outputs for {subject_stem} have invalid shapes: "
                    f"T1 {existing_t1_shape}, log-Jacobian {existing_logjacobian_shape}. "
                    "They will be recreated."
                )

        except Exception as exc:
            print(
                f"Could not validate existing outputs for {subject_stem}: {repr(exc)}. "
                "They will be recreated."
            )
            existing_outputs_valid = False

    if outputs_exist and existing_outputs_valid and not FORCE_RERUN_CROP:
        status = "skipped_existing"

        t1_crop_shape = expected_crop_shape
        logjacobian_crop_shape = expected_crop_shape

    else:
        t1_img = ants.image_read(str(input_t1_path))
        logjacobian_img = ants.image_read(str(input_logjacobian_path))

        assert tuple(t1_img.shape) == tuple(logjacobian_img.shape), (
            f"T1 and log-Jacobian shapes do not match for {subject_stem}: "
            f"{tuple(t1_img.shape)} vs {tuple(logjacobian_img.shape)}"
        )

        assert np.all(crop_end <= np.array(t1_img.shape)), (
            f"Crop end {crop_end.tolist()} does not fit image shape {tuple(t1_img.shape)} "
            f"for {subject_stem}."
        )

        t1_cropped_img = crop_ants_image_with_shape_check(
            t1_img,
            crop_start,
            crop_end,
            expected_crop_shape,
        )

        logjacobian_cropped_img = crop_ants_image_with_shape_check(
            logjacobian_img,
            crop_start,
            crop_end,
            expected_crop_shape,
        )

        t1_crop_arr = t1_cropped_img.numpy().astype(np.float32)
        logjacobian_crop_arr = logjacobian_cropped_img.numpy().astype(np.float32)

        assert tuple(t1_crop_arr.shape) == expected_crop_shape, (
            f"T1 crop shape mismatch for {subject_stem}: {t1_crop_arr.shape}"
        )

        assert tuple(logjacobian_crop_arr.shape) == expected_crop_shape, (
            f"log-Jacobian crop shape mismatch for {subject_stem}: {logjacobian_crop_arr.shape}"
        )

        assert np.isfinite(t1_crop_arr).all(), f"Non-finite values found in cropped T1 for {subject_stem}."
        assert np.isfinite(logjacobian_crop_arr).all(), (
            f"Non-finite values found in cropped log-Jacobian for {subject_stem}."
        )

        ants.image_write(t1_cropped_img, str(output_t1_nii_path))
        ants.image_write(logjacobian_cropped_img, str(output_logjacobian_nii_path))

        np.save(output_t1_npy_path, t1_crop_arr)
        np.save(output_logjacobian_npy_path, logjacobian_crop_arr)

        status = "created"

        t1_crop_shape = tuple(t1_crop_arr.shape)
        logjacobian_crop_shape = tuple(logjacobian_crop_arr.shape)

    report_rows.append({
        RID_COL: row[RID_COL],
        IMAGE_ID_COL: row[IMAGE_ID_COL],
        LABEL_COL: row[LABEL_COL],
        "output_stem": subject_stem,
        "input_mni_nii_path": str(input_t1_path),
        "input_logjacobian_nii_path": str(input_logjacobian_path),
        "cropped_t1_nii_path": str(output_t1_nii_path),
        "cropped_logjacobian_nii_path": str(output_logjacobian_nii_path),
        "cropped_t1_npy_path": str(output_t1_npy_path),
        "cropped_logjacobian_npy_path": str(output_logjacobian_npy_path),
        "crop_x_start": int(crop_start[0]),
        "crop_x_end": int(crop_end[0]),
        "crop_y_start": int(crop_start[1]),
        "crop_y_end": int(crop_end[1]),
        "crop_z_start": int(crop_start[2]),
        "crop_z_end": int(crop_end[2]),
        "crop_shape": str(expected_crop_shape),
        "t1_crop_shape": str(t1_crop_shape),
        "logjacobian_crop_shape": str(logjacobian_crop_shape),
        "crop_status": status,
    })

    if row_number % 25 == 0 or row_number == len(crop_pool):
        elapsed_minutes = (time.time() - start_time) / 60
        print(
            f"Processed {row_number}/{len(crop_pool)} subjects "
            f"for fixed rectangular cropping in {elapsed_minutes:.1f} minutes."
        )

crop_report = pd.DataFrame(report_rows)

crop_report.to_csv(CROPPED_REPORT_PATH, index=False)

cropped_manifest = work_manifest.merge(
    crop_report[
        [
            RID_COL,
            IMAGE_ID_COL,
            "cropped_t1_nii_path",
            "cropped_logjacobian_nii_path",
            "cropped_t1_npy_path",
            "cropped_logjacobian_npy_path",
            "crop_x_start",
            "crop_x_end",
            "crop_y_start",
            "crop_y_end",
            "crop_z_start",
            "crop_z_end",
            "crop_shape",
            "crop_status",
        ]
    ],
    on=[RID_COL, IMAGE_ID_COL],
    how="left",
)

cropped_manifest["cropped_t1_nii_exists"] = cropped_manifest["cropped_t1_nii_path"].apply(
    lambda p: Path(p).exists()
)

cropped_manifest["cropped_logjacobian_nii_exists"] = cropped_manifest["cropped_logjacobian_nii_path"].apply(
    lambda p: Path(p).exists()
)

cropped_manifest["cropped_t1_npy_exists"] = cropped_manifest["cropped_t1_npy_path"].apply(
    lambda p: Path(p).exists()
)

cropped_manifest["cropped_logjacobian_npy_exists"] = cropped_manifest["cropped_logjacobian_npy_path"].apply(
    lambda p: Path(p).exists()
)

cropped_manifest.to_csv(CROPPED_MANIFEST_PATH, index=False)

print("\nSaved fixed rectangular crop report:")
print(CROPPED_REPORT_PATH)

print("\nSaved fixed rectangular cropped manifest:")
print(CROPPED_MANIFEST_PATH)

print("\nCrop status counts:")
display(crop_report["crop_status"].value_counts().to_frame("count"))

print("\nOutput existence checks:")
display(
    cropped_manifest[
        [
            "cropped_t1_nii_exists",
            "cropped_logjacobian_nii_exists",
            "cropped_t1_npy_exists",
            "cropped_logjacobian_npy_exists",
        ]
    ].sum().to_frame("existing_count")
)

print("\nCropped cohort counts:")
display(cropped_manifest[LABEL_COL].value_counts().to_frame("count"))

## 1.52. Final disk-based QC of the cropped MRI branch

The fixed rectangular crop has now been applied to all 1063 subjects. The processing report shows that all cropped T1 and log-Jacobian NIfTI and NPY files were created successfully.

Here perform a final disk-based QC by reopening the saved cropped NPY files from disk and checking that every subject has the expected shape, finite values, and available output paths. This verifies the saved crop branch itself, rather than relying only on variables kept in memory during the cropping run.

In [ ]:
from pathlib import Path
import ast
import time

import numpy as np
import pandas as pd

assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "LABEL_COL" in globals(), "LABEL_COL is not defined."
assert "RID_COL" in globals(), "RID_COL is not defined."
assert "IMAGE_ID_COL" in globals(), "IMAGE_ID_COL is not defined."

CROPPED_MANIFEST_PATH = (
    PROJECT_DIR
    / "manifest"
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_manifest_1063.csv"
)

assert CROPPED_MANIFEST_PATH.exists(), f"Cropped manifest not found: {CROPPED_MANIFEST_PATH}"

cropped_manifest = pd.read_csv(CROPPED_MANIFEST_PATH)

expected_shape = (177, 213, 183)

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "cropped_t1_nii_path",
    "cropped_logjacobian_nii_path",
    "cropped_t1_npy_path",
    "cropped_logjacobian_npy_path",
]

missing_cols = [col for col in required_cols if col not in cropped_manifest.columns]
assert not missing_cols, f"Missing required columns in cropped manifest: {missing_cols}"

qc_rows = []

start_time = time.time()

for row_number, (_, row) in enumerate(cropped_manifest.iterrows(), start=1):
    t1_nii_path = Path(row["cropped_t1_nii_path"])
    logjac_nii_path = Path(row["cropped_logjacobian_nii_path"])
    t1_npy_path = Path(row["cropped_t1_npy_path"])
    logjac_npy_path = Path(row["cropped_logjacobian_npy_path"])

    t1_nii_exists = t1_nii_path.exists()
    logjac_nii_exists = logjac_nii_path.exists()
    t1_npy_exists = t1_npy_path.exists()
    logjac_npy_exists = logjac_npy_path.exists()

    t1_npy_shape = None
    logjac_npy_shape = None
    t1_all_finite = False
    logjac_all_finite = False
    t1_min = np.nan
    t1_max = np.nan
    logjac_min = np.nan
    logjac_max = np.nan

    if t1_npy_exists:
        t1_arr = np.load(t1_npy_path, mmap_mode="r")
        t1_npy_shape = tuple(t1_arr.shape)
        t1_all_finite = bool(np.isfinite(t1_arr).all())
        t1_min = float(np.nanmin(t1_arr))
        t1_max = float(np.nanmax(t1_arr))

    if logjac_npy_exists:
        logjac_arr = np.load(logjac_npy_path, mmap_mode="r")
        logjac_npy_shape = tuple(logjac_arr.shape)
        logjac_all_finite = bool(np.isfinite(logjac_arr).all())
        logjac_min = float(np.nanmin(logjac_arr))
        logjac_max = float(np.nanmax(logjac_arr))

    t1_shape_ok = t1_npy_shape == expected_shape
    logjac_shape_ok = logjac_npy_shape == expected_shape

    qc_pass = (
        t1_nii_exists
        and logjac_nii_exists
        and t1_npy_exists
        and logjac_npy_exists
        and t1_shape_ok
        and logjac_shape_ok
        and t1_all_finite
        and logjac_all_finite
    )

    qc_rows.append({
        RID_COL: row[RID_COL],
        IMAGE_ID_COL: row[IMAGE_ID_COL],
        LABEL_COL: row[LABEL_COL],
        "cropped_t1_nii_exists": t1_nii_exists,
        "cropped_logjacobian_nii_exists": logjac_nii_exists,
        "cropped_t1_npy_exists": t1_npy_exists,
        "cropped_logjacobian_npy_exists": logjac_npy_exists,
        "t1_npy_shape": str(t1_npy_shape),
        "logjacobian_npy_shape": str(logjac_npy_shape),
        "t1_shape_ok": t1_shape_ok,
        "logjacobian_shape_ok": logjac_shape_ok,
        "t1_all_finite": t1_all_finite,
        "logjacobian_all_finite": logjac_all_finite,
        "t1_min": t1_min,
        "t1_max": t1_max,
        "logjacobian_min": logjac_min,
        "logjacobian_max": logjac_max,
        "cropped_branch_qc_pass": qc_pass,
    })

    if row_number % 50 == 0 or row_number == len(cropped_manifest):
        elapsed_minutes = (time.time() - start_time) / 60
        print(
            f"Checked {row_number}/{len(cropped_manifest)} cropped subjects "
            f"in {elapsed_minutes:.1f} minutes."
        )

cropped_qc_report = pd.DataFrame(qc_rows)

CROPPED_QC_REPORT_PATH = (
    PROJECT_DIR
    / "qc"
    / "clean_90d_mni_n4_syn_jacobian"
    / "mni_n4_syn_rectcrop_177x213x183"
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_disk_qc_report.csv"
)

CROPPED_QC_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
cropped_qc_report.to_csv(CROPPED_QC_REPORT_PATH, index=False)

print("\nSaved cropped disk QC report:")
print(CROPPED_QC_REPORT_PATH)

print("\nQC pass counts:")
display(cropped_qc_report["cropped_branch_qc_pass"].value_counts().to_frame("count"))

print("\nShape checks:")
display(
    cropped_qc_report[
        [
            "t1_shape_ok",
            "logjacobian_shape_ok",
            "t1_all_finite",
            "logjacobian_all_finite",
        ]
    ].sum().to_frame("passing_count")
)

print("\nCohort counts:")
display(cropped_qc_report[LABEL_COL].value_counts().to_frame("count"))

print("\nIntensity range summary:")
display(
    cropped_qc_report[
        ["t1_min", "t1_max", "logjacobian_min", "logjacobian_max"]
    ].describe()
)

## 1.53. Inspect one cropped log-Jacobian map directly as numerical data

A log-Jacobian map is not an MRI intensity image. It is a 3D numerical array where each voxel represents local deformation after nonlinear registration. Values near `0` mean little or no local volume change, positive values indicate local expansion, and negative values indicate local contraction.

Here load one saved cropped log-Jacobian `.npy` file directly from disk, inspect its shape and value distribution, and print a small patch of raw voxel values. This helps me see the map as numerical deformation data rather than only as a colored visualization.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "RID_COL" in globals(), "RID_COL is not defined."
assert "IMAGE_ID_COL" in globals(), "IMAGE_ID_COL is not defined."
assert "LABEL_COL" in globals(), "LABEL_COL is not defined."

CROPPED_MANIFEST_PATH = (
    PROJECT_DIR
    / "manifest"
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_manifest_1063.csv"
)

assert CROPPED_MANIFEST_PATH.exists(), f"Cropped manifest not found: {CROPPED_MANIFEST_PATH}"

cropped_manifest = pd.read_csv(CROPPED_MANIFEST_PATH)

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "cropped_logjacobian_npy_path",
]

missing_cols = [col for col in required_cols if col not in cropped_manifest.columns]
assert not missing_cols, f"Missing required columns: {missing_cols}"

# I choose one example subject.
# You can change this to another group such as "CN", "AD", "pMCI", or "sMCI".
EXAMPLE_GROUP = "pMCI"

example_row = (
    cropped_manifest
    .loc[cropped_manifest[LABEL_COL] == EXAMPLE_GROUP]
    .sort_values([RID_COL, IMAGE_ID_COL])
    .iloc[0]
)

logjac_path = Path(example_row["cropped_logjacobian_npy_path"])

assert logjac_path.exists(), f"Log-Jacobian file not found: {logjac_path}"

logjac_arr = np.load(logjac_path).astype(np.float32)

print("Selected subject:")
print(f"Group: {example_row[LABEL_COL]}")
print(f"RID: {example_row[RID_COL]}")
print(f"Image ID: {example_row[IMAGE_ID_COL]}")
print(f"Path: {logjac_path}")

print("\nArray information:")
print("Shape:", logjac_arr.shape)
print("Dtype:", logjac_arr.dtype)
print("Finite values:", np.isfinite(logjac_arr).all())

print("\nValue summary:")
print(f"Minimum: {np.nanmin(logjac_arr):.6f}")
print(f"1st percentile: {np.nanpercentile(logjac_arr, 1):.6f}")
print(f"5th percentile: {np.nanpercentile(logjac_arr, 5):.6f}")
print(f"Median: {np.nanmedian(logjac_arr):.6f}")
print(f"95th percentile: {np.nanpercentile(logjac_arr, 95):.6f}")
print(f"99th percentile: {np.nanpercentile(logjac_arr, 99):.6f}")
print(f"Maximum: {np.nanmax(logjac_arr):.6f}")
print(f"Mean: {np.nanmean(logjac_arr):.6f}")
print(f"Standard deviation: {np.nanstd(logjac_arr):.6f}")

# I inspect the central voxel and a small local patch around it.
x_mid = logjac_arr.shape[0] // 2
y_mid = logjac_arr.shape[1] // 2
z_mid = logjac_arr.shape[2] // 2

print("\nCentral voxel location:")
print(f"x={x_mid}, y={y_mid}, z={z_mid}")
print(f"Central voxel log-Jacobian value: {logjac_arr[x_mid, y_mid, z_mid]:.6f}")

# A small raw-number patch from the central axial slice.
patch_radius = 4

raw_patch = logjac_arr[
    x_mid - patch_radius:x_mid + patch_radius + 1,
    y_mid - patch_radius:y_mid + patch_radius + 1,
    z_mid,
]

raw_patch_df = pd.DataFrame(np.round(raw_patch, 4))

print("\nSmall 9×9 raw value patch from the central axial slice:")
display(raw_patch_df)

# I also show a histogram of values to understand the distribution numerically.
plt.figure(figsize=(7, 4))
plt.hist(logjac_arr.flatten(), bins=100)
plt.title("Log-Jacobian value distribution")
plt.xlabel("log-Jacobian value")
plt.ylabel("Voxel count")
plt.tight_layout()
plt.show()

# Finally, I display the same map as slices for orientation.
vmax = float(np.nanpercentile(np.abs(logjac_arr), 99))
vmax = max(vmax, 1e-6)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

views = [
    ("Sagittal", np.rot90(logjac_arr[x_mid, :, :])),
    ("Coronal", np.rot90(logjac_arr[:, y_mid, :])),
    ("Axial", np.rot90(logjac_arr[:, :, z_mid])),
]

for ax, (title, image_slice) in zip(axes, views):
    im = ax.imshow(image_slice, cmap="viridis", vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")

plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75, label="log-Jacobian value")
plt.show()

## 1.54. T1 intensity normalization decision QC

Before choosing a final T1 intensity normalization method, inspect the intensity distributions of the cropped N4-corrected T1 images. This is only a decision/QC step. I am not normalizing or saving new images here.

The goal is to understand whether the current cropped T1 intensities vary strongly across subjects or scanner metadata such as field strength, manufacturer, scanner model family, ADNI phase, or diagnostic group. This matters because N4 bias correction reduces smooth within-scan intensity inhomogeneity, but it does not necessarily make T1 intensity scales comparable across subjects or scanners.

compute raw intensity summaries from the cropped T1 arrays only. not touch the log-Jacobian maps in this section.

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "RID_COL" in globals(), "RID_COL is not defined."
assert "IMAGE_ID_COL" in globals(), "IMAGE_ID_COL is not defined."
assert "LABEL_COL" in globals(), "LABEL_COL is not defined."

CROPPED_MANIFEST_PATH = (
    PROJECT_DIR
    / "manifest"
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_manifest_1063.csv"
)

assert CROPPED_MANIFEST_PATH.exists(), f"Cropped manifest not found: {CROPPED_MANIFEST_PATH}"

cropped_manifest = pd.read_csv(CROPPED_MANIFEST_PATH)

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "cropped_t1_npy_path",
]

missing_cols = [col for col in required_cols if col not in cropped_manifest.columns]
assert not missing_cols, f"Missing required columns in cropped manifest: {missing_cols}"

EXPECTED_CROP_SHAPE = (177, 213, 183)

T1_INTENSITY_QC_DIR = (
    PROJECT_DIR
    / "qc"
    / "clean_90d_mni_n4_syn_jacobian"
    / "mni_n4_syn_rectcrop_177x213x183"
    / "t1_intensity_normalization_decision"
)

T1_INTENSITY_QC_DIR.mkdir(parents=True, exist_ok=True)

T1_INTENSITY_QC_REPORT_PATH = (
    T1_INTENSITY_QC_DIR
    / "cropped_n4_t1_raw_intensity_distribution_qc_1063.csv"
)

T1_INTENSITY_QC_CHECKPOINT_PATH = (
    T1_INTENSITY_QC_DIR
    / "cropped_n4_t1_raw_intensity_distribution_qc_1063_checkpoint.csv"
)

T1_INTENSITY_GROUP_SUMMARY_PATH = (
    T1_INTENSITY_QC_DIR
    / "cropped_n4_t1_raw_intensity_distribution_group_summaries.csv"
)

FORCE_RERUN_T1_INTENSITY_QC = False

metadata_cols = [
    "final_group",
    "baseline_diagnosis",
    "baseline_phase",
    "baseline_date",
    "study_date",
    "days_from_baseline_mri",
    "field_strength_t_clean",
    "field_strength_category",
    "manufacturer_clean",
    "model_family",
    "research_group",
    "phase",
    "manifest_source",
]

metadata_cols = [col for col in metadata_cols if col in cropped_manifest.columns]

already_done_keys = set()
qc_rows = []

if T1_INTENSITY_QC_CHECKPOINT_PATH.exists() and not FORCE_RERUN_T1_INTENSITY_QC:
    checkpoint_df = pd.read_csv(T1_INTENSITY_QC_CHECKPOINT_PATH)
    qc_rows = checkpoint_df.to_dict("records")
    already_done_keys = set(
        zip(
            checkpoint_df[RID_COL].astype(str),
            checkpoint_df[IMAGE_ID_COL].astype(str),
        )
    )
    print(f"Loaded checkpoint with {len(qc_rows)} completed rows.")

start_time = time.time()

for row_number, (_, row) in enumerate(cropped_manifest.iterrows(), start=1):
    row_key = (str(row[RID_COL]), str(row[IMAGE_ID_COL]))

    if row_key in already_done_keys and not FORCE_RERUN_T1_INTENSITY_QC:
        continue

    t1_path = Path(row["cropped_t1_npy_path"])
    assert t1_path.exists(), f"Cropped T1 NPY missing: {t1_path}"

    t1_arr = np.load(t1_path, mmap_mode="r")

    assert tuple(t1_arr.shape) == EXPECTED_CROP_SHAPE, (
        f"Unexpected cropped T1 shape for RID {row[RID_COL]}, image {row[IMAGE_ID_COL]}: "
        f"{tuple(t1_arr.shape)}"
    )

    # I materialize the array once because percentile and finite-value calculations
    # need to scan the image anyway.
    t1_arr = np.asarray(t1_arr, dtype=np.float32)

    finite_mask = np.isfinite(t1_arr)
    finite_values = t1_arr[finite_mask]
    positive_values = t1_arr[finite_mask & (t1_arr > 0)]

    assert finite_values.size > 0, f"No finite values found for {t1_path}"
    assert positive_values.size > 0, f"No finite positive values found for {t1_path}"

    total_voxels = int(t1_arr.size)
    finite_voxels = int(finite_mask.sum())
    positive_voxels = int(positive_values.size)
    zero_voxels = int((finite_mask & (t1_arr == 0)).sum())
    negative_voxels = int((finite_mask & (t1_arr < 0)).sum())

    positive_percentiles = np.percentile(
        positive_values,
        [0.5, 1, 5, 25, 50, 75, 95, 99, 99.5],
    )

    finite_percentiles = np.percentile(
        finite_values,
        [0.5, 1, 5, 25, 50, 75, 95, 99, 99.5],
    )

    qc_row = {
        RID_COL: row[RID_COL],
        IMAGE_ID_COL: row[IMAGE_ID_COL],
        LABEL_COL: row[LABEL_COL],
        "cropped_t1_npy_path": str(t1_path),
        "t1_shape": str(tuple(t1_arr.shape)),
        "t1_dtype": str(t1_arr.dtype),
        "total_voxels": total_voxels,
        "finite_voxels": finite_voxels,
        "positive_voxels": positive_voxels,
        "zero_voxels": zero_voxels,
        "negative_voxels": negative_voxels,
        "finite_fraction": finite_voxels / total_voxels,
        "positive_fraction": positive_voxels / total_voxels,
        "zero_fraction": zero_voxels / total_voxels,
        "negative_fraction": negative_voxels / total_voxels,
        "finite_min": float(np.min(finite_values)),
        "finite_p00_5": float(finite_percentiles[0]),
        "finite_p01": float(finite_percentiles[1]),
        "finite_p05": float(finite_percentiles[2]),
        "finite_p25": float(finite_percentiles[3]),
        "finite_p50": float(finite_percentiles[4]),
        "finite_p75": float(finite_percentiles[5]),
        "finite_p95": float(finite_percentiles[6]),
        "finite_p99": float(finite_percentiles[7]),
        "finite_p99_5": float(finite_percentiles[8]),
        "finite_max": float(np.max(finite_values)),
        "finite_mean": float(np.mean(finite_values)),
        "finite_std": float(np.std(finite_values)),
        "positive_min": float(np.min(positive_values)),
        "positive_p00_5": float(positive_percentiles[0]),
        "positive_p01": float(positive_percentiles[1]),
        "positive_p05": float(positive_percentiles[2]),
        "positive_p25": float(positive_percentiles[3]),
        "positive_p50": float(positive_percentiles[4]),
        "positive_p75": float(positive_percentiles[5]),
        "positive_p95": float(positive_percentiles[6]),
        "positive_p99": float(positive_percentiles[7]),
        "positive_p99_5": float(positive_percentiles[8]),
        "positive_max": float(np.max(positive_values)),
        "positive_mean": float(np.mean(positive_values)),
        "positive_std": float(np.std(positive_values)),
        "positive_p99_to_p01_ratio": float(positive_percentiles[7] / max(positive_percentiles[1], 1e-8)),
        "positive_p99_to_p50_ratio": float(positive_percentiles[7] / max(positive_percentiles[4], 1e-8)),
    }

    for metadata_col in metadata_cols:
        qc_row[metadata_col] = row[metadata_col]

    qc_rows.append(qc_row)

    if row_number % 25 == 0 or row_number == len(cropped_manifest):
        checkpoint_df = pd.DataFrame(qc_rows)
        checkpoint_df.to_csv(T1_INTENSITY_QC_CHECKPOINT_PATH, index=False)

        elapsed_minutes = (time.time() - start_time) / 60
        print(
            f"Processed up to manifest row {row_number}/{len(cropped_manifest)} "
            f"for raw T1 intensity QC in {elapsed_minutes:.1f} minutes."
        )

t1_intensity_qc = pd.DataFrame(qc_rows)

# I align the QC report back to the original manifest order.
t1_intensity_qc = cropped_manifest[
    [RID_COL, IMAGE_ID_COL]
].merge(
    t1_intensity_qc,
    on=[RID_COL, IMAGE_ID_COL],
    how="left",
)

assert len(t1_intensity_qc) == len(cropped_manifest), (
    f"Expected {len(cropped_manifest)} QC rows, got {len(t1_intensity_qc)}."
)

assert t1_intensity_qc["positive_p99"].notna().all(), (
    "Some subjects are missing T1 intensity QC values."
)

t1_intensity_qc.to_csv(T1_INTENSITY_QC_REPORT_PATH, index=False)

print("\nSaved raw cropped T1 intensity QC report:")
print(T1_INTENSITY_QC_REPORT_PATH)

print("\nOverall intensity summary across subjects:")
display(
    t1_intensity_qc[
        [
            "zero_fraction",
            "positive_fraction",
            "positive_p01",
            "positive_p50",
            "positive_p99",
            "positive_mean",
            "positive_std",
            "positive_p99_to_p01_ratio",
            "positive_p99_to_p50_ratio",
        ]
    ].describe()
)

print("\nCohort counts:")
display(t1_intensity_qc[LABEL_COL].value_counts().to_frame("count"))

# I create compact group summaries for the metadata columns that exist.
summary_frames = []

grouping_cols = [
    LABEL_COL,
    "field_strength_category",
    "manufacturer_clean",
    "model_family",
    "research_group",
    "phase",
    "baseline_phase",
]

grouping_cols = [col for col in grouping_cols if col in t1_intensity_qc.columns]

for group_col in grouping_cols:
    group_summary = (
        t1_intensity_qc
        .groupby(group_col, dropna=False)
        .agg(
            n_subjects=(RID_COL, "count"),
            median_positive_p01=("positive_p01", "median"),
            median_positive_p50=("positive_p50", "median"),
            median_positive_p99=("positive_p99", "median"),
            median_positive_mean=("positive_mean", "median"),
            median_positive_std=("positive_std", "median"),
            median_zero_fraction=("zero_fraction", "median"),
            max_positive_p99=("positive_p99", "max"),
            min_positive_p99=("positive_p99", "min"),
        )
        .reset_index()
    )

    group_summary.insert(0, "grouping_variable", group_col)
    group_summary = group_summary.rename(columns={group_col: "group_value"})

    summary_frames.append(group_summary)

group_summaries = pd.concat(summary_frames, ignore_index=True)
group_summaries.to_csv(T1_INTENSITY_GROUP_SUMMARY_PATH, index=False)

print("\nSaved group summaries:")
print(T1_INTENSITY_GROUP_SUMMARY_PATH)

for group_col in grouping_cols:
    print(f"\nSummary by {group_col}:")
    display(
        group_summaries
        .loc[group_summaries["grouping_variable"] == group_col]
        .sort_values("n_subjects", ascending=False)
        .head(30)
    )

## 1.55. Preview candidate T1 normalization methods on extreme intensity-scale cases

The raw cropped N4 T1 intensity QC showed large scanner/manufacturer-related intensity scale differences, especially between Siemens and Philips scans. This means that the raw N4-corrected T1 values should not be used directly as model inputs.

Before saving a final normalized T1 branch, visually compare a few candidate normalization methods on the same subjects with extreme raw intensity scales. This is only a preview step. I am not saving new normalized images here, and I am not touching the log-Jacobian maps.

The candidate methods are:

1. Raw cropped N4 T1, shown for reference.
2. Per-subject positive-voxel `p1-p99` min-max scaling.
3. Per-subject positive-voxel `p0.5-p99.5` min-max scaling.
4. Per-subject positive-voxel `p1-p99` clipping followed by z-score normalization.

These methods do not use a brain mask or skull stripping. They only use finite positive T1 voxels to avoid exact zero background dominating the statistics.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "RID_COL" in globals(), "RID_COL is not defined."
assert "IMAGE_ID_COL" in globals(), "IMAGE_ID_COL is not defined."
assert "LABEL_COL" in globals(), "LABEL_COL is not defined."

EXPECTED_CROP_SHAPE = (177, 213, 183)

T1_INTENSITY_QC_REPORT_PATH = (
    PROJECT_DIR
    / "qc"
    / "clean_90d_mni_n4_syn_jacobian"
    / "mni_n4_syn_rectcrop_177x213x183"
    / "t1_intensity_normalization_decision"
    / "cropped_n4_t1_raw_intensity_distribution_qc_1063.csv"
)

CROPPED_MANIFEST_PATH = (
    PROJECT_DIR
    / "manifest"
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_manifest_1063.csv"
)

assert T1_INTENSITY_QC_REPORT_PATH.exists(), f"Missing QC report: {T1_INTENSITY_QC_REPORT_PATH}"
assert CROPPED_MANIFEST_PATH.exists(), f"Missing cropped manifest: {CROPPED_MANIFEST_PATH}"

t1_intensity_qc = pd.read_csv(T1_INTENSITY_QC_REPORT_PATH)
cropped_manifest = pd.read_csv(CROPPED_MANIFEST_PATH)

# I merge the cropped T1 path back into the intensity QC table.
t1_intensity_qc = t1_intensity_qc.merge(
    cropped_manifest[
        [
            RID_COL,
            IMAGE_ID_COL,
            "cropped_t1_npy_path",
        ]
    ],
    on=[RID_COL, IMAGE_ID_COL],
    how="left",
    suffixes=("", "_manifest"),
)

assert t1_intensity_qc["cropped_t1_npy_path"].notna().all(), (
    "Some intensity QC rows are missing cropped T1 paths."
)

# I choose extreme cases from different failure modes:
# very low p99, very high p99, high std, and a few manufacturer representatives.
N_PER_EXTREME_TYPE = 3

lowest_p99_cases = (
    t1_intensity_qc
    .sort_values("positive_p99", ascending=True)
    .head(N_PER_EXTREME_TYPE)
)

highest_p99_cases = (
    t1_intensity_qc
    .sort_values("positive_p99", ascending=False)
    .head(N_PER_EXTREME_TYPE)
)

highest_std_cases = (
    t1_intensity_qc
    .sort_values("positive_std", ascending=False)
    .head(N_PER_EXTREME_TYPE)
)

manufacturer_examples = (
    t1_intensity_qc
    .sort_values([LABEL_COL, RID_COL, IMAGE_ID_COL])
    .groupby("manufacturer_clean", dropna=False, group_keys=False)
    .head(2)
)

preview_cases = (
    pd.concat(
        [
            lowest_p99_cases,
            highest_p99_cases,
            highest_std_cases,
            manufacturer_examples,
        ],
        ignore_index=True,
    )
    .drop_duplicates(subset=[RID_COL, IMAGE_ID_COL])
    .head(14)
    .copy()
)

display_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "field_strength_category",
    "manufacturer_clean",
    "model_family",
    "research_group",
    "phase",
    "positive_p01",
    "positive_p50",
    "positive_p99",
    "positive_mean",
    "positive_std",
    "zero_fraction",
    "cropped_t1_npy_path",
]

display_cols = [col for col in display_cols if col in preview_cases.columns]

print(f"Selected {len(preview_cases)} cases for normalization preview.")
display(preview_cases[display_cols])


def positive_voxel_values(arr):
    finite_mask = np.isfinite(arr)
    values = arr[finite_mask & (arr > 0)]

    if values.size == 0:
        raise ValueError("No finite positive voxels found.")

    return values


def normalize_pminmax(arr, low_percentile, high_percentile):
    values = positive_voxel_values(arr)

    p_low = float(np.percentile(values, low_percentile))
    p_high = float(np.percentile(values, high_percentile))

    if p_high <= p_low:
        raise ValueError(f"Invalid percentile range: p_low={p_low}, p_high={p_high}")

    out = (arr.astype(np.float32) - p_low) / (p_high - p_low)
    out = np.clip(out, 0.0, 1.0)
    out[~np.isfinite(out)] = 0.0

    return out.astype(np.float32)


def normalize_clipped_zscore(arr, low_percentile=1.0, high_percentile=99.0):
    values = positive_voxel_values(arr)

    p_low = float(np.percentile(values, low_percentile))
    p_high = float(np.percentile(values, high_percentile))

    if p_high <= p_low:
        raise ValueError(f"Invalid percentile range: p_low={p_low}, p_high={p_high}")

    clipped = np.clip(arr.astype(np.float32), p_low, p_high)

    clipped_values = clipped[np.isfinite(arr) & (arr > 0)]
    mean = float(np.mean(clipped_values))
    std = float(np.std(clipped_values))

    if std <= 0 or not np.isfinite(std):
        raise ValueError(f"Invalid z-score std: {std}")

    out = (clipped - mean) / std
    out[~np.isfinite(out)] = 0.0

    return out.astype(np.float32)


def get_middle_slices(arr):
    x_mid = arr.shape[0] // 2
    y_mid = arr.shape[1] // 2
    z_mid = arr.shape[2] // 2

    return {
        "Sagittal": np.rot90(arr[x_mid, :, :]),
        "Coronal": np.rot90(arr[:, y_mid, :]),
        "Axial": np.rot90(arr[:, :, z_mid]),
    }


# I use coronal view first because it is usually easiest to judge tissue contrast.
VIEW = "Coronal"

n_rows = len(preview_cases)
n_cols = 5

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 3.1 * n_rows),
    gridspec_kw={"width_ratios": [2.4, 3, 3, 3, 3]},
)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

column_titles = [
    "Metadata",
    "Raw N4 T1",
    "p1–p99 min-max",
    "p0.5–p99.5 min-max",
    "p1–p99 clipped z-score",
]

for col_idx, title in enumerate(column_titles):
    axes[0, col_idx].set_title(title, fontsize=11)

for row_idx, (_, row) in enumerate(preview_cases.iterrows()):
    t1_arr = np.load(row["cropped_t1_npy_path"]).astype(np.float32)

    assert t1_arr.shape == EXPECTED_CROP_SHAPE, (
        f"Unexpected cropped T1 shape for RID {row[RID_COL]}: {t1_arr.shape}"
    )

    t1_p1p99_minmax = normalize_pminmax(t1_arr, 1.0, 99.0)
    t1_p005p995_minmax = normalize_pminmax(t1_arr, 0.5, 99.5)
    t1_p1p99_zscore = normalize_clipped_zscore(t1_arr, 1.0, 99.0)

    raw_vmax = float(row["positive_p99"])
    raw_vmax = max(raw_vmax, 1e-6)

    z_abs = float(np.nanpercentile(np.abs(t1_p1p99_zscore), 99))
    z_abs = max(z_abs, 1e-6)

    metadata_lines = [
        f"Group: {row[LABEL_COL]}",
        f"RID: {row[RID_COL]}",
        f"Image: {row[IMAGE_ID_COL]}",
        f"p50: {row['positive_p50']:.2f}",
        f"p99: {row['positive_p99']:.2f}",
        f"std: {row['positive_std']:.2f}",
    ]

    for metadata_col in [
        "field_strength_category",
        "manufacturer_clean",
        "model_family",
        "research_group",
        "phase",
    ]:
        if metadata_col in row.index and pd.notna(row[metadata_col]):
            metadata_lines.append(f"{metadata_col}: {row[metadata_col]}")

    ax = axes[row_idx, 0]
    ax.axis("off")
    ax.text(
        0.0,
        0.5,
        "\n".join(metadata_lines),
        ha="left",
        va="center",
        fontsize=8,
        transform=ax.transAxes,
    )

    image_panels = [
        (t1_arr, "gray", 0.0, raw_vmax),
        (t1_p1p99_minmax, "gray", 0.0, 1.0),
        (t1_p005p995_minmax, "gray", 0.0, 1.0),
        (t1_p1p99_zscore, "gray", -z_abs, z_abs),
    ]

    for col_idx, (panel_arr, cmap_name, vmin, vmax) in enumerate(image_panels, start=1):
        ax = axes[row_idx, col_idx]
        ax.imshow(
            get_middle_slices(panel_arr)[VIEW],
            cmap=cmap_name,
            vmin=vmin,
            vmax=vmax,
        )
        ax.axis("off")

plt.suptitle(
    f"T1 normalization preview on raw intensity-scale edge cases | {VIEW} view",
    fontsize=14,
    y=0.995,
)

plt.subplots_adjust(
    left=0.03,
    right=0.99,
    top=0.96,
    bottom=0.02,
    wspace=0.05,
    hspace=0.12,
)

plt.show()

## 1.56. Sanity-check the p1-p99 T1 normalization scale

The raw intensity QC showed that the cropped N4-corrected T1 images have very different numerical scales across scanner manufacturers and model families. Before saving a normalized T1 branch, sanity-check the candidate `p1-p99` min-max normalization.

This check confirms that the raw p1/p50/p99 values may differ strongly between subjects, but after normalization the values are mapped onto a common `[0, 1]` scale. This is only a numerical check. I am not saving normalized images in this cell, and I am not touching the log-Jacobian maps.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "RID_COL" in globals(), "RID_COL is not defined."
assert "IMAGE_ID_COL" in globals(), "IMAGE_ID_COL is not defined."
assert "LABEL_COL" in globals(), "LABEL_COL is not defined."

EXPECTED_CROP_SHAPE = (177, 213, 183)

T1_INTENSITY_QC_REPORT_PATH = (
    PROJECT_DIR
    / "qc"
    / "clean_90d_mni_n4_syn_jacobian"
    / "mni_n4_syn_rectcrop_177x213x183"
    / "t1_intensity_normalization_decision"
    / "cropped_n4_t1_raw_intensity_distribution_qc_1063.csv"
)

CROPPED_MANIFEST_PATH = (
    PROJECT_DIR
    / "manifest"
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_manifest_1063.csv"
)

assert T1_INTENSITY_QC_REPORT_PATH.exists(), f"Missing QC report: {T1_INTENSITY_QC_REPORT_PATH}"
assert CROPPED_MANIFEST_PATH.exists(), f"Missing cropped manifest: {CROPPED_MANIFEST_PATH}"

t1_intensity_qc = pd.read_csv(T1_INTENSITY_QC_REPORT_PATH)
cropped_manifest = pd.read_csv(CROPPED_MANIFEST_PATH)

if "cropped_t1_npy_path" not in t1_intensity_qc.columns:
    t1_intensity_qc = t1_intensity_qc.merge(
        cropped_manifest[
            [
                RID_COL,
                IMAGE_ID_COL,
                "cropped_t1_npy_path",
            ]
        ],
        on=[RID_COL, IMAGE_ID_COL],
        how="left",
    )

assert t1_intensity_qc["cropped_t1_npy_path"].notna().all(), (
    "Some QC rows are missing cropped T1 paths."
)


def normalize_p1_p99_minmax(arr):
    """
    I apply the candidate T1 normalization without saving it.

    Finite positive voxels define the 1st and 99th percentiles.
    The full array is clipped to this range and linearly rescaled to [0, 1].
    """
    arr = arr.astype(np.float32)

    finite_mask = np.isfinite(arr)
    positive_values = arr[finite_mask & (arr > 0)]

    if positive_values.size == 0:
        raise ValueError("No finite positive voxels found.")

    p01 = float(np.percentile(positive_values, 1))
    p99 = float(np.percentile(positive_values, 99))

    if not np.isfinite(p01) or not np.isfinite(p99) or p99 <= p01:
        raise ValueError(f"Invalid p1/p99 range: p1={p01}, p99={p99}")

    normalized = (arr - p01) / (p99 - p01)
    normalized = np.clip(normalized, 0.0, 1.0)
    normalized[~finite_mask] = 0.0

    return normalized.astype(np.float32), p01, p99


# I reuse the same kind of high-contrast examples:
# lowest raw p99, highest raw p99, highest raw std, and manufacturer representatives.
N_PER_EXTREME_TYPE = 3

lowest_p99_cases = (
    t1_intensity_qc
    .sort_values("positive_p99", ascending=True)
    .head(N_PER_EXTREME_TYPE)
)

highest_p99_cases = (
    t1_intensity_qc
    .sort_values("positive_p99", ascending=False)
    .head(N_PER_EXTREME_TYPE)
)

highest_std_cases = (
    t1_intensity_qc
    .sort_values("positive_std", ascending=False)
    .head(N_PER_EXTREME_TYPE)
)

if "manufacturer_clean" in t1_intensity_qc.columns:
    manufacturer_examples = (
        t1_intensity_qc
        .sort_values([LABEL_COL, RID_COL, IMAGE_ID_COL])
        .groupby("manufacturer_clean", dropna=False, group_keys=False)
        .head(2)
    )
else:
    manufacturer_examples = t1_intensity_qc.head(0)

sanity_cases = (
    pd.concat(
        [
            lowest_p99_cases,
            highest_p99_cases,
            highest_std_cases,
            manufacturer_examples,
        ],
        ignore_index=True,
    )
    .drop_duplicates(subset=[RID_COL, IMAGE_ID_COL])
    .head(14)
    .copy()
)

sanity_rows = []

for _, row in sanity_cases.iterrows():
    t1_path = Path(row["cropped_t1_npy_path"])
    assert t1_path.exists(), f"Cropped T1 file missing: {t1_path}"

    raw_arr = np.load(t1_path).astype(np.float32)

    assert raw_arr.shape == EXPECTED_CROP_SHAPE, (
        f"Unexpected cropped T1 shape for RID {row[RID_COL]}: {raw_arr.shape}"
    )

    norm_arr, raw_p01_used, raw_p99_used = normalize_p1_p99_minmax(raw_arr)

    finite_norm = norm_arr[np.isfinite(norm_arr)]
    positive_norm = norm_arr[np.isfinite(norm_arr) & (norm_arr > 0)]

    sanity_rows.append({
        RID_COL: row[RID_COL],
        IMAGE_ID_COL: row[IMAGE_ID_COL],
        LABEL_COL: row[LABEL_COL],
        "field_strength_category": row.get("field_strength_category", np.nan),
        "manufacturer_clean": row.get("manufacturer_clean", np.nan),
        "model_family": row.get("model_family", np.nan),
        "raw_positive_p01_from_qc": row["positive_p01"],
        "raw_positive_p50_from_qc": row["positive_p50"],
        "raw_positive_p99_from_qc": row["positive_p99"],
        "raw_positive_std_from_qc": row["positive_std"],
        "raw_p01_used_now": raw_p01_used,
        "raw_p99_used_now": raw_p99_used,
        "norm_min": float(np.min(finite_norm)),
        "norm_p01_all_finite": float(np.percentile(finite_norm, 1)),
        "norm_p50_all_finite": float(np.percentile(finite_norm, 50)),
        "norm_p99_all_finite": float(np.percentile(finite_norm, 99)),
        "norm_max": float(np.max(finite_norm)),
        "norm_positive_p01": float(np.percentile(positive_norm, 1)) if positive_norm.size else np.nan,
        "norm_positive_p50": float(np.percentile(positive_norm, 50)) if positive_norm.size else np.nan,
        "norm_positive_p99": float(np.percentile(positive_norm, 99)) if positive_norm.size else np.nan,
        "norm_mean_all_finite": float(np.mean(finite_norm)),
        "norm_std_all_finite": float(np.std(finite_norm)),
        "norm_shape": str(tuple(norm_arr.shape)),
        "norm_all_finite": bool(np.isfinite(norm_arr).all()),
        "norm_in_0_1": bool((norm_arr.min() >= 0.0) and (norm_arr.max() <= 1.0)),
    })

sanity_check_df = pd.DataFrame(sanity_rows)

print("Sanity-check examples:")
display(
    sanity_check_df[
        [
            RID_COL,
            IMAGE_ID_COL,
            LABEL_COL,
            "manufacturer_clean",
            "model_family",
            "raw_positive_p01_from_qc",
            "raw_positive_p50_from_qc",
            "raw_positive_p99_from_qc",
            "norm_min",
            "norm_p01_all_finite",
            "norm_p50_all_finite",
            "norm_p99_all_finite",
            "norm_max",
            "norm_positive_p50",
            "norm_positive_p99",
            "norm_all_finite",
            "norm_in_0_1",
        ]
    ]
)

print("\nSummary of normalized values in sanity-check cases:")
display(
    sanity_check_df[
        [
            "norm_min",
            "norm_p01_all_finite",
            "norm_p50_all_finite",
            "norm_p99_all_finite",
            "norm_max",
            "norm_positive_p50",
            "norm_positive_p99",
            "norm_mean_all_finite",
            "norm_std_all_finite",
        ]
    ].describe()
)

print("\nAll selected cases finite after normalization:")
print(sanity_check_df["norm_all_finite"].all())

print("\nAll selected cases within [0, 1] after normalization:")
print(sanity_check_df["norm_in_0_1"].all())

## 1.57. Plot raw versus normalized T1 intensity scale

plot the raw p99 values against the post-normalization p99 values for the sanity-check examples. The raw values should vary widely, while the normalized values should be close to `1.0`.

In [ ]:
assert "sanity_check_df" in globals(), "sanity_check_df is not available."

plt.figure(figsize=(8, 4))
plt.scatter(
    range(len(sanity_check_df)),
    sanity_check_df["raw_positive_p99_from_qc"],
)
plt.yscale("log")
plt.xlabel("Sanity-check subject index")
plt.ylabel("Raw positive p99, log scale")
plt.title("Raw cropped N4 T1 p99 varies strongly across selected subjects")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.scatter(
    range(len(sanity_check_df)),
    sanity_check_df["norm_p99_all_finite"],
)
plt.xlabel("Sanity-check subject index")
plt.ylabel("Post-normalization p99")
plt.title("After p1–p99 min-max normalization, p99 is on a common scale")
plt.tight_layout()
plt.show()

## 1.58. Apply full T1 p1-p99 percentile normalization

The raw cropped N4 T1 intensity QC showed large scanner/manufacturer-related intensity-scale differences. therefore create a normalized T1 branch using per-subject positive-voxel percentile clipping followed by affine min-max scaling.

For each cropped T1 MRI, I will:

1. Load the saved cropped T1 array.
2. Use finite positive voxels to estimate the 1st and 99th percentiles.
3. Clip the full image to that range.
4. Rescale the clipped image to `[0, 1]`.
5. Save both `.npy` arrays for training and `.nii.gz` files for QC.

This step does not skull-strip, does not apply a brain mask, and does not modify the log-Jacobian maps.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

import ants

assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "RID_COL" in globals(), "RID_COL is not defined."
assert "IMAGE_ID_COL" in globals(), "IMAGE_ID_COL is not defined."
assert "LABEL_COL" in globals(), "LABEL_COL is not defined."

CROPPED_MANIFEST_PATH = (
    PROJECT_DIR
    / "manifest"
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_manifest_1063.csv"
)

assert CROPPED_MANIFEST_PATH.exists(), f"Cropped manifest not found: {CROPPED_MANIFEST_PATH}"

cropped_manifest = pd.read_csv(CROPPED_MANIFEST_PATH)

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "cropped_t1_nii_path",
    "cropped_t1_npy_path",
]

missing_cols = [col for col in required_cols if col not in cropped_manifest.columns]
assert not missing_cols, f"Missing required columns in cropped manifest: {missing_cols}"

EXPECTED_CROP_SHAPE = (177, 213, 183)

T1_NORM_BRANCH_NAME = "mni_n4_syn_rectcrop_177x213x183_t1norm_p01p99"

NORMALIZED_ROOT = PROJECT_DIR / "processed" / T1_NORM_BRANCH_NAME
NORMALIZED_T1_NII_DIR = NORMALIZED_ROOT / "nii_t1"
NORMALIZED_T1_NPY_DIR = NORMALIZED_ROOT / "npy_t1"

NORMALIZED_T1_NII_DIR.mkdir(parents=True, exist_ok=True)
NORMALIZED_T1_NPY_DIR.mkdir(parents=True, exist_ok=True)

T1_NORM_QC_DIR = (
    PROJECT_DIR
    / "qc"
    / "clean_90d_mni_n4_syn_jacobian"
    / T1_NORM_BRANCH_NAME
)

T1_NORM_QC_DIR.mkdir(parents=True, exist_ok=True)

T1_NORM_REPORT_PATH = (
    T1_NORM_QC_DIR
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_t1norm_p01p99_report.csv"
)

T1_NORM_CHECKPOINT_REPORT_PATH = (
    T1_NORM_QC_DIR
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_t1norm_p01p99_checkpoint_report.csv"
)

T1_NORM_MANIFEST_PATH = (
    PROJECT_DIR
    / "manifest"
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_t1norm_p01p99_manifest_1063.csv"
)

FORCE_RERUN_T1_NORMALIZATION = False


def make_subject_stem(row):
    """
    I create a stable filename stem for each subject/image.
    If the existing manifest already contains an output stem, I reuse it.
    """
    if "output_stem" in row.index and pd.notna(row["output_stem"]):
        return str(row["output_stem"])

    rid = str(row[RID_COL]).replace(".0", "")
    image_id = str(row[IMAGE_ID_COL]).replace(".0", "")

    return f"RID-{rid}_image-{image_id}"


def normalize_t1_p01_p99_minmax(t1_arr):
    """
    I normalize one cropped T1 image using finite positive voxels.

    The 1st and 99th percentiles are estimated from positive voxels only.
    The full image is clipped to that range and linearly rescaled to [0, 1].
    This is not skull stripping and not masking.
    """
    t1_arr = t1_arr.astype(np.float32)

    finite_mask = np.isfinite(t1_arr)
    positive_values = t1_arr[finite_mask & (t1_arr > 0)]

    if positive_values.size == 0:
        raise ValueError("No finite positive voxels found.")

    p01 = float(np.percentile(positive_values, 1.0))
    p99 = float(np.percentile(positive_values, 99.0))

    if not np.isfinite(p01) or not np.isfinite(p99) or p99 <= p01:
        raise ValueError(f"Invalid p1/p99 range: p1={p01}, p99={p99}")

    normalized = (t1_arr - p01) / (p99 - p01)
    normalized = np.clip(normalized, 0.0, 1.0)
    normalized[~finite_mask] = 0.0

    return normalized.astype(np.float32), p01, p99


def existing_normalized_output_is_valid(npy_path, expected_shape):
    """
    I use the saved normalized NPY file as the main checkpoint.
    If it exists, has the expected shape, contains finite values, and is within [0, 1],
    I treat the subject as already completed.
    """
    if not Path(npy_path).exists():
        return False

    try:
        arr = np.load(npy_path, mmap_mode="r")

        if tuple(arr.shape) != tuple(expected_shape):
            return False

        arr_min = float(np.nanmin(arr))
        arr_max = float(np.nanmax(arr))

        if not np.isfinite(arr_min) or not np.isfinite(arr_max):
            return False

        if arr_min < 0.0 or arr_max > 1.0:
            return False

        return True

    except Exception:
        return False


report_rows = []
start_time = time.time()

for row_number, (_, row) in enumerate(cropped_manifest.iterrows(), start=1):
    subject_stem = make_subject_stem(row)

    input_t1_nii_path = Path(row["cropped_t1_nii_path"])
    input_t1_npy_path = Path(row["cropped_t1_npy_path"])

    assert input_t1_nii_path.exists(), f"Cropped T1 NIfTI missing: {input_t1_nii_path}"
    assert input_t1_npy_path.exists(), f"Cropped T1 NPY missing: {input_t1_npy_path}"

    output_t1_nii_path = (
        NORMALIZED_T1_NII_DIR
        / f"{subject_stem}_space-MNI152NLin2009cSym_desc-n4SynRectcrop177x213x183_t1norm-p01p99_T1w.nii.gz"
    )

    output_t1_npy_path = (
        NORMALIZED_T1_NPY_DIR
        / f"{subject_stem}_space-MNI152NLin2009cSym_desc-n4SynRectcrop177x213x183_t1norm-p01p99_T1w.npy"
    )

    outputs_exist = output_t1_nii_path.exists() and output_t1_npy_path.exists()

    output_valid = (
        outputs_exist
        and not FORCE_RERUN_T1_NORMALIZATION
        and existing_normalized_output_is_valid(output_t1_npy_path, EXPECTED_CROP_SHAPE)
    )

    # I load the raw cropped T1 so I can record the exact p1/p99 values used.
    raw_t1_arr = np.load(input_t1_npy_path).astype(np.float32)

    assert raw_t1_arr.shape == EXPECTED_CROP_SHAPE, (
        f"Unexpected cropped T1 shape for {subject_stem}: {raw_t1_arr.shape}"
    )

    # I compute the normalization values even for skipped subjects so the report is complete.
    _, raw_p01, raw_p99 = normalize_t1_p01_p99_minmax(raw_t1_arr)

    if output_valid:
        status = "skipped_existing"

        normalized_t1_arr = np.load(output_t1_npy_path, mmap_mode="r")
        normalized_shape = tuple(normalized_t1_arr.shape)

        normalized_min = float(np.nanmin(normalized_t1_arr))
        normalized_p01 = float(np.nanpercentile(normalized_t1_arr, 1))
        normalized_p50 = float(np.nanpercentile(normalized_t1_arr, 50))
        normalized_p99 = float(np.nanpercentile(normalized_t1_arr, 99))
        normalized_max = float(np.nanmax(normalized_t1_arr))
        normalized_mean = float(np.nanmean(normalized_t1_arr))
        normalized_std = float(np.nanstd(normalized_t1_arr))

    else:
        normalized_t1_arr, raw_p01, raw_p99 = normalize_t1_p01_p99_minmax(raw_t1_arr)

        assert normalized_t1_arr.shape == EXPECTED_CROP_SHAPE, (
            f"Unexpected normalized T1 shape for {subject_stem}: {normalized_t1_arr.shape}"
        )

        assert np.isfinite(normalized_t1_arr).all(), (
            f"Non-finite values found after T1 normalization for {subject_stem}."
        )

        assert normalized_t1_arr.min() >= 0.0 and normalized_t1_arr.max() <= 1.0, (
            f"Normalized T1 values outside [0, 1] for {subject_stem}."
        )

        # I preserve the cropped NIfTI spatial metadata and replace only voxel values.
        cropped_t1_img = ants.image_read(str(input_t1_nii_path))
        normalized_t1_img = cropped_t1_img.new_image_like(normalized_t1_arr)

        ants.image_write(normalized_t1_img, str(output_t1_nii_path))
        np.save(output_t1_npy_path, normalized_t1_arr)

        status = "created"

        normalized_shape = tuple(normalized_t1_arr.shape)
        normalized_min = float(np.nanmin(normalized_t1_arr))
        normalized_p01 = float(np.nanpercentile(normalized_t1_arr, 1))
        normalized_p50 = float(np.nanpercentile(normalized_t1_arr, 50))
        normalized_p99 = float(np.nanpercentile(normalized_t1_arr, 99))
        normalized_max = float(np.nanmax(normalized_t1_arr))
        normalized_mean = float(np.nanmean(normalized_t1_arr))
        normalized_std = float(np.nanstd(normalized_t1_arr))

    report_row = {
        RID_COL: row[RID_COL],
        IMAGE_ID_COL: row[IMAGE_ID_COL],
        LABEL_COL: row[LABEL_COL],
        "output_stem": subject_stem,
        "cropped_t1_nii_path": str(input_t1_nii_path),
        "cropped_t1_npy_path": str(input_t1_npy_path),
        "normalized_t1_nii_path": str(output_t1_nii_path),
        "normalized_t1_npy_path": str(output_t1_npy_path),
        "t1_norm_method": "positive_voxel_p01_p99_clip_minmax",
        "t1_norm_uses_brain_mask": False,
        "t1_norm_uses_skull_stripping": False,
        "t1_norm_lower_percentile": 1.0,
        "t1_norm_upper_percentile": 99.0,
        "raw_t1_positive_p01_used": raw_p01,
        "raw_t1_positive_p99_used": raw_p99,
        "normalized_t1_shape": str(normalized_shape),
        "normalized_t1_min": normalized_min,
        "normalized_t1_p01": normalized_p01,
        "normalized_t1_p50": normalized_p50,
        "normalized_t1_p99": normalized_p99,
        "normalized_t1_max": normalized_max,
        "normalized_t1_mean": normalized_mean,
        "normalized_t1_std": normalized_std,
        "t1_normalization_status": status,
    }

    for metadata_col in [
        "baseline_diagnosis",
        "baseline_phase",
        "baseline_date",
        "study_date",
        "days_from_baseline_mri",
        "field_strength_t_clean",
        "field_strength_category",
        "manufacturer_clean",
        "model_family",
        "research_group",
        "phase",
        "manifest_source",
    ]:
        if metadata_col in cropped_manifest.columns:
            report_row[metadata_col] = row[metadata_col]

    report_rows.append(report_row)

    if row_number % 25 == 0 or row_number == len(cropped_manifest):
        checkpoint_df = pd.DataFrame(report_rows)
        checkpoint_df.to_csv(T1_NORM_CHECKPOINT_REPORT_PATH, index=False)

        elapsed_minutes = (time.time() - start_time) / 60
        print(
            f"Processed {row_number}/{len(cropped_manifest)} T1 images "
            f"in {elapsed_minutes:.1f} minutes."
        )

t1_norm_report = pd.DataFrame(report_rows)

assert len(t1_norm_report) == len(cropped_manifest), (
    f"Expected {len(cropped_manifest)} report rows, got {len(t1_norm_report)}."
)

t1_norm_report.to_csv(T1_NORM_REPORT_PATH, index=False)

t1_norm_manifest = cropped_manifest.merge(
    t1_norm_report[
        [
            RID_COL,
            IMAGE_ID_COL,
            "normalized_t1_nii_path",
            "normalized_t1_npy_path",
            "t1_norm_method",
            "t1_norm_uses_brain_mask",
            "t1_norm_uses_skull_stripping",
            "t1_norm_lower_percentile",
            "t1_norm_upper_percentile",
            "raw_t1_positive_p01_used",
            "raw_t1_positive_p99_used",
            "normalized_t1_shape",
            "normalized_t1_min",
            "normalized_t1_p01",
            "normalized_t1_p50",
            "normalized_t1_p99",
            "normalized_t1_max",
            "normalized_t1_mean",
            "normalized_t1_std",
            "t1_normalization_status",
        ]
    ],
    on=[RID_COL, IMAGE_ID_COL],
    how="left",
)

t1_norm_manifest["normalized_t1_nii_exists"] = t1_norm_manifest["normalized_t1_nii_path"].apply(
    lambda p: Path(p).exists()
)

t1_norm_manifest["normalized_t1_npy_exists"] = t1_norm_manifest["normalized_t1_npy_path"].apply(
    lambda p: Path(p).exists()
)

t1_norm_manifest["normalized_t1_npy_valid"] = t1_norm_manifest["normalized_t1_npy_path"].apply(
    lambda p: existing_normalized_output_is_valid(p, EXPECTED_CROP_SHAPE)
)

t1_norm_manifest.to_csv(T1_NORM_MANIFEST_PATH, index=False)

print("\nSaved T1 normalization report:")
print(T1_NORM_REPORT_PATH)

print("\nSaved T1-normalized manifest:")
print(T1_NORM_MANIFEST_PATH)

print("\nT1 normalization status counts:")
display(t1_norm_report["t1_normalization_status"].value_counts().to_frame("count"))

print("\nOutput existence and validity checks:")
display(
    t1_norm_manifest[
        [
            "normalized_t1_nii_exists",
            "normalized_t1_npy_exists",
            "normalized_t1_npy_valid",
        ]
    ].sum().to_frame("count")
)

print("\nNormalized T1 value summary:")
display(
    t1_norm_report[
        [
            "normalized_t1_min",
            "normalized_t1_p01",
            "normalized_t1_p50",
            "normalized_t1_p99",
            "normalized_t1_max",
            "normalized_t1_mean",
            "normalized_t1_std",
        ]
    ].describe()
)

print("\nCohort counts:")
display(t1_norm_manifest[LABEL_COL].value_counts().to_frame("count"))

assert t1_norm_manifest["normalized_t1_nii_exists"].all(), "Some normalized T1 NIfTI files are missing."
assert t1_norm_manifest["normalized_t1_npy_exists"].all(), "Some normalized T1 NPY files are missing."
assert t1_norm_manifest["normalized_t1_npy_valid"].all(), "Some normalized T1 NPY files are invalid."

print("\nFull T1 normalization completed successfully.")

## 1.59. Disk-based QC of the normalized T1 branch

The full T1 normalization step completed successfully. verify the saved normalized T1 branch from disk. This checks that every normalized T1 `.npy` and `.nii.gz` file exists, that each normalized array has the expected crop shape `177 × 213 × 183`, that all values are finite, and that all values remain within `[0, 1]`.

This QC is performed on the saved normalized files, not on arrays kept in memory during normalization.

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

assert "PROJECT_DIR" in globals(), "PROJECT_DIR is not defined."
assert "RID_COL" in globals(), "RID_COL is not defined."
assert "IMAGE_ID_COL" in globals(), "IMAGE_ID_COL is not defined."
assert "LABEL_COL" in globals(), "LABEL_COL is not defined."

EXPECTED_CROP_SHAPE = (177, 213, 183)

T1_NORM_BRANCH_NAME = "mni_n4_syn_rectcrop_177x213x183_t1norm_p01p99"

T1_NORM_MANIFEST_PATH = (
    PROJECT_DIR
    / "manifest"
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_t1norm_p01p99_manifest_1063.csv"
)

assert T1_NORM_MANIFEST_PATH.exists(), f"T1-normalized manifest not found: {T1_NORM_MANIFEST_PATH}"

t1_norm_manifest = pd.read_csv(T1_NORM_MANIFEST_PATH)

required_cols = [
    RID_COL,
    IMAGE_ID_COL,
    LABEL_COL,
    "normalized_t1_nii_path",
    "normalized_t1_npy_path",
]

missing_cols = [col for col in required_cols if col not in t1_norm_manifest.columns]
assert not missing_cols, f"Missing required columns in T1-normalized manifest: {missing_cols}"

T1_NORM_QC_DIR = (
    PROJECT_DIR
    / "qc"
    / "clean_90d_mni_n4_syn_jacobian"
    / T1_NORM_BRANCH_NAME
)

T1_NORM_QC_DIR.mkdir(parents=True, exist_ok=True)

T1_NORM_DISK_QC_REPORT_PATH = (
    T1_NORM_QC_DIR
    / "clean_90d_mni_n4_syn_rectcrop_177x213x183_t1norm_p01p99_disk_qc_report.csv"
)

qc_rows = []
start_time = time.time()

for row_number, (_, row) in enumerate(t1_norm_manifest.iterrows(), start=1):
    normalized_t1_nii_path = Path(row["normalized_t1_nii_path"])
    normalized_t1_npy_path = Path(row["normalized_t1_npy_path"])

    nii_exists = normalized_t1_nii_path.exists()
    npy_exists = normalized_t1_npy_path.exists()

    arr_shape = None
    arr_dtype = None
    all_finite = False
    value_range_ok = False

    arr_min = np.nan
    arr_p01 = np.nan
    arr_p50 = np.nan
    arr_p99 = np.nan
    arr_max = np.nan
    arr_mean = np.nan
    arr_std = np.nan
    zero_fraction = np.nan
    one_fraction = np.nan

    qc_error = ""

    if npy_exists:
        try:
            arr = np.load(normalized_t1_npy_path, mmap_mode="r")

            arr_shape = tuple(arr.shape)
            arr_dtype = str(arr.dtype)

            shape_ok = arr_shape == EXPECTED_CROP_SHAPE

            # I scan the saved array from disk to confirm that it is finite and bounded.
            arr_min = float(np.nanmin(arr))
            arr_max = float(np.nanmax(arr))
            all_finite = bool(np.isfinite(arr).all())
            value_range_ok = bool(arr_min >= 0.0 and arr_max <= 1.0)

            arr_p01 = float(np.nanpercentile(arr, 1))
            arr_p50 = float(np.nanpercentile(arr, 50))
            arr_p99 = float(np.nanpercentile(arr, 99))
            arr_mean = float(np.nanmean(arr))
            arr_std = float(np.nanstd(arr))
            zero_fraction = float(np.mean(arr == 0.0))
            one_fraction = float(np.mean(arr == 1.0))

        except Exception as exc:
            shape_ok = False
            qc_error = repr(exc)

    else:
        shape_ok = False
        qc_error = "Normalized T1 NPY file does not exist."

    qc_pass = bool(
        nii_exists
        and npy_exists
        and shape_ok
        and all_finite
        and value_range_ok
    )

    qc_row = {
        RID_COL: row[RID_COL],
        IMAGE_ID_COL: row[IMAGE_ID_COL],
        LABEL_COL: row[LABEL_COL],
        "normalized_t1_nii_path": str(normalized_t1_nii_path),
        "normalized_t1_npy_path": str(normalized_t1_npy_path),
        "normalized_t1_nii_exists": nii_exists,
        "normalized_t1_npy_exists": npy_exists,
        "normalized_t1_shape": str(arr_shape),
        "normalized_t1_dtype": arr_dtype,
        "normalized_t1_shape_ok": shape_ok,
        "normalized_t1_all_finite": all_finite,
        "normalized_t1_range_ok": value_range_ok,
        "normalized_t1_min": arr_min,
        "normalized_t1_p01": arr_p01,
        "normalized_t1_p50": arr_p50,
        "normalized_t1_p99": arr_p99,
        "normalized_t1_max": arr_max,
        "normalized_t1_mean": arr_mean,
        "normalized_t1_std": arr_std,
        "normalized_t1_zero_fraction": zero_fraction,
        "normalized_t1_one_fraction": one_fraction,
        "normalized_t1_disk_qc_pass": qc_pass,
        "qc_error": qc_error,
    }

    for metadata_col in [
        "field_strength_category",
        "manufacturer_clean",
        "model_family",
        "research_group",
        "phase",
        "baseline_phase",
    ]:
        if metadata_col in t1_norm_manifest.columns:
            qc_row[metadata_col] = row[metadata_col]

    qc_rows.append(qc_row)

    if row_number % 50 == 0 or row_number == len(t1_norm_manifest):
        elapsed_minutes = (time.time() - start_time) / 60
        print(
            f"Checked {row_number}/{len(t1_norm_manifest)} normalized T1 files "
            f"in {elapsed_minutes:.1f} minutes."
        )

t1_norm_disk_qc = pd.DataFrame(qc_rows)
t1_norm_disk_qc.to_csv(T1_NORM_DISK_QC_REPORT_PATH, index=False)

print("\nSaved normalized T1 disk QC report:")
print(T1_NORM_DISK_QC_REPORT_PATH)

print("\nDisk QC pass counts:")
display(t1_norm_disk_qc["normalized_t1_disk_qc_pass"].value_counts().to_frame("count"))

print("\nExistence / shape / value checks:")
display(
    t1_norm_disk_qc[
        [
            "normalized_t1_nii_exists",
            "normalized_t1_npy_exists",
            "normalized_t1_shape_ok",
            "normalized_t1_all_finite",
            "normalized_t1_range_ok",
        ]
    ].sum().to_frame("passing_count")
)

print("\nNormalized T1 value summary:")
display(
    t1_norm_disk_qc[
        [
            "normalized_t1_min",
            "normalized_t1_p01",
            "normalized_t1_p50",
            "normalized_t1_p99",
            "normalized_t1_max",
            "normalized_t1_mean",
            "normalized_t1_std",
            "normalized_t1_zero_fraction",
            "normalized_t1_one_fraction",
        ]
    ].describe()
)

print("\nCohort counts:")
display(t1_norm_disk_qc[LABEL_COL].value_counts().to_frame("count"))

failed_qc = t1_norm_disk_qc.loc[~t1_norm_disk_qc["normalized_t1_disk_qc_pass"]].copy()

if len(failed_qc) > 0:
    print("\nFailed normalized T1 QC cases:")
    display(failed_qc)
else:
    print("\nAll normalized T1 files passed disk QC.")

## 1.60. Result of normalized T1 disk QC

The saved normalized T1 branch passed disk-based QC for all 1063 subjects. All normalized T1 NIfTI and NPY files were present, all NPY arrays had the expected shape `177 × 213 × 183`, all values were finite, and all values were within the expected `[0, 1]` range.

The value distribution was consistent with the selected `p1-p99` positive-voxel clipping plus min-max scaling strategy. All subjects had minimum value `0` and maximum value `1`; the median normalized p99 was close to `1`, confirming that the raw scanner-dependent intensity-scale differences were removed. The small fraction of voxels equal to `1` is expected because values above the subject-specific 99th percentile were clipped.

This confirms that the normalized T1 MRI branch is ready to be used as the main MRI input branch for modelling.